# On-device Online-SDFT: learn from the route you actually took

<a href="https://colab.research.google.com/github/lin826/Online-SDFT-Benchmark/blob/main/online_sdft_bandit_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

An on-device assistant must act before feedback exists. It can learn only after
the selected route produces a real OS/app callback. This notebook implements
that delayed mobile timeline with a **same-network** hindsight teacher and no
counterfactual outcomes.

This notebook makes that concrete with a real
[`LiquidAI/LFM2.5-230M`](https://huggingface.co/LiquidAI/LFM2.5-230M)
student. It is **self-contained**: the simulator and pipeline implementation
and the audited compact reference artifacts are embedded; it reads no
repository files. Strict reproduction uses the original Apple MPS/FP32 stack.
A T4 or CPU can run the same six-method protocol in portable mode, but different
numerical kernels are not claimed to produce identical bytes (Sections 5+).

### What you need

Sections 1–4 run in any Python notebook. Sections 5–6 need internet access and
Python 3.11+ to install a pinned LLM stack and download the public model. Every
notification and response is **synthetic**, and the run reproduces the
simulator rather than measuring latency or energy. Model use follows the
[`LFM1.0` license](https://huggingface.co/LiquidAI/LFM2.5-230M/blob/main/LICENSE).

### Path (play → understand → reproduce)

| Do this | Why |
| --- | --- |
| [1. Play the router](#1-play-the-router) | Feel the information bottleneck yourself |
| [2–4. Follow the pipeline](#2-protocol-at-a-glance) | See what is observed, delayed, and learned |
| [5. Reproduce](#5-reproduce-the-six-method-benchmark) | Run three paired seeds × all six methods and compare exact bytes |
| [6. Inspect](#6-plots-and-audits) | Plot every method and audit censored feedback |

### Map for RL / ML / AI engineers

| Term you already know | In this notebook |
| --- | --- |
| Contextual bandit $x_t$ | title/body + category/time/regime + local importance estimate |
| Bandit feedback | Observable user selection from the **executed** route, or `UNKNOWN` |
| Soft distillation / SDFT | Same-LFM hindsight + reliability-conditioned causal fusion |
| Hidden sampled preference | One seeded draw from $q_i\propto p_i^{100}$ after utility normalization ($T=0.01$); **simulation and accuracy only**, never training |
| Utility-optimal route | Full-information utility argmax; **regret only**, never training |
| Cumulative regret here | Σ utility gap vs clairvoyant $a^\star_t$, not best-in-class of $x$ alone |

> Long code cells are collapsed where the viewer supports it. Stay in the
> markdown + game unless you want implementation guts.

## 1. Play the router

Use **only** the visible cues. Run the next two cells in order: initialize the
collapsed game engine once, then set `SCENARIO_ID` / `MY_ACTION` and run the
visible play cell. Commit mentally before you reveal the debrief.

| ID | Notification | Visible cues |
| ---: | --- | --- |
| `0` | “Mobile review starts in 10 minutes” | `calendar` · 15:19 · weekday · importance 0.97 |
| `1` | “Critical: Identity errors above threshold” | `monitoring` · 00:25 · on-call · importance 0.94 |
| `2` | “Maya sent you a message” | `social` · 20:11 · off-hours · importance 0.45 |
| `3` | “New grocery recommendations selected for you” | `promo` · 15:03 · weekday · importance 0.06 |

These are exact rows from seed 0 of the versioned
`semantic-title-body-sharp-t001` dataset. The engine cell is collapsed so its
hidden scoring state does not spoil the game.

In [ ]:
# @title Hidden game engine and scenario state { display-mode: "form" }
from IPython.display import Markdown, display

GAME_ACTIONS = ("INTERRUPT", "LATER", "ARCHIVE")
GAME_SCENARIOS = (
    {
        "name": "Weekday calendar alert",
        "dataset_version": "semantic-title-body-sharp-t001",
        "sampled_preference": "INTERRUPT",
        "event_id": "s0-p0-0057",
        "title": "Mobile review starts in 10 minutes",
        "body": "Sam asked you to join on time. Tap to open the video call.",
        "category": "calendar", "time": "15:19", "regime": "weekday",
        "importance": 0.9685343635272166,
        "deadline": 0.9941276450659722,
        "affinity": 0.6330453855667049,
        "busy": 0.6530045112295922,
        "incident": 0.0, "manager": 0.0, "social": 0.0,
        "quiet_work": 0.0,
        "useful_horizon_minutes": 10,
    },
    {
        "name": "On-call monitoring incident",
        "dataset_version": "semantic-title-body-sharp-t001",
        "sampled_preference": "INTERRUPT",
        "event_id": "s0-p1-0034",
        "title": "Critical: Identity errors above threshold",
        "body": "Production failures reached 8%. An acknowledgement is requested within 15 minutes.",
        "category": "monitoring", "time": "00:25", "regime": "on-call",
        "importance": 0.9396736910360113,
        "deadline": 0.9966364522020464,
        "affinity": 0.18741322127881177,
        "busy": 0.4304177654866886,
        "incident": 1.0, "manager": 0.0, "social": 0.0,
        "quiet_work": 0.0,
        "useful_horizon_minutes": 15,
    },
    {
        "name": "Off-hours message from a close friend",
        "dataset_version": "semantic-title-body-sharp-t001",
        "sampled_preference": "LATER",
        "event_id": "s0-p2-0061",
        "title": "Maya sent you a message",
        "body": "That trail looks great. Are you free sometime this weekend?",
        "category": "social", "time": "20:11", "regime": "off-hours",
        "importance": 0.4487328928136745,
        "deadline": 0.08691709463106391,
        "affinity": 0.9392671694976312,
        "busy": 0.20804404669712193,
        "incident": 0.0, "manager": 0.0, "social": 0.5,
        "quiet_work": 0.0,
        "useful_horizon_minutes": None,
    },
    {
        "name": "Weekday promotion",
        "dataset_version": "semantic-title-body-sharp-t001",
        "sampled_preference": "ARCHIVE",
        "event_id": "s0-p0-0027",
        "title": "New grocery recommendations selected for you",
        "body": "Browse this week's grocery offers whenever you have time.",
        "category": "promo", "time": "15:03", "regime": "weekday",
        "importance": 0.06147452888802866,
        "deadline": 0.01871778380358485,
        "affinity": 0.0789604719804393,
        "busy": 0.7007237011671852,
        "incident": 0.0, "manager": 0.0, "social": 0.0,
        "quiet_work": 0.0,
        "useful_horizon_minutes": None,
    },
)


def _game_utilities(scenario):
    importance = scenario["importance"]
    affinity = scenario["affinity"]
    urgency = importance * scenario["deadline"]
    busy_cost = 1.20 * scenario["busy"] * (1.0 - 0.65 * urgency)
    interrupt = (1.45 * urgency + 0.42 * affinity - busy_cost
                 + 1.00 * scenario["incident"] + 0.60 * scenario["manager"]
                 + 0.50 * scenario["social"]
                 - 0.65 * scenario["quiet_work"] * (1.0 - urgency))
    later = (0.72 * importance + 0.58 * affinity - 0.62 * urgency
             + 0.22 * scenario["busy"] - 0.62 * scenario["incident"])
    archive = (0.72 * (1 - importance) + 0.36 * (1 - affinity)
               - 0.80 * urgency - 0.50 * scenario["social"])
    horizon = scenario["useful_horizon_minutes"]
    if horizon is not None and horizon < 120:
        missed_fraction = 1.0 - horizon / 120
        stale_digest = archive - missed_fraction * (0.10 + 0.25 * urgency)
        later = min(later, stale_digest)
    return (interrupt, later, archive)


def _execute_game_action(action_index, preference_index):
    if action_index == 0:
        outcomes = (
            ("OPENED_IMMEDIATELY", "INTERRUPT", "MATCH", 1, 5.0),
            ("OPENED_AFTER_DELAY", "LATER", "MISS", 120, -1.0),
            ("DELETED_NOTIFICATION", "ARCHIVE", "MISS", 15, -2.0),
        )
        return outcomes[preference_index]
    if action_index == 1:
        outcomes = (
            ("OPENED_DIGEST", "LATER", "MATCH", 120, 0.25),
            ("OPENED_DIGEST", "LATER", "MATCH", 120, 0.25),
            ("DELETED_FROM_DIGEST", "ARCHIVE", "MISS", 120, -1.0),
        )
        return outcomes[preference_index]
    return ("NO_OBSERVABLE_SELECTION", "UNKNOWN", "UNKNOWN", 240, 0.0)


def play_notification_round(scenario_id, action):
    if not 0 <= scenario_id < len(GAME_SCENARIOS):
        raise ValueError(f"scenario_id must be 0–{len(GAME_SCENARIOS) - 1}")
    action = action.upper()
    if action not in GAME_ACTIONS:
        raise ValueError(f"action must be one of {GAME_ACTIONS}")

    scenario = GAME_SCENARIOS[scenario_id]
    action_index = GAME_ACTIONS.index(action)
    utilities = _game_utilities(scenario)
    best_index = max(range(len(GAME_ACTIONS)), key=lambda index: utilities[index])
    sampled_index = GAME_ACTIONS.index(scenario["sampled_preference"])
    outcome, selection, status, delay, reward = _execute_game_action(
        action_index, sampled_index
    )
    regret = utilities[best_index] - utilities[action_index]

    display(Markdown(f"""
### Your round: {scenario['name']}

| Visible before acting | Value |
| --- | --- |
| Title | **{scenario['title']}** |
| Body | {scenario['body']} |
| Category | `{scenario['category']}` |
| Local time | {scenario['time']} |
| Regime | `{scenario['regime']}` |
| Importance | `{scenario['importance']:.2f}` |
| Dataset row | `{scenario['dataset_version']}` · `{scenario['event_id']}` |

**You committed to:** `{action}`

**One factual outcome:** `{outcome}` after {delay} minute(s)<br>
**Observed user selection:** `{selection}` · `{status}` · shared reward `{reward:+.1f}`

#### Debrief — evaluator only

Current busyness was `{scenario['busy']:.2f}`. The seeded sampled preference was
`{GAME_ACTIONS[sampled_index]}`; the utility-optimal route was
`{GAME_ACTIONS[best_index]}`. This action's regret was `{regret:.3f}` utility
units.

> REINFORCE maps this matured factual outcome to a learner-only reward and
> applies a batched action-token update to its LoRA adapter. That training map
> is distinct from the shared reward displayed above. Online-SDFT instead
> receives a same-LM soft target conditioned on the observed selection. No
> method receives the hidden busyness, preference draw, scoring utilities, or
> optimal route shown in this debrief. `UNKNOWN` is never replaced by either
> evaluator-only answer.
"""))

display(Markdown(
    "✅ **Game engine ready.** Set `SCENARIO_ID` and `MY_ACTION` in the next "
    "cell, then run it."
))

In [ ]:
# Change these two values, then run this cell.
SCENARIO_ID = 0
MY_ACTION = "INTERRUPT"  # INTERRUPT, LATER, or ARCHIVE

play_notification_round(SCENARIO_ID, MY_ACTION)

### What to notice

- A click does not prove the route maximized total utility (interruption cost).
- Only the chosen action creates an observable selection.
- A digest read under `LATER` reveals `LATER`, even when the hidden evaluator preferred an immediate open.
- Every `ARCHIVE` case stays `UNKNOWN`.
- The debrief is for you; the learner never sees the sampled preference or utility-optimal route.
- Feedback can improve the **next** decision only.

## 2. Protocol at a glance

The simulator's causal callback clock advances 15 minutes per decision; the
displayed local time is separate ordered context. An interrupt can expose an
immediate open after 1 minute, deletion after 15 minutes, or a
delayed read after 120 minutes. Digest observations mature after 120 minutes;
archive remains `UNKNOWN` after 240 minutes. A lesson can affect only requests
that arrive after its event-specific release. A matured `UNKNOWN` record is
audited and cannot become a route label or gradient update. The ICL/RAG
baselines may retain it only as explicitly unlabeled factual history.

Section 5 executes this exact delayed-feedback protocol for Base, ICL, RAG,
REINFORCE, RFT, and Online-SDFT on the three canonical paired streams.

### REINFORCE benchmark configuration

The promoted REINFORCE baseline uses only matured callbacks from its executed
delivery surface. Its learner-only outcome map is `+5` for an immediate push
open, `-1` for a delayed push open, `-5` for a push deletion, `0` for a digest
open, and `-5` for a digest deletion. `UNKNOWN` is censored before mapping and
causes no update. This shaped training signal is separate from the shared
observable-reward metric reported for all methods.

It trains the common rank-4 Q/K/V/O LoRA adapter with learning rate `1e-4`,
batch size eight, a fixed zero baseline, entropy coefficient `1.0`, and
gradient-norm clipping at `1.0`. Those settings were selected in-sample on the
same canonical seeds 0–2, using exact action-match accuracy with utility regret
as a strict secondary gate; no disjoint confirmation seeds were run. All methods
retain the same `semantic-title-body-sharp-t001` streams and $T=0.01$ evaluator
sampling. The hidden preference, utilities, deadline, urgency, affinity, and
scenario taxonomy never enter the learner.

## 3. Understand the setting

### Contextual-bandit contract

| Moment | Available | Sealed away |
| --- | --- | --- |
| Decision snapshot | title/body, category, local time, regime and importance | busy/interruption state, exact deadline/affinity, future callback, sampled preference, utility-optimal route |
| Student rollout | semantic context, parameters, matured past records | current callback, sampled preference, utility-optimal route |
| Observation window | user selection from the **selected** action, or `UNKNOWN` | both counterfactual outcomes |
| Future update | reliability-conditioned soft target: teacher + frozen decision prior + causal support; replay 64, batch 8 | scalar reward, sampled preference, or utility-optimal route as a label |

### Why this is not batch learning

| Batch | This online stream |
| --- | --- |
| Shuffled labeled set before serving | Each callback is consumed only after its delay expires |
| Shuffle + many epochs | One pass, in time order |
| Held-out exam afterward | Every live action is scored |
| Errors can vanish from test set | Cold-start errors stay in the metric |

The stream drifts weekday → on-call → off-hours. The objective is usefulness
**during** adaptation.

### Versioned evaluator sampling

The benchmark first normalizes evaluator utilities (shifting by the negative
minimum only when needed), then samples one hidden preference from
$q_i=p_i^{1/T}/\sum_j p_j^{1/T}$ at $T=0.01$, equivalently
$q_i\propto p_i^{100}$. The transform is evaluated in stable log space and
preserves exact zero support. Accuracy checks whether the executed action
exactly matches that sampled hidden preference. The utilities shape this
upstream distribution, but their numeric values are used directly only for
regret against the utility-optimal route; none of these evaluator-only fields
enters the learner.

The canonical seeds 0–2 have SHA-256 fingerprint
`986cdf1a7d5fcc04c2b33f1bf90a1fc4f24a97ee85e663370382d8a67e4c932d`.
Strict reproduction gates on that fingerprint, the audited source manifest,
the pinned model snapshot, and the exact three compact artifact byte strings.

## 4. Follow the six-method pipeline

The notebook embeds the production stream, model wrapper, six method agents,
delayed-feedback loop, aggregation code, and compact serializer. Each seed gets
a fresh pinned LFM+LoRA instance; every method starts from the identical adapter
snapshot. Base, ICL, and RAG keep that zero-effect adapter frozen, while
REINFORCE, RFT, and Online-SDFT train the same 172,032-parameter LoRA capacity.

### Deployment contract

| Stage | Information |
| --- | --- |
| Student rollout | Visible `x_t` only |
| Decision snapshot | Title/body, category, local time, regime and importance |
| Teacher timing | After the selected route's callback matures |
| Teacher input | `x_t`, executed `a_t`, factual `f_t`, observed selection `s_t` or `UNKNOWN` |
| Initialization | The common zero-initialized rank-4 Q/K/V/O LoRA state |
| Retained target | Complete causal-fusion distribution: same-LFM teacher + frozen decision prior + causal support |
| Local update | A rank-4 Q/K/V/O LoRA adapter with 172,032 trainable parameters at learning rate `1e-3`; replay 64 with a 32-step recency half-life, batch 8, two steps after a four-example warm-up |
| Serving policy | 2% epsilon-greedy baseline; while max student confidence is at most 60%, mix a 15% `INTERRUPT` probe with an 80-step half-life; after step 160, taper the full exploratory distribution toward argmax with a five-step half-life |

The causal-fusion weights depend on callback reliability. Reliable singleton
callbacks use 5% fixed-initial teacher, 5% frozen decision prior, and 90%
causal support. For an ambiguous digest open, the frozen decision prior is
projected onto the compatible `{INTERRUPT, LATER}` support; the teacher gets
zero weight, so the learner preserves the uncertainty the callback leaves
unresolved. Replay sampling is recency-weighted and selection-balanced.
`ARCHIVE` cannot expose any preference, and `UNKNOWN` is audited but never
retained as a target. Serving uses the learned adapter only; no replay example
is inserted into the prompt.

## 5. Reproduce the six-method benchmark

This section executes 4,320 chronological decisions: three paired seeds × six
methods × 240 decisions. On the audited Apple MPS host it normally takes several
minutes; CPU can take substantially longer.

### 5.1 Install the LLM runtime

In Colab, **T4 GPU is preferred** for a portable run. A CPU-only runtime is
supported if GPU budget is exhausted. The setup cell
must run **before any cell that imports PyTorch**. It keeps Colab’s
CUDA-matched PyTorch and already-loaded numerical stack, pins the LLM package,
and verifies that the numerical and model runtime import cleanly. A normal
Colab run does not require a session restart.

Strict byte reproduction requires Python 3.11.9, NumPy 2.4.6, Torch 2.13.0,
Transformers 5.13.1, PEFT 0.19.1, Apple MPS, and FP32. Colab's CUDA/FP16 stack
can execute portable mode but cannot honestly claim byte identity with MPS.

> **Recovering after an earlier dependency or LFM model-loading error:** choose
> **Runtime → Disconnect and delete runtime**, reconnect, then run this corrected
> setup cell first. Those messages came from the old live dependency state,
> not from the embedded Online-SDFT pipeline.

In [ ]:
# @title Install the Liquid LFM runtime { display-mode: "form" }
import importlib.metadata as package_metadata
import importlib.util
import os
import subprocess
import sys


in_colab = bool(os.environ.get("COLAB_RELEASE_TAG"))
_ONLINE_SDFT_RUNTIME_READY = False
# Colab imports NumPy for its own variable inspector before user cells run.
# Keep that internally consistent numerical stack in Colab; exact numerical
# pins remain the policy for local kernels and the canonical benchmark.
numeric_targets = {} if in_colab else {
    "numpy": ("numpy", "2.4.6"),
}
loaded_conflicts = []
for module_name, (distribution_name, target_version) in numeric_targets.items():
    loaded_module = sys.modules.get(module_name)
    if loaded_module is None:
        continue
    try:
        installed_version = package_metadata.version(distribution_name)
    except package_metadata.PackageNotFoundError:
        installed_version = None
    loaded_version = getattr(loaded_module, "__version__", None)
    if installed_version != target_version or loaded_version != target_version:
        loaded_conflicts.append(
            f"{module_name} loaded={loaded_version}, "
            f"installed={installed_version}, target={target_version}"
        )

# PEFT probes this optional package while dispatching every LoRA target. Colab
# currently preinstalls an old compiled TorchAO that PEFT 0.19 rejects, even
# though this unquantized benchmark never uses TorchAO. Remove the optional
# distribution instead of replacing Colab's CUDA-matched Torch stack.
if in_colab:
    try:
        package_metadata.version("torchao")
    except package_metadata.PackageNotFoundError:
        pass
    else:
        if "torchao" in sys.modules:
            raise RuntimeError(
                "Colab loaded TorchAO before setup. Choose Runtime > "
                "Disconnect and delete runtime, reconnect, and run this "
                "setup cell first."
            )
        subprocess.check_call(
            [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"]
        )
        importlib.invalidate_caches()

packages = [
    "transformers==5.13.1",
    "peft==0.19.1",
]
if in_colab:
    # These are already present in supported Colab images. Install only a
    # genuinely missing package; never replace a module Colab may have loaded.
    for module_name, requirement in (
        ("numpy", "numpy>=1.17,<3"),
        ("matplotlib", "matplotlib>=3.8,<4"),
        ("PIL", "Pillow>=10,<13"),
    ):
        if importlib.util.find_spec(module_name) is None:
            packages.append(requirement)
else:
    packages.extend(
        [
            "numpy==2.4.6",
            "torch==2.13.0",
            "matplotlib>=3.8,<4",
            "Pillow>=10,<13",
        ]
    )
if in_colab and importlib.util.find_spec("torch") is None:
    # Unusual Colab fallback; normal images already supply CUDA-matched Torch.
    packages.append("torch>=2.4,<3")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade-strategy",
        "only-if-needed",
        *packages,
    ]
)

# A local notebook may already have loaded a different pinned numerical stack.
# Refuse to mix its native extensions with newly installed files. Colab cannot
# enter this branch because its native numerical stack is deliberately retained.
if loaded_conflicts:
    raise RuntimeError(
        "The live kernel still holds an older numerical stack: "
        f"{'; '.join(loaded_conflicts)}. Restart the Jupyter kernel, then "
        "run this setup cell first."
    )

try:
    import numpy as np
    import numpy.testing  # catches an in-process NumPy upgrade immediately
    import torch
    import transformers
    import peft
    from peft import LoraConfig, get_peft_model
    from transformers import AutoTokenizer, Lfm2Config, Lfm2ForCausalLM

    # Imports alone did not catch Colab's incompatible TorchAO. Construct a
    # tiny LFM with the same Q/K/V/O PEFT LoRA path as the benchmark before
    # declaring the runtime ready; no auxiliary policy implementation is used.
    smoke_base = Lfm2ForCausalLM(
        Lfm2Config(
            vocab_size=32,
            hidden_size=16,
            intermediate_size=32,
            num_hidden_layers=1,
            num_attention_heads=2,
            num_key_value_heads=1,
            max_position_embeddings=32,
            block_multiple_of=8,
            full_attn_idxs=[0],
            tie_word_embeddings=False,
        )
    )
    smoke_lora = get_peft_model(
        smoke_base,
        LoraConfig(
            r=1,
            lora_alpha=2,
            target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
            bias="none",
            task_type="CAUSAL_LM",
        ),
    )
    assert any(
        "lora_A" in name and parameter.requires_grad
        for name, parameter in smoke_lora.named_parameters()
    )
    with torch.no_grad():
        smoke_lora(input_ids=torch.zeros((1, 2), dtype=torch.long))
    del smoke_lora, smoke_base
except Exception as exc:
    recovery = (
        "In Colab, choose Runtime > Disconnect and delete runtime, reconnect, "
        "and run this corrected setup cell first."
        if in_colab
        else "Create a fresh virtual environment and select it as the kernel."
    )
    raise RuntimeError(
        "Dependency smoke test failed; the live runtime may contain an "
        f"inconsistent numerical or model stack. {recovery}"
    ) from exc

_ONLINE_SDFT_RUNTIME_READY = True

if in_colab:
    if torch.cuda.is_available():
        print(f"GPU runtime detected: {torch.cuda.get_device_name(0)}")
    else:
        print(
            "CPU-only runtime detected. This is fully supported and uses FP32; "
            "the pipeline will take longer than on a T4 GPU."
        )

if torch.cuda.is_available():
    device_name = f"{torch.cuda.get_device_name(0)} (CUDA, FP16)"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_name = "Apple MPS (FP32)"
else:
    device_name = "CPU (FP32)"
print(
    "Runtime ready | "
    f"python={sys.version.split()[0]} | numpy={np.__version__} | "
    f"torch={torch.__version__} | transformers={transformers.__version__} | "
    f"peft={peft.__version__} | "
    f"device={device_name} | LoRA adapter smoke test=passed"
)

### 5.2 Load the embedded simulator and pipeline

This long cell is copied from the production modules by the notebook builder.
It defines the stream, every baseline, delayed feedback, same-network teacher,
LoRA learners, aggregation, and exact compact serializer. It reads no external
repository file.

In [ ]:
# @title Embedded production benchmark implementation { display-mode: "form" }
if not globals().get("_ONLINE_SDFT_RUNTIME_READY", False):
    raise RuntimeError(
        "Section 5.1 did not finish successfully. Run the setup cell and "
        "follow its recovery instruction before loading or running Online-SDFT."
    )

from pathlib import Path

"""Shared experiment configuration.

This module contains names and hyperparameters only. Environment dynamics live
in :mod:`online_sdft.environment`; learning algorithms live in
:mod:`online_sdft.methods`.
"""



DATASET_VERSION = "semantic-title-body-sharp-t001"
DATASET_NUMPY_VERSION = "2.4.6"
PUBLISHED_RESULTS_DATASET_VERSION = DATASET_VERSION
# Sample the evaluator-only user preference from a power-sharpened version of
# the proportional utility distribution: q_i is proportional to p_i**(1 / T).
PREFERENCE_SAMPLING_TEMPERATURE = 1e-2
RFT_PROTOCOL_VERSION = "teacher-categorical-k1-temperature8-singleton"
RFT_SETTINGS_PROVENANCE = (
    "fixed-temperature8-lr7e-4-same-lora-architecture"
)
RFT_CANDIDATE_COUNT = 1
RFT_SAMPLING_TEMPERATURE = 8.0
RFT_SAMPLING_MODE = "categorical"
RFT_LR = 7e-4

ACTIONS = ("INTERRUPT", "LATER", "ARCHIVE")
ACTION_CODES = ("A", "B", "C")
METHODS = (
    "Base",
    "ICL",
    "RAG",
    "REINFORCE",
    "RFT",
    "Online-SDFT",
)
CATEGORIES = (
    "manager",
    "calendar",
    "monitoring",
    "teammate",
    "social",
    "commerce",
    "promo",
)
REGIMES = ("weekday", "on-call", "off-hours")

PHASE_LENGTH = 80
STREAM_LENGTH = PHASE_LENGTH * len(REGIMES)
FEATURE_DIM = len(CATEGORIES) + 5
# Non-trainable RAG retrieval metadata: category one-hot + importance + hour
# sin/cos + regime + bias. LFM route scores come from the rendered prompt.
# Exact deadline and affinity floats stay simulator-only, although scenario
# wording intentionally provides coarse semantic evidence about salience.

# The simulator receives one notification every 15 minutes. A callback is
# released when that particular user event could have been observed. An
# immediate push open is visible quickly; a delayed read or digest interaction
# takes longer; silence after archiving remains uninformative.
DECISION_INTERVAL_MINUTES = 15
# LATER places the item in the next digest. Keeping delivery time separate from
# callback maturation makes the useful-horizon check in the evaluator explicit.
DIGEST_DELIVERY_DELAY_MINUTES = 120
FEEDBACK_WINDOWS_MINUTES = {
    "INTERRUPT_IMMEDIATE": 1,
    "INTERRUPT_DISMISSAL": 15,
    "INTERRUPT_DELAYED_READ": 120,
    "LATER": DIGEST_DELIVERY_DELAY_MINUTES,
    "ARCHIVE": 240,
}

# Reward is computed only from the factual event exposed by the executed
# delivery surface. A digest open is useful engagement evidence, but it is not
# proof that delaying was the right route: an immediately useful notification
# can also be opened later from the digest. Giving that ambiguous event the
# same +1 as an immediate open makes LATER dominate the contextual-bandit
# objective. The smaller engagement credit preserves the ordering supported by
# each observable trajectory without inventing a hidden route label:
#
#   immediate open: INTERRUPT > LATER > ARCHIVE
#   delayed/digest open: LATER > ARCHIVE > INTERRUPT
#   deletion: ARCHIVE > INTERRUPT/LATER
#
# ARCHIVE still exposes no interaction and remains neutral/censored.
OBSERVED_OUTCOME_REWARDS = {
    # Correct interruption is intentionally asymmetric: immediate utility is
    # easy to under-learn from sparse bandit feedback, while an unnecessary
    # interruption has a meaningful user cost. For INTERRUPT trajectories the
    # resulting reward vector is (+5, -1, -2).
    "OPENED_IMMEDIATELY": 5.0,
    "OPENED_AFTER_DELAY": -1.0,
    "DELETED_NOTIFICATION": -2.0,
    "OPENED_DIGEST": 0.25,
    "DELETED_FROM_DIGEST": -1.0,
    "NO_OBSERVABLE_SELECTION": 0.0,
}

EXPLORATION_EPSILON = 0.06
REPLAY_SIZE = 24
ONLINE_BATCH_SIZE = 4
ICL_K = 3
# The canonical memory baselines use three reliable demonstrations.
RAG_K = 3
# Weight assigned to visible title/body token overlap in RAG retrieval. The
# remainder is metadata similarity over category, importance, local hour, and
# regime.
RAG_TEXT_WEIGHT = 0.5
PROMPT_STYLE = "causal_demos"
# Canonical causal-demo prompts must fit this input budget. Other prompt/K
# combinations are supported only when their rendered input also fits; runtime
# validation fails loudly instead of truncating.
PROMPT_TOKEN_BUDGET = 768
TEACHER_PROMPT_VERSION = "concise-causal-v3"
PROMPT_STYLES = (
    "legacy",
    "compact",
    "causal_demos",
    "balanced_routes",
    "interrupt_narrow",
    "interrupt_context",
    "interaction_match",
    "history_guarded",
    "history_relevance",
)
INTERRUPT_PROMPT_SUFFIXES = {
    "causal_demos": (
        "Each assistant route in the history is a reliable observed user "
        "selection from a completed causal interaction. Use it only when "
        "the past notification is relevant to the current one."
    ),
    "balanced_routes": (
        "Treat all three routes as viable. Use the visible context and the "
        "closest past choices; do not default to LATER or ARCHIVE merely "
        "because they are quieter."
    ),
    "interrupt_narrow": (
        "A high-importance monitoring notification during on-call can warrant "
        "an immediate interruption. Otherwise follow visible context and close "
        "past choices."
    ),
    "interrupt_context": (
        "High-importance calendar, manager, or monitoring notifications can "
        "warrant an immediate interruption. Social, commerce, and promotion "
        "items usually should not interrupt unless close past choices support it."
    ),
    "interaction_match": (
        "A known observed selection is useful only when its notification "
        "category and regime resemble the current one. Do not copy unrelated "
        "interactions, and never turn an UNKNOWN selection into a route label."
    ),
    "history_guarded": (
        "Past interactions are weak, case-specific evidence. Never copy the "
        "latest route merely because it appears last. Use a past label only "
        "when its category and regime match the current notification; "
        "otherwise make the same decision you would make without history."
    ),
    "history_relevance": (
        "Each past interaction is tagged EXACT_MATCH or DIFFERENT_CONTEXT. "
        "Only an EXACT_MATCH label is relevant to the current notification. "
        "Ignore every DIFFERENT_CONTEXT label and never label UNKNOWN."
    ),
}

# Online-SDFT: soft targets and a small local replay window (no student ICL).
SDFT_REPLAY_SIZE = 32
SDFT_BATCH_SIZE = 8
SDFT_UPDATE_STEPS = 2
SDFT_WARMUP_EXAMPLES = 4
SDFT_DISTILL_TEMPERATURE = 1.0

MODEL_ID = "LiquidAI/LFM2.5-230M"
# Every adaptive arm resets and trains this PEFT LoRA adapter on the same
# physical Liquid model. Online-SDFT and RFT temporarily disable that adapter
# when the same model performs hindsight inference; REINFORCE has no teacher.
LORA_R = 4
LORA_ALPHA = 8
LORA_DROPOUT = 0.0
# LFM2.5 is a hybrid architecture: convolution blocks also expose a module
# named ``out_proj``. Use the qualified attention suffix so PEFT does not
# silently adapt those convolution projections as well.
LORA_TARGET_MODULES = (
    "q_proj",
    "k_proj",
    "v_proj",
    "self_attn.out_proj",
)
# LFM2.5-230M alternates convolution and attention blocks. These are its six
# attention-layer indices; pinning them makes the intended adapter shell
# auditable across PEFT releases and prevents hybrid-block name collisions.
LORA_LAYERS_TO_TRANSFORM = (2, 4, 6, 8, 10, 12)
SDFT_LR = 1e-3
SDFT_OPTIMIZER_WEIGHT_DECAY = 0.0
SDFT_MAX_GRAD_NORM = 1.0
# REINFORCE applies a factual-outcome policy-gradient objective to the same
# action-token LoRA adapter. These defaults were selected in-sample on the
# canonical seed-0--2 streams; there is no disjoint confirmation set.
REINFORCE_LR = 1e-4
REINFORCE_BATCH_SIZE = 8
REINFORCE_BASELINE_STEP = 0.0
REINFORCE_ENTROPY_COEF = 1.0
REINFORCE_MAX_GRAD_NORM = 1.0
# Learner-only reward shaping keyed solely by the matured callback from the
# executed delivery surface. This does not replace OBSERVED_OUTCOME_REWARDS in
# rollouts or reported cumulative observed reward. UNKNOWN selections remain
# censored before this map is consulted.
REINFORCE_TRAINING_OUTCOME_REWARDS = {
    "OPENED_IMMEDIATELY": 5.0,
    "OPENED_AFTER_DELAY": -1.0,
    "DELETED_NOTIFICATION": -5.0,
    "OPENED_DIGEST": 0.0,
    "DELETED_FROM_DIGEST": -5.0,
    "NO_OBSERVABLE_SELECTION": 0.0,
}
TEACHER_TEMPERATURE = 1.0
STUDENT_TEMPERATURE = 1.0

SYSTEM_PROMPT = """You are an on-device notification router.
Assess the partial evidence, then choose exactly one route:
A = INTERRUPT now
B = LATER in a digest
C = ARCHIVE without a notification
Use the current notification and any past completed interactions. Do not add explanation."""

TEACHER_SYSTEM_PROMPT = """Choose a route for a similar future notification:
A = INTERRUPT now
B = LATER in a digest
C = ARCHIVE silently
Use the notification and observed callback. No hidden label or unchosen outcome is available. A digest open after LATER leaves INTERRUPT versus LATER unresolved. UNKNOWN supports no route. Keep alternatives possible."""

TEACHER_REASONING_SYSTEM_PROMPT = """In one short paragraph, assess what the notification and observed callback imply for a similar future case.
The executed surface reveals only its observed behavior. Do not invent a hidden label or unchosen outcome. Explain uncertainty without choosing a route or giving a route code."""

"""Phone-observable evidence used by the hindsight teacher.

The live protocol deliberately avoids simulator latents, evaluator labels,
counterfactual outcomes, hand-written demonstrations, and a shadow policy. It
models an Android OS-integrated or user-authorized notification assistant that
can retain a small local event record:

* the notification and serving information already available to the policy;
* the route the assistant actually executed; and
* a later immediate open, delayed read, deletion, digest interaction, or
  explicit lack of an observable selection.

Simulator state never crosses this boundary. The scalar simulator reward is
intentionally not included: a phone observes events, not the benchmark's
engineered utility function.
"""


from dataclasses import dataclass

ROUTE_NARRATIVES = {
    "INTERRUPT": "delivered the notification as an immediate interruption",
    "LATER": "placed the notification in a later digest",
    "ARCHIVE": "archived the item without delivering a notification",
}

OUTCOME_NARRATIVES = {
    "OPENED_IMMEDIATELY": "The user opened it",
    "OPENED_AFTER_DELAY": "The user opened it",
    "DELETED_NOTIFICATION": "The user deleted the immediate notification",
    "OPENED_DIGEST": "The user opened it from the digest",
    "DELETED_FROM_DIGEST": "The user deleted it from the digest",
    "NO_OBSERVABLE_SELECTION": (
        "No delivered notification surface revealed a user choice"
    ),
}


@dataclass(frozen=True)
class FactualCallback:
    """Only callback fields permitted to cross into the teacher boundary."""

    action_taken: str
    outcome: str
    observed_user_selection: str
    delay_minutes: int


def project_factual_callback(feedback: dict) -> FactualCallback:
    """Discard reward, evaluator match, and transport bookkeeping eagerly."""
    return FactualCallback(
        action_taken=str(feedback["action_taken"]),
        outcome=str(feedback["outcome"]),
        observed_user_selection=str(feedback["observed_user_selection"]),
        delay_minutes=int(feedback["delay_minutes"]),
    )


def narrative_mobile_teacher_evidence(callback: FactualCallback) -> str:
    """Explain one factual callback as plain prose for the small teacher."""
    route = callback.action_taken
    outcome = callback.outcome
    selection = callback.observed_user_selection
    delay = callback.delay_minutes
    sentences = [
        f"The router {ROUTE_NARRATIVES[route]}.",
    ]
    if outcome == "NO_OBSERVABLE_SELECTION":
        sentences.append(
            f"{OUTCOME_NARRATIVES[outcome]} during the {delay} minute "
            "observation window."
        )
    elif delay == 1:
        sentences.append(f"{OUTCOME_NARRATIVES[outcome]} one minute later.")
    else:
        sentences.append(f"{OUTCOME_NARRATIVES[outcome]} {delay} minutes later.")
    if selection == "UNKNOWN":
        sentences.append(
            "The user's preferred route remains unknown because the "
            "executed surface revealed no selection."
        )
    else:
        sentences.append(
            f"This behavior revealed {selection} as the observed user "
            "selection on the executed surface."
        )
    return " ".join(sentences)

"""Notification-routing environment, outcomes, and mobile evidence views.

The environment owns the causal world: stream generation, action-dependent
feedback, and evaluator-only utilities. It exposes two disjoint projections of
:class:`Event` so that neither the deployed methods nor the language-model
teacher can read state they are not entitled to.

:class:`StudentObservation` is the serving view. Only after the executed route
produces a factual app/OS callback does the environment assemble
:class:`TeacherObservation`. Neither
view ever contains :meth:`NotificationRoutingEnvironment.oracle_utilities`,
which exists only to score the benchmark.

The soft teacher distribution is *not* computed here. The same Liquid LFM used
as the student reads a :class:`TeacherObservation` in
:meth:`online_sdft.methods.LiquidLLMPolicy.teacher_probs`. The evidence
boundary is defined in :mod:`online_sdft.privilege`.
"""


import hashlib
import json
import math
from dataclasses import dataclass
from typing import Iterable

import numpy as np



def probability_power_temperature(
    probabilities: np.ndarray,
    temperature: float,
) -> np.ndarray:
    """Temperature-scale a probability vector in stable log space.

    This computes ``q_i = p_i**(1 / temperature) / Z`` without taking the
    power directly. Exact zeros retain zero mass, equal inputs retain equal
    mass, and the input argmax is therefore unchanged.
    """
    values = np.asarray(probabilities, dtype=float)
    if values.ndim != 1 or values.size == 0:
        raise ValueError("probabilities must be a non-empty vector")
    if not np.isfinite(values).all() or (values < 0.0).any():
        raise ValueError("probabilities must be finite and non-negative")
    total = float(values.sum())
    if not np.isfinite(total) or total <= 0.0:
        raise ValueError("probabilities must have positive finite mass")
    temperature = float(temperature)
    if not np.isfinite(temperature) or temperature <= 0.0:
        raise ValueError("temperature must be positive and finite")

    normalized = values / total
    positive = normalized > 0.0
    log_probabilities = np.log(normalized[positive]) / temperature
    log_probabilities -= np.max(log_probabilities)
    scaled = np.zeros_like(normalized)
    with np.errstate(over="ignore", under="ignore"):
        scaled[positive] = np.exp(log_probabilities)
    return scaled / scaled.sum()


def one_hot(index: int, size: int) -> np.ndarray:
    values = np.zeros(size)
    values[index] = 1.0
    return values


@dataclass(frozen=True)
class NotificationScenario:
    """One auditable content pattern and its causal latent-state centers."""

    scenario_id: str
    tier: str
    title: str
    body: str
    importance_mean: float
    deadline_mean: float
    affinity_mean: float
    useful_horizon_minutes: int | None = None
    uses_relative_minutes: bool = False

    def __post_init__(self) -> None:
        if self.tier not in {"low", "routine", "urgent"}:
            raise ValueError(f"unknown scenario tier: {self.tier}")
        if not all(
            0.0 < value < 1.0
            for value in (
                self.importance_mean,
                self.deadline_mean,
                self.affinity_mean,
            )
        ):
            raise ValueError("scenario probability centers must lie in (0, 1)")
        if self.useful_horizon_minutes is not None and (
            self.useful_horizon_minutes <= 0
        ):
            raise ValueError("fixed useful horizon must be positive")
        if self.uses_relative_minutes and self.useful_horizon_minutes is not None:
            raise ValueError("relative and fixed useful horizons are exclusive")


PEOPLE = (
    "Maya", "Jordan", "Priya", "Luis", "Avery", "Sam", "Nora", "Eli",
    "Mei", "Omar", "Sofia", "Theo", "Rina", "Dev", "Kai", "Leah",
)
PROJECTS = (
    "Atlas", "Beacon", "Checkout", "Mobile", "Search", "Payments",
    "Identity", "Analytics", "Growth", "Notifications", "Billing",
    "Recommendations", "Messaging", "Accounts", "Reporting", "Platform",
)
MERCHANTS = (
    "Northstar Market", "Harbor Shop", "Cedar & Co.", "Juniper Goods",
    "Maple Market", "Summit Store", "Willow Supply", "Bluebird Outfitters",
    "Pine & Main", "Riverbend Market", "Oak Street Goods", "Lumen Shop",
    "Fieldstone Supply", "Redwood Outfitters", "Seaside Market", "Elm & Co.",
)
PRODUCTS = (
    "wireless earbuds", "trail shoes", "coffee grinder", "desk lamp",
    "travel backpack", "running watch", "camping stove", "standing mat",
    "water bottle", "mechanical keyboard", "yoga mat", "carry-on suitcase",
    "reading light", "bike helmet", "portable speaker", "wool blanket",
)
SOCIAL_TOPICS = (
    "hiking", "photography", "cooking", "cycling", "book club",
    "local event", "gardening", "travel", "running", "art", "music",
    "nature", "film", "volunteering", "design", "science",
)
PROMO_DEPARTMENTS = (
    "home", "outdoor", "electronics", "travel", "kitchen", "fitness",
    "office", "wellness", "audio", "pets", "beauty", "books", "sports",
    "automotive", "toys", "grocery",
)

SCENARIO_SALIENCE = {"low": 0.0, "routine": 0.5, "urgent": 1.0}
RELATIVE_DEADLINE_ADJUSTMENTS = {10: 0.08, 15: 0.04, 20: 0.0, 30: -0.08}
QUIET_HOURS_CATEGORIES = frozenset(
    {"manager", "calendar", "monitoring", "teammate", "commerce"}
)

# Each category contains a low-salience, routine, and time-sensitive scenario.
# Absolute scenario centers avoid the old category-prior pathology in which an
# optional meeting next week had a higher latent deadline than a call starting
# in minutes. The language still never names a route or gold label.
NOTIFICATION_SCENARIOS = {
    "manager": (
        NotificationScenario(
            "manager_recap", "low",
            "{name} shared a {project} status recap",
            "For your records. No response is needed.",
            0.25, 0.10, 0.45,
        ),
        NotificationScenario(
            "manager_review_tomorrow", "routine",
            "{project} plan needs your review",
            "{name}: Please leave comments by tomorrow afternoon.",
            0.65, 0.35, 0.55, 24 * 60,
        ),
        NotificationScenario(
            "manager_release_decision", "urgent",
            "{name} needs a decision on {project}",
            "Please approve or reject the release exception within {minutes} minutes.",
            0.95, 0.93, 0.65, uses_relative_minutes=True,
        ),
    ),
    "calendar": (
        NotificationScenario(
            "calendar_optional_next_week", "low",
            "Optional {project} office hours next week",
            "{name} invited you to a non-required session on Friday at 3:00 PM.",
            0.25, 0.10, 0.35, 7 * 24 * 60,
        ),
        NotificationScenario(
            "calendar_sync_tomorrow", "routine",
            "{project} project sync tomorrow",
            "A 30-minute meeting with {name} is scheduled for 2:00 PM.",
            0.60, 0.35, 0.45, 24 * 60,
        ),
        NotificationScenario(
            "calendar_starting_soon", "urgent",
            "{project} review starts in {minutes} minutes",
            "{name} asked you to join on time. Tap to open the video call.",
            0.95, 0.95, 0.55, uses_relative_minutes=True,
        ),
    ),
    "monitoring": (
        NotificationScenario(
            "monitoring_normal_report", "low",
            "{project} health summary is ready",
            "The latest production checks completed within the normal range.",
            0.20, 0.05, 0.15,
        ),
        NotificationScenario(
            "monitoring_latency_warning", "routine",
            "{project} latency warning",
            "P95 latency has been elevated for 15 minutes with no confirmed "
            "user impact.",
            0.62, 0.40, 0.25,
        ),
        NotificationScenario(
            "monitoring_critical_errors", "urgent",
            "Critical: {project} errors above threshold",
            "Production failures reached 8%. An acknowledgement is "
            "requested within {minutes} minutes.",
            0.95, 0.95, 0.35, uses_relative_minutes=True,
        ),
    ),
    "teammate": (
        NotificationScenario(
            "teammate_lunch_photos", "low",
            "{name} shared photos from the team lunch",
            "New photos were added to the social channel.",
            0.18, 0.05, 0.48,
        ),
        NotificationScenario(
            "teammate_dashboard_question", "routine",
            "{name} asked about the {project} dashboard",
            "When you have a moment, could you check the updated labels?",
            0.40, 0.20, 0.58,
        ),
        NotificationScenario(
            "teammate_blocked", "urgent",
            "{name} is blocked on {project}",
            "Can you confirm the rollback setting within {minutes} minutes?",
            0.82, 0.88, 0.68, uses_relative_minutes=True,
        ),
    ),
    "social": (
        NotificationScenario(
            "social_suggested_posts", "low",
            "New {topic} posts you may have missed",
            "See this week's suggested {topic} posts and community updates.",
            0.08, 0.01, 0.55,
        ),
        NotificationScenario(
            "social_weekend_message", "routine",
            "{name} sent you a message",
            "That trail looks great. Are you free sometime this weekend?",
            0.30, 0.10, 0.78,
        ),
        NotificationScenario(
            "social_live_call", "urgent",
            "{name} invited you to a live event",
            "The private group call starts in {minutes} minutes.",
            0.62, 0.75, 0.88, uses_relative_minutes=True,
        ),
    ),
    "commerce": (
        NotificationScenario(
            "commerce_receipt_available", "low",
            "Your {merchant} receipt is available",
            "A receipt for last week's purchase was added to your account.",
            0.10, 0.02, 0.28,
        ),
        NotificationScenario(
            "commerce_order_shipped", "routine",
            "Your {merchant} order has shipped",
            "The package is expected to arrive tomorrow between 1:00 and 4:00 PM.",
            0.25, 0.10, 0.38, 24 * 60,
        ),
        NotificationScenario(
            "commerce_payment_failed", "urgent",
            "Payment failed for your {merchant} order",
            "Update your payment method within 24 hours to keep the order.",
            0.70, 0.35, 0.48, 24 * 60,
        ),
    ),
    "promo": (
        NotificationScenario(
            "promo_recommendations", "low",
            "New {department} recommendations selected for you",
            "Browse this week's {department} offers whenever you have time.",
            0.03, 0.01, 0.10,
        ),
        NotificationScenario(
            "promo_member_offer", "routine",
            "{merchant} member offer: 15% off",
            "Your member offer is available for the next 12 hours.",
            0.15, 0.20, 0.18, 12 * 60,
        ),
        NotificationScenario(
            "promo_watchlist_price_drop", "urgent",
            "Price drop: {item} from your watchlist",
            "The sale price expires in {minutes} minutes while stock lasts.",
            0.50, 0.70, 0.60, uses_relative_minutes=True,
        ),
    ),
}


@dataclass
class Event:
    """Full simulator event, including evaluator/teacher-only state."""

    event_id: str
    phase: int
    category: str
    scenario_id: str
    scenario_tier: str
    title: str
    body: str
    hour: float
    useful_horizon_minutes: int | None
    importance: float
    deadline: float
    affinity: float
    busy: float
    x: np.ndarray
    z: dict
    sampled_preference: int | None = None


@dataclass(frozen=True)
class StudentObservation:
    """The complete and only view supplied to a deployed method."""

    text: str
    features: np.ndarray


@dataclass(frozen=True)
class TeacherObservation:
    """Hindsight view containing one real trajectory and no answer key."""

    context: str
    evidence: str
    observed_user_selection: str = "UNKNOWN"


class NotificationRoutingEnvironment:
    """Causal contextual-bandit simulator for one notification stream."""

    @staticmethod
    def _sample_probability(
        rng: np.random.Generator,
        mean: float,
        concentration: float = 30.0,
    ) -> float:
        """Sample a strictly bounded logit-normal probability.

        The former clipped Gaussian created large masses at zero and one. A
        beta replacement removed explicit clipping but still returned exact
        endpoints in floating point when a shape parameter fell below one.
        Sampling finite log-odds keeps every generated value strictly inside
        ``(0, 1)``; ``concentration`` retains its variance-control role.
        """
        if concentration <= 0:
            raise ValueError("concentration must be positive")
        mean = float(np.clip(mean, 0.005, 0.995))
        log_odds = math.log(mean) - math.log1p(-mean)
        scale = 0.55 * math.sqrt(30.0 / concentration)
        sampled_log_odds = log_odds + float(rng.normal(0.0, scale))
        if sampled_log_odds >= 0.0:
            return 1.0 / (1.0 + math.exp(-sampled_log_odds))
        odds = math.exp(sampled_log_odds)
        return odds / (1.0 + odds)

    @staticmethod
    def _sample_local_hour_key(
        rng: np.random.Generator,
        phase: int,
    ) -> float:
        """Sample an ordered local-time key appropriate to the regime.

        Off-hours spans one continuous overnight window from 18:00 through
        08:00; values above 24 are wrapped only when rendered.
        """
        if phase == 0:
            return float(rng.uniform(8.0, 18.0))
        if phase == 1:
            return float(rng.uniform(0.0, 24.0))
        return float(rng.uniform(18.0, 32.0))

    @staticmethod
    def _sample_local_hour_keys(
        rng: np.random.Generator,
        phase: int,
        count: int,
    ) -> list[float]:
        """Return distinct, ordered local times with bounded random jitter."""
        if count <= 0:
            return []
        starts = (8.0, 0.0, 18.0)
        ends = (18.0, 24.0, 32.0)
        start, end = starts[phase], ends[phase]
        width = (end - start) / count
        centers = start + (np.arange(count, dtype=float) + 0.5) * width
        jitter = rng.uniform(-0.20 * width, 0.20 * width, size=count)
        return (centers + jitter).tolist()

    @staticmethod
    def _deadline_mean(
        scenario: NotificationScenario,
        rendered_minutes: int,
    ) -> float:
        """Tie a stated relative deadline monotonically to latent urgency."""
        adjustment = (
            RELATIVE_DEADLINE_ADJUSTMENTS[rendered_minutes]
            if scenario.uses_relative_minutes
            else 0.0
        )
        return float(np.clip(scenario.deadline_mean + adjustment, 0.005, 0.995))

    def make_event(
        self,
        rng: np.random.Generator,
        phase: int,
        index: int,
        prefix: str,
        *,
        content_rng: np.random.Generator | None = None,
        entity_affinity: dict[tuple[str, str], float] | None = None,
        used_texts: set[tuple[str, str]] | None = None,
        hour: float | None = None,
        busy: float | None = None,
        preference_rng: np.random.Generator | None = None,
    ) -> Event:
        # Rotate the phase remainder so category counts differ by at most one
        # across the full stream (80 is not divisible by seven).
        category_offset = phase * (PHASE_LENGTH % len(CATEGORIES))
        category = CATEGORIES[(index + category_offset) % len(CATEGORIES)]
        occurrence = index // len(CATEGORIES)
        scenario = NOTIFICATION_SCENARIOS[category][(occurrence + phase) % 3]
        if content_rng is None:
            content_rng = rng
        for _ in range(256):
            content_fields = {
                "name": str(content_rng.choice(PEOPLE)),
                "project": str(content_rng.choice(PROJECTS)),
                "merchant": str(content_rng.choice(MERCHANTS)),
                "item": str(content_rng.choice(PRODUCTS)),
                "topic": str(content_rng.choice(SOCIAL_TOPICS)),
                "department": str(content_rng.choice(PROMO_DEPARTMENTS)),
                "minutes": (10, 15, 20, 30)[(occurrence + phase) % 4],
            }
            title = scenario.title.format(**content_fields)
            body = scenario.body.format(**content_fields)
            if used_texts is None or (title, body) not in used_texts:
                break
        else:  # pragma: no cover - catalog capacity is guarded by tests
            raise RuntimeError("could not render a unique notification")
        if used_texts is not None:
            used_texts.add((title, body))
        rendered_minutes = int(content_fields["minutes"])
        useful_horizon_minutes = (
            rendered_minutes
            if scenario.uses_relative_minutes
            else scenario.useful_horizon_minutes
        )
        if hour is None:
            hour = self._sample_local_hour_key(rng, phase) % 24.0
        importance = self._sample_probability(rng, scenario.importance_mean)
        deadline = self._sample_probability(
            rng,
            self._deadline_mean(scenario, rendered_minutes),
        )
        template = f"{scenario.title} {scenario.body}"
        affinity_offsets = entity_affinity or {}
        referenced_entities = [
            (field, value)
            for field, value in content_fields.items()
            if field != "minutes"
            and f"{{{field}}}" in template
            and (field, value) in affinity_offsets
        ]
        affinity_offset = (
            float(
                np.mean(
                    [affinity_offsets[entity] for entity in referenced_entities]
                )
            )
            if referenced_entities
            else 0.0
        )
        affinity = self._sample_probability(
            rng,
            float(np.clip(scenario.affinity_mean + affinity_offset, 0.02, 0.98)),
        )

        if busy is None:
            busy = self._sample_probability(
                rng,
                (0.68, 0.36, 0.18)[phase],
                concentration=20.0,
            )

        # Context bonuses scale with actual scenario salience. A normal health
        # report is not an incident merely because the user is on call, and
        # generic suggested posts are not a close social interruption merely
        # because they arrive off-hours.
        salience = SCENARIO_SALIENCE[scenario.tier]
        incident_on_call = float(
            phase == 1 and category == "monitoring"
        ) * salience
        leisure_social = float(phase == 2 and category == "social") * salience
        manager_focus = float(phase == 0 and category == "manager") * salience
        off_hours_quiet = float(
            phase == 2 and category in QUIET_HOURS_CATEGORIES
        )

        category_features = one_hot(CATEGORIES.index(category), len(CATEGORIES))
        # Numeric runtime features. Importance can come from app/OS ranking
        # metadata. Exact deadline and affinity remain simulator/evaluator-only
        # utility terms, while title/body provide coarse semantic evidence.
        features = np.concatenate(
            [
                category_features,
                np.array(
                    [
                        importance,
                        math.sin(2 * math.pi * hour / 24),
                        math.cos(2 * math.pi * hour / 24),
                        phase / 2.0,
                        1.0,
                    ]
                ),
            ]
        )
        privileged = {
            "busy": busy,
            "incident_on_call": incident_on_call,
            "leisure_social": leisure_social,
            "manager_focus": manager_focus,
            "off_hours_quiet": off_hours_quiet,
        }
        event = Event(
            event_id=f"{prefix}-{index:04d}",
            phase=phase,
            category=category,
            scenario_id=scenario.scenario_id,
            scenario_tier=scenario.tier,
            title=title,
            body=body,
            hour=hour,
            useful_horizon_minutes=useful_horizon_minutes,
            importance=importance,
            deadline=deadline,
            affinity=affinity,
            busy=busy,
            x=features,
            z=privileged,
        )
        if preference_rng is None:
            preference_rng = rng
        event.sampled_preference = int(
            preference_rng.choice(
                len(ACTIONS),
                p=self.gold_action_distribution(event),
            )
        )
        return event

    def make_stream(self, seed: int) -> list[Event]:
        (
            dynamics_seed,
            content_seed,
            profile_seed,
            order_seed,
            preference_seed,
        ) = (
            np.random.SeedSequence(seed).spawn(5)
        )
        rng = np.random.default_rng(dynamics_seed)
        content_rng = np.random.default_rng(content_seed)
        profile_rng = np.random.default_rng(profile_seed)
        order_rng = np.random.default_rng(order_seed)
        preference_rng = np.random.default_rng(preference_seed)
        entity_pools = {
            "name": PEOPLE,
            "project": PROJECTS,
            "merchant": MERCHANTS,
            "item": PRODUCTS,
            "topic": SOCIAL_TOPICS,
            "department": PROMO_DEPARTMENTS,
        }
        entity_affinity = {
            (field, entity): float(profile_rng.normal(0.0, 0.09))
            for field, pool in entity_pools.items()
            for entity in pool
        }
        used_texts: set[tuple[str, str]] = set()
        events = []
        for phase in range(3):
            indices = np.arange(PHASE_LENGTH)
            order_rng.shuffle(indices)
            hour_keys = self._sample_local_hour_keys(
                rng,
                phase,
                PHASE_LENGTH,
            )
            busy_mean = (0.68, 0.36, 0.18)[phase]
            busy_state = self._sample_probability(
                rng,
                busy_mean,
                concentration=20.0,
            )
            phase_events = []
            for position, index in enumerate(indices.tolist()):
                # Slowly varying interruptibility creates learnable local
                # continuity instead of an iid hidden label-flipping variable.
                busy_draw = self._sample_probability(
                    rng,
                    busy_mean,
                    concentration=20.0,
                )
                busy_state = 0.82 * busy_state + 0.18 * busy_draw
                phase_events.append(
                    self.make_event(
                        rng,
                        phase,
                        index,
                        f"s{seed}-p{phase}",
                        content_rng=content_rng,
                        entity_affinity=entity_affinity,
                        used_texts=used_texts,
                        hour=hour_keys[position] % 24.0,
                        busy=busy_state,
                        preference_rng=preference_rng,
                    )
                )
            events.extend(phase_events)
        return events

    def stream_fingerprint(self, seeds: Iterable[int]) -> str:
        """Hash model inputs, latent state, utilities, and routes."""
        digest = hashlib.sha256()
        for seed in seeds:
            for event in self.make_stream(int(seed)):
                observation = self.student_observation(event)
                record = {
                    "seed": int(seed),
                    "event_id": event.event_id,
                    "phase": event.phase,
                    "category": event.category,
                    "scenario_id": event.scenario_id,
                    "scenario_tier": event.scenario_tier,
                    "title": event.title,
                    "body": event.body,
                    "hour": event.hour.hex(),
                    "useful_horizon_minutes": event.useful_horizon_minutes,
                    "importance": event.importance.hex(),
                    "deadline": event.deadline.hex(),
                    "affinity": event.affinity.hex(),
                    "busy": event.busy.hex(),
                    "z": {
                        key: float(value).hex()
                        for key, value in sorted(event.z.items())
                    },
                    "student_text": observation.text,
                    "student_features": [
                        float(value).hex() for value in observation.features
                    ],
                    "oracle_utilities": [
                        float(value).hex()
                        for value in self.oracle_utilities(event)
                    ],
                    "gold_action_distribution": [
                        float(value).hex()
                        for value in self.gold_action_distribution(event)
                    ],
                    "gold_action": self.gold_action(event),
                }
                digest.update(
                    json.dumps(
                        record,
                        sort_keys=True,
                        separators=(",", ":"),
                    ).encode("utf-8")
                )
                digest.update(b"\n")
        return digest.hexdigest()

    @staticmethod
    def student_observation(event: Event) -> StudentObservation:
        """Project a full event onto student-visible information only.

        Title/body and local importance are available to all compared methods.
        Exact deadline and affinity values remain evaluator-only.
        """
        # Floor instead of round so a valid 23:59.x sample cannot render as an
        # unintended 00:00 wrap at the end of the on-call phase.
        total_minutes = int(math.floor(event.hour * 60)) % (24 * 60)
        hour, minute = divmod(total_minutes, 60)
        local_time = f"{hour:02d}:{minute:02d}"
        text = (
            f"The notification title is {event.title}. "
            f"The message says {event.body} "
            f"This is a {event.category} notification that arrived at "
            f"{local_time} local time during the {REGIMES[event.phase]} "
            "period. Its on-device importance score is "
            f"{event.importance:.2f} out of 1."
        )
        return StudentObservation(text=text, features=event.x.copy())

    @staticmethod
    def oracle_utilities(event: Event) -> np.ndarray:
        """Return evaluator-only utility; never a method training target."""
        z = event.z
        urgency = event.importance * event.deadline
        # Imminent valuable events remain interruptible even when the user is
        # busy; routine interruptions continue to pay the full disruption cost.
        busy_cost = 1.20 * z["busy"] * (1.0 - 0.65 * urgency)
        interrupt = (
            1.45 * urgency
            + 0.42 * event.affinity
            - busy_cost
            + 1.00 * z["incident_on_call"]
            + 0.60 * z["manager_focus"]
            + 0.50 * z["leisure_social"]
            - 0.65 * z["off_hours_quiet"] * (1.0 - urgency)
        )
        later = (
            0.72 * event.importance
            + 0.58 * event.affinity
            - 0.62 * urgency
            + 0.22 * z["busy"]
            - 0.62 * z["incident_on_call"]
        )
        archive = (
            0.72 * (1 - event.importance)
            + 0.36 * (1 - event.affinity)
            - 0.80 * urgency
            - 0.50 * z["leisure_social"]
        )
        if (
            event.useful_horizon_minutes is not None
            and event.useful_horizon_minutes < DIGEST_DELIVERY_DELAY_MINUTES
        ):
            missed_fraction = 1.0 - (
                event.useful_horizon_minutes / DIGEST_DELIVERY_DELAY_MINUTES
            )
            # Once the digest arrives after the content's useful horizon, it
            # is stale and adds clutter relative to archiving. This is a route
            # feasibility constraint inside the causal utility—not a post-hoc
            # gold-label override. Valuable imminent content can still favor
            # INTERRUPT; low-value imminent content can favor ARCHIVE.
            stale_digest = archive - missed_fraction * (0.10 + 0.25 * urgency)
            later = min(later, stale_digest)
        return np.array([interrupt, later, archive])

    def gold_action_distribution(self, event: Event) -> np.ndarray:
        """Sharpen proportional evaluator-utility choice probabilities.

        A nonnegative vector such as ``(20, 30, 50)`` maps exactly to
        ``(0.2, 0.3, 0.5)`` before probability-power temperature scaling. The
        current utility model can produce negative values, which are not valid
        sampling weights, so only those vectors are shifted by their minimum
        before normalization. An all-equal vector falls back to the uniform
        distribution. Temperature scaling is performed in log space so the
        configured sharp temperature remains numerically stable.
        """
        weights = np.asarray(self.oracle_utilities(event), dtype=float).copy()
        if weights.shape != (len(ACTIONS),) or not np.isfinite(weights).all():
            raise ValueError(
                "oracle utilities must be one finite value per action"
            )
        minimum = float(weights.min())
        if minimum < 0.0:
            weights -= minimum
        total = float(weights.sum())
        if total <= 0.0:
            return np.full(len(ACTIONS), 1.0 / len(ACTIONS))
        proportional = weights / total
        return probability_power_temperature(
            proportional,
            PREFERENCE_SAMPLING_TEMPERATURE,
        )

    def gold_action(self, event: Event) -> int:
        """Return the sampled hidden preference used by simulator/evaluator.

        The preference is drawn once from :meth:`gold_action_distribution`
        when the event is created. Keeping that draw on the immutable stream
        makes repeated calls and paired benchmark arms agree without exposing
        the preference to feedback, prompts, or updates.
        """
        if event.sampled_preference is None:
            raise ValueError("event preference was not sampled")
        return int(event.sampled_preference)

    def execute(
        self,
        event: Event,
        action: int,
        rng: np.random.Generator,
    ) -> dict:
        """Materialize only the user event observable under ``action``.

        This is a deterministic causal observation matrix conditioned on the
        event's stored preference draw. The returned record contains only what
        the executed delivery surface could reveal:

        * INTERRUPT can reveal immediate open, delayed read, or deletion;
        * LATER can reveal digest read or deletion. A digest read is observed
          as LATER even if the hidden evaluator preference was INTERRUPT; and
        * ARCHIVE exposes no notification interaction and is always UNKNOWN.

        ``rng`` stays in the public signature for injected environments, but
        the default mapping is deterministic so identical event/action pairs
        produce identical feedback for every benchmark arm.
        """
        del rng
        gold = self.gold_action(event)
        if action == 0:
            channel = "push_notification"
            if gold == 0:
                outcome = "OPENED_IMMEDIATELY"
                observed_selection = "INTERRUPT"
                match_status = "MATCH"
                delay = FEEDBACK_WINDOWS_MINUTES["INTERRUPT_IMMEDIATE"]
            elif gold == 1:
                outcome = "OPENED_AFTER_DELAY"
                observed_selection = "LATER"
                match_status = "MISS"
                delay = FEEDBACK_WINDOWS_MINUTES["INTERRUPT_DELAYED_READ"]
            else:
                outcome = "DELETED_NOTIFICATION"
                observed_selection = "ARCHIVE"
                match_status = "MISS"
                delay = FEEDBACK_WINDOWS_MINUTES["INTERRUPT_DISMISSAL"]
        elif action == 1:
            channel = "digest_inbox"
            delay = FEEDBACK_WINDOWS_MINUTES["LATER"]
            if gold == 0:
                # The missed immediate-open preference stays hidden, but the
                # later digest read is itself an observable LATER selection.
                outcome = "OPENED_DIGEST"
                observed_selection = "LATER"
                match_status = "MATCH"
            elif gold == 1:
                outcome = "OPENED_DIGEST"
                observed_selection = "LATER"
                match_status = "MATCH"
            else:
                outcome = "DELETED_FROM_DIGEST"
                observed_selection = "ARCHIVE"
                match_status = "MISS"
        else:
            channel = "notification_not_delivered"
            outcome = "NO_OBSERVABLE_SELECTION"
            observed_selection = "UNKNOWN"
            match_status = "UNKNOWN"
            delay = FEEDBACK_WINDOWS_MINUTES["ARCHIVE"]

        return {
            "action_taken": ACTIONS[action],
            "channel": channel,
            "outcome": outcome,
            "observed_user_selection": observed_selection,
            "match_status": match_status,
            "delay_minutes": delay,
            # Reward the observable delivery outcome, not whether the route
            # name happens to equal the surface-constrained selection. In
            # particular, OPENED_DIGEST receives only partial engagement
            # credit because it cannot distinguish a preferred delay from a
            # missed immediate need.
            "reward": OBSERVED_OUTCOME_REWARDS[outcome],
        }

    @staticmethod
    def teacher_observation(
        observation: StudentObservation,
        action: int,
        callback: FactualCallback,
    ) -> TeacherObservation:
        """Build a hindsight view only after factual feedback is available."""
        if callback.action_taken != ACTIONS[action]:
            raise ValueError("feedback must belong to the executed route")
        return TeacherObservation(
            context=observation.text,
            evidence=narrative_mobile_teacher_evidence(callback),
            observed_user_selection=callback.observed_user_selection,
        )


DEFAULT_ENVIRONMENT = NotificationRoutingEnvironment()


# Compatibility functions keep the public API small while tests and notebooks
# can dependency-inject NotificationRoutingEnvironment directly.
def make_event(
    rng: np.random.Generator,
    phase: int,
    index: int,
    prefix: str,
) -> Event:
    return DEFAULT_ENVIRONMENT.make_event(rng, phase, index, prefix)


def make_stream(seed: int) -> list[Event]:
    return DEFAULT_ENVIRONMENT.make_stream(seed)


def stream_fingerprint(seeds: Iterable[int]) -> str:
    return DEFAULT_ENVIRONMENT.stream_fingerprint(seeds)


def student_observation(event: Event) -> StudentObservation:
    return DEFAULT_ENVIRONMENT.student_observation(event)


def teacher_observation(
    observation: StudentObservation,
    action: int,
    callback: FactualCallback,
) -> TeacherObservation:
    """Project a sealed serving view and narrow callback into hindsight."""
    return DEFAULT_ENVIRONMENT.teacher_observation(
        observation,
        action,
        callback,
    )


def context_text(event: Event) -> str:
    return student_observation(event).text


def oracle_utilities(event: Event) -> np.ndarray:
    return DEFAULT_ENVIRONMENT.oracle_utilities(event)


def gold_action(event: Event) -> int:
    return DEFAULT_ENVIRONMENT.gold_action(event)


def factual_feedback(
    event: Event,
    action: int,
    rng: np.random.Generator,
) -> dict:
    return DEFAULT_ENVIRONMENT.execute(event, action, rng)

"""LLM policy and the compared online-learning methods.

Every method consumes :class:`StudentObservation`, which contains only the
natural-language serving view. For SDFT, the same Liquid network later acts as
a hindsight teacher over the original serving view and the factual callback
produced by the executed route. It never receives a counterfactual
outcome, benchmark reward, or evaluator label.
Environment execution and rewards are intentionally not simulated in this
module; REINFORCE receives the executed route's scalar reward, or applies an
explicit arm-specific map to its factual outcome, only after acting.
"""


from collections.abc import Mapping
from collections import Counter
from dataclasses import asdict, dataclass
from numbers import Real
import re
import string
from typing import Any, Protocol

import numpy as np



ASSESSMENT_FALLBACK = (
    "The callback is partial evidence, so unobserved alternatives remain "
    "possible."
)

_ASSESSMENT_ACTION = r"(?:INTERRUPT|LATER|ARCHIVE|(?-i:[ABC]))"
_ASSESSMENT_HIDDEN_FIELD_PATTERN = re.compile(
    r"\b(?:busy|busyness|deadline|urgency|affinity)\b|"
    r"\binterruption(?:[\s_-]+)filter\b",
    re.IGNORECASE,
)
_ASSESSMENT_DIRECTIVE_PATTERN = re.compile(
    r"\b(?:copy\s+exactly|correct\s+(?:output|answer|route|choice)|"
    r"do\s+not\s+add\s+(?:an?\s+)?explanation|output\s+only|"
    r"respond\s+(?:only\s+)?with|return\s+(?:only\s+)?"
    + _ASSESSMENT_ACTION
    + r")\b",
    re.IGNORECASE,
)
_ASSESSMENT_ANSWER_PATTERN = re.compile(
    r"(?:^|\b)(?:answer|choose|select|pick|recommend)\s+"
    r"(?:is\s+|route\s+|option\s+)?"
    + _ASSESSMENT_ACTION
    + r"\b|"
    r"\b(?:final|correct|best|preferred|recommended)\s+"
    r"(?:route|answer|output|choice|option)\s*(?:is|:)?\s*"
    + _ASSESSMENT_ACTION
    + r"\b|^[abc][.!]?$",
    re.IGNORECASE,
)
_ASSESSMENT_PREFERENCE_PATTERN = re.compile(
    r"\b"
    + _ASSESSMENT_ACTION
    + r"\s+(?:(?:is|seems|appears|looks)\s+"
    r"(?:the\s+)?(?:best|correct|preferred|recommended|most\s+likely|"
    r"plausible|appropriate|suitable|right)|"
    r"(?:may|could)\s+(?:be\s+)?(?:the\s+)?(?:most\s+likely|plausible|"
    r"appropriate|suitable|right|fit|work))\b|"
    r"\b(?:evidence|callback|notification|assessment|behavior)\s+"
    r"(?:strongly\s+)?(?:favors|favours|points\s+to|indicates|recommends)\s+"
    r"(?:route\s+|option\s+)?"
    + _ASSESSMENT_ACTION
    + r"\b",
    re.IGNORECASE,
)
_ASSESSMENT_DECISIVE_PATTERN = re.compile(
    r"\b(?:best|correct|preferred|recommended|most\s+likely|right)\b",
    re.IGNORECASE,
)


def _assessment_action_mentions(assessment: str) -> set[str]:
    """Return explicit route names/codes without treating the article 'a' as A."""
    mentions = {
        route.upper()
        for route in re.findall(
            r"\b(?:INTERRUPT|LATER|ARCHIVE)\b",
            assessment,
            flags=re.IGNORECASE,
        )
    }
    mentions.update(re.findall(r"\b[ABC]\b", assessment))
    return mentions


def _assessment_crosses_boundary(assessment: str) -> bool:
    """Reject generated text that would widen or force the scoring prompt."""
    forbidden_phrases = (
        "route code",
        "final route",
        "correct route",
        "best route is",
        "gold action",
        "scalar reward",
        "counterfactual",
        "oracle utility",
    )
    lower = assessment.lower()
    if any(phrase in lower for phrase in forbidden_phrases):
        return True
    if _ASSESSMENT_HIDDEN_FIELD_PATTERN.search(assessment):
        return True
    if _ASSESSMENT_DIRECTIVE_PATTERN.search(assessment):
        return True
    if _ASSESSMENT_ANSWER_PATTERN.search(assessment):
        return True
    preference = _ASSESSMENT_PREFERENCE_PATTERN.search(assessment)
    if preference is None:
        return False
    mentions = _assessment_action_mentions(assessment)
    return len(mentions) < 2 or bool(
        _ASSESSMENT_DECISIVE_PATTERN.search(preference.group(0))
    )


@dataclass(frozen=True)
class OnlineSDFTSettings:
    """Configured LoRA, causal-update, and teacher settings for Online-SDFT."""

    learning_rate: float = SDFT_LR
    replay_size: int = SDFT_REPLAY_SIZE
    replay_prompt_examples: int = 0
    batch_size: int = SDFT_BATCH_SIZE
    update_steps: int = SDFT_UPDATE_STEPS
    warmup_examples: int = SDFT_WARMUP_EXAMPLES
    lora_rank: int = LORA_R
    lora_alpha: int = LORA_ALPHA
    lora_dropout: float = LORA_DROPOUT
    lora_target_modules: tuple[str, ...] = LORA_TARGET_MODULES
    lora_layers_to_transform: tuple[int, ...] | None = LORA_LAYERS_TO_TRANSFORM
    lora_a_learning_rate_scale: float = 1.0
    lm_head_lora_a_learning_rate_scale: float | None = None
    optimizer_weight_decay: float = SDFT_OPTIMIZER_WEIGHT_DECAY
    optimizer_beta1: float = 0.9
    max_grad_norm: float = SDFT_MAX_GRAD_NORM
    lm_head_learning_rate: float | None = None
    ambiguous_replay_group_weight: float = 0.05
    teacher_temperature: float = TEACHER_TEMPERATURE
    reasoning_tokens: int = 0
    target_mode: str = "causal_fusion"
    reliable_teacher_weight: float = 0.05
    reliable_decision_weight: float = 0.05
    reliable_behavior_weight: float = 0.90
    ambiguous_teacher_weight: float = 0.0
    ambiguous_decision_weight: float = 1.0
    ambiguous_behavior_weight: float = 0.0
    ambiguous_projection: str = "causal_support"
    replay_strategy: str = "selection_balanced"
    replay_recency_half_life: float | None = None
    ambiguous_update_mode: str = "immediate"
    force_newest_every_step: bool = True
    base_kl_weight: float = 0.0
    behavior_mode: str = "epsilon_greedy"
    behavior_epsilon: float = EXPLORATION_EPSILON
    behavior_epsilon_half_life: float | None = None
    exploration_taper_start_step: int | None = None
    exploration_taper_half_life: float | None = None
    archive_probe_mix: float = 0.0
    archive_policy_min_feedback: float = 0.0
    interrupt_probe_mix: float = 0.0
    interrupt_probe_half_life: float | None = None
    interrupt_probe_max_confidence: float = 1.0
    propensity_weight_mode: str = "none"
    propensity_weight_cap: float = 4.0

    def __post_init__(self) -> None:
        if self.learning_rate <= 0 or self.teacher_temperature <= 0:
            raise ValueError("learning rate and teacher temperature must be positive")
        integer_fields = (self.lora_rank, self.lora_alpha)
        if any(
            isinstance(value, bool) or not isinstance(value, int) or value <= 0
            for value in integer_fields
        ):
            raise ValueError("LoRA rank and alpha must be positive integers")
        if not 0.0 <= self.lora_dropout < 1.0:
            raise ValueError("LoRA dropout must be in [0, 1)")
        raw_targets = self.lora_target_modules
        targets = (raw_targets,) if isinstance(raw_targets, str) else tuple(raw_targets)
        if not targets or any(not isinstance(name, str) or not name for name in targets):
            raise ValueError("LoRA target modules must be non-empty strings")
        if len(set(targets)) != len(targets):
            raise ValueError("LoRA target modules must be unique")
        object.__setattr__(self, "lora_target_modules", targets)
        raw_layers = self.lora_layers_to_transform
        if raw_layers is not None:
            layers = (raw_layers,) if isinstance(raw_layers, int) else tuple(raw_layers)
            if not layers:
                raise ValueError("LoRA layer indices cannot be empty")
            if any(
                isinstance(layer, bool)
                or not isinstance(layer, int)
                or layer < 0
                for layer in layers
            ):
                raise ValueError("LoRA layer indices must be non-negative integers")
            if len(set(layers)) != len(layers):
                raise ValueError("LoRA layer indices must be unique")
            object.__setattr__(self, "lora_layers_to_transform", layers)
        if self.optimizer_weight_decay < 0:
            raise ValueError("LoRA optimizer weight decay cannot be negative")
        if (
            isinstance(self.lora_a_learning_rate_scale, bool)
            or not np.isfinite(self.lora_a_learning_rate_scale)
            or not 0.0 <= self.lora_a_learning_rate_scale <= 1.0
        ):
            raise ValueError(
                "LoRA A learning-rate scale must be finite and in [0, 1]"
            )
        if self.lm_head_lora_a_learning_rate_scale is not None and (
            isinstance(self.lm_head_lora_a_learning_rate_scale, bool)
            or not np.isfinite(self.lm_head_lora_a_learning_rate_scale)
            or not 0.0 <= self.lm_head_lora_a_learning_rate_scale <= 1.0
        ):
            raise ValueError(
                "LM-head LoRA A learning-rate scale must be finite and in [0, 1]"
            )
        if (
            self.lm_head_lora_a_learning_rate_scale is not None
            and "lm_head" not in targets
        ):
            raise ValueError(
                "LM-head LoRA A learning-rate scale requires an LM-head adapter"
            )
        if (
            isinstance(self.optimizer_beta1, bool)
            or not np.isfinite(self.optimizer_beta1)
            or not 0.0 <= self.optimizer_beta1 < 1.0
        ):
            raise ValueError("LoRA optimizer beta1 must be finite and in [0, 1)")
        if self.max_grad_norm <= 0:
            raise ValueError("LoRA maximum gradient norm must be positive")
        if self.lm_head_learning_rate is not None and (
            isinstance(self.lm_head_learning_rate, bool)
            or not np.isfinite(self.lm_head_learning_rate)
            or self.lm_head_learning_rate <= 0.0
        ):
            raise ValueError("LM-head learning rate must be finite and positive")
        if (
            self.lm_head_learning_rate is not None
            and "lm_head" not in targets
        ):
            raise ValueError("LM-head learning rate requires an LM-head adapter")
        if not 0.0 < self.ambiguous_replay_group_weight <= 1.0:
            raise ValueError(
                "ambiguous replay group weight must be in (0, 1]"
            )
        if min(
            self.replay_size,
            self.batch_size,
            self.update_steps,
            self.warmup_examples,
        ) <= 0:
            raise ValueError(
                "replay, batch, update, and warmup counts must be positive"
            )
        if self.batch_size > self.replay_size:
            raise ValueError("SDFT batch size cannot exceed replay size")
        if not 0 <= self.replay_prompt_examples <= self.replay_size:
            raise ValueError(
                "SDFT replay prompt examples must be between zero and replay size"
            )
        if self.warmup_examples > self.replay_size:
            raise ValueError("SDFT warmup cannot exceed replay size")
        if self.reasoning_tokens < 0:
            raise ValueError("reasoning token count cannot be negative")
        if self.reasoning_tokens > 64:
            raise ValueError("reasoning token count cannot exceed 64")
        if self.target_mode not in {
            "teacher_only",
            "causal_fusion",
            "support_likelihood",
        }:
            raise ValueError("unknown SDFT target mode")
        if self.ambiguous_projection not in {"none", "causal_support"}:
            raise ValueError("unknown ambiguous projection mode")
        if self.replay_strategy not in {
            "uniform",
            "selection_balanced",
        }:
            raise ValueError("unknown SDFT replay strategy")
        if self.replay_recency_half_life is not None:
            if (
                isinstance(self.replay_recency_half_life, bool)
                or not isinstance(self.replay_recency_half_life, Real)
                or not np.isfinite(self.replay_recency_half_life)
                or self.replay_recency_half_life <= 0.0
            ):
                raise ValueError(
                    "replay recency half-life must be finite and positive"
                )
            if self.replay_strategy != "selection_balanced":
                raise ValueError(
                    "replay recency requires selection-balanced replay"
                )
        if self.ambiguous_update_mode not in {"immediate", "skip", "defer"}:
            raise ValueError("unknown ambiguous update mode")
        if not isinstance(self.force_newest_every_step, bool):
            raise ValueError("force-newest setting must be boolean")
        if self.base_kl_weight < 0:
            raise ValueError("fixed-base KL weight cannot be negative")
        if self.base_kl_weight and self.target_mode == "support_likelihood":
            raise ValueError(
                "fixed-base KL is incompatible with support likelihood"
            )
        if self.base_kl_weight and self.replay_prompt_examples:
            raise ValueError(
                "fixed-base KL requires pure parametric replay prompts"
            )
        if self.behavior_mode not in {
            "epsilon_greedy",
            "policy_sampling",
            "archive_policy_sampling",
            "archive_uniform_probe",
            "archive_policy_feedback_floor",
            "uncertainty_interrupt_probe",
        }:
            raise ValueError("unknown Online-SDFT behavior mode")
        if (
            isinstance(self.behavior_epsilon, bool)
            or not isinstance(self.behavior_epsilon, Real)
            or not np.isfinite(self.behavior_epsilon)
            or not 0.0 <= self.behavior_epsilon <= 1.0
        ):
            raise ValueError(
                "Online-SDFT behavior epsilon must be finite and in [0, 1]"
            )
        if self.behavior_epsilon_half_life is not None and (
            isinstance(self.behavior_epsilon_half_life, bool)
            or not isinstance(self.behavior_epsilon_half_life, Real)
            or not np.isfinite(self.behavior_epsilon_half_life)
            or self.behavior_epsilon_half_life <= 0.0
        ):
            raise ValueError(
                "behavior epsilon half-life must be finite and positive"
            )
        if self.exploration_taper_start_step is not None and (
            isinstance(self.exploration_taper_start_step, bool)
            or not isinstance(self.exploration_taper_start_step, int)
            or self.exploration_taper_start_step <= 0
        ):
            raise ValueError(
                "exploration taper start step must be a positive integer"
            )
        if self.exploration_taper_half_life is not None and (
            isinstance(self.exploration_taper_half_life, bool)
            or not isinstance(self.exploration_taper_half_life, Real)
            or not np.isfinite(self.exploration_taper_half_life)
            or self.exploration_taper_half_life <= 0.0
        ):
            raise ValueError(
                "exploration taper half-life must be finite and positive"
            )
        if (
            self.exploration_taper_start_step is None
            or self.exploration_taper_half_life is None
        ) and (
            self.exploration_taper_start_step is not None
            or self.exploration_taper_half_life is not None
        ):
            raise ValueError(
                "exploration taper start step and half-life must be configured "
                "together"
            )
        if (
            self.exploration_taper_start_step is not None
            and self.behavior_mode
            not in {
                "epsilon_greedy",
                "archive_uniform_probe",
                "uncertainty_interrupt_probe",
            }
        ):
            raise ValueError(
                "exploration taper requires an epsilon/probe behavior mode"
            )
        if (
            isinstance(self.archive_probe_mix, bool)
            or not np.isfinite(self.archive_probe_mix)
            or not 0.0 <= self.archive_probe_mix <= 1.0
        ):
            raise ValueError("archive probe mix must be a finite number in [0, 1]")
        if self.behavior_mode == "archive_uniform_probe":
            if self.archive_probe_mix == 0.0:
                raise ValueError("archive-uniform probing requires a positive mix")
        elif self.archive_probe_mix != 0.0:
            raise ValueError("archive probe mix requires archive-uniform probing")
        if (
            isinstance(self.archive_policy_min_feedback, bool)
            or not isinstance(self.archive_policy_min_feedback, Real)
            or not np.isfinite(self.archive_policy_min_feedback)
            or not 0.0 <= self.archive_policy_min_feedback <= 1.0
        ):
            raise ValueError(
                "archive policy minimum feedback must be a finite number in [0, 1]"
            )
        if self.behavior_mode == "archive_policy_feedback_floor":
            if self.archive_policy_min_feedback == 0.0:
                raise ValueError(
                    "archive policy feedback-floor behavior requires a positive "
                    "minimum"
                )
        elif self.archive_policy_min_feedback != 0.0:
            raise ValueError(
                "archive policy minimum feedback requires feedback-floor behavior"
            )
        if (
            isinstance(self.interrupt_probe_mix, bool)
            or not isinstance(self.interrupt_probe_mix, Real)
            or not np.isfinite(self.interrupt_probe_mix)
            or not 0.0 <= self.interrupt_probe_mix < 1.0
        ):
            raise ValueError(
                "interrupt probe mix must be finite and in [0, 1)"
            )
        if self.interrupt_probe_half_life is not None and (
            isinstance(self.interrupt_probe_half_life, bool)
            or not isinstance(self.interrupt_probe_half_life, Real)
            or not np.isfinite(self.interrupt_probe_half_life)
            or self.interrupt_probe_half_life <= 0.0
        ):
            raise ValueError(
                "interrupt probe half-life must be finite and positive"
            )
        minimum_confidence = 1.0 / len(ACTIONS)
        if (
            isinstance(self.interrupt_probe_max_confidence, bool)
            or not isinstance(self.interrupt_probe_max_confidence, Real)
            or not np.isfinite(self.interrupt_probe_max_confidence)
            or not minimum_confidence
            <= self.interrupt_probe_max_confidence
            <= 1.0
        ):
            raise ValueError(
                "interrupt probe maximum confidence must be finite and in "
                f"[{minimum_confidence}, 1]"
            )
        if self.behavior_mode == "uncertainty_interrupt_probe":
            if self.interrupt_probe_mix == 0.0:
                raise ValueError(
                    "uncertainty interrupt probing requires a positive mix"
                )
            if self.interrupt_probe_half_life is None:
                raise ValueError(
                    "uncertainty interrupt probing requires a half-life"
                )
        elif (
            self.interrupt_probe_mix != 0.0
            or self.interrupt_probe_half_life is not None
            or self.interrupt_probe_max_confidence != 1.0
        ):
            raise ValueError(
                "interrupt probe settings require uncertainty interrupt probing"
            )
        if self.propensity_weight_mode not in {
            "none",
            "feedback_surface_snips",
        }:
            raise ValueError("unknown SDFT propensity-weight mode")
        if (
            isinstance(self.propensity_weight_cap, bool)
            or not isinstance(self.propensity_weight_cap, Real)
            or not np.isfinite(self.propensity_weight_cap)
            or self.propensity_weight_cap < 1.0
        ):
            raise ValueError(
                "SDFT propensity-weight cap must be finite and at least one"
            )
        if (
            self.propensity_weight_mode == "none"
            and self.propensity_weight_cap != 4.0
        ):
            raise ValueError(
                "a custom propensity-weight cap requires propensity weighting"
            )
        reliable_weights = (
            self.reliable_teacher_weight,
            self.reliable_decision_weight,
            self.reliable_behavior_weight,
        )
        ambiguous_weights = (
            self.ambiguous_teacher_weight,
            self.ambiguous_decision_weight,
            self.ambiguous_behavior_weight,
        )
        profiles = (reliable_weights, ambiguous_weights)
        if any(weight < 0 for weights in profiles for weight in weights):
            raise ValueError("SDFT target weights cannot be negative")
        if self.target_mode == "teacher_only":
            if not all(weights == (1.0, 0.0, 0.0) for weights in profiles):
                raise ValueError("teacher-only mode requires its canonical weights")
        elif self.target_mode == "causal_fusion":
            if not all(np.isclose(sum(weights), 1.0) for weights in profiles):
                raise ValueError("causal-fusion target weights must sum to one")
            if self.reliable_behavior_weight < self.ambiguous_behavior_weight:
                raise ValueError(
                    "reliable callbacks cannot receive less causal weight than "
                    "ambiguous callbacks"
                )
        elif any(weight != 0.0 for weights in profiles for weight in weights):
            raise ValueError(
                "support-likelihood mode requires zero unused fusion weights"
            )

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass(frozen=True)
class REINFORCESettings:
    """Configured factual-reward LoRA settings for REINFORCE.

    ``reward_outcome_map`` is learner-only shaping keyed solely by the matured
    callback's factual outcome. It never changes the shared observable-reward
    metric. An empty map instead consumes the environment's scalar reward.
    """

    learning_rate: float = REINFORCE_LR
    batch_size: int = REINFORCE_BATCH_SIZE
    baseline_step: float = REINFORCE_BASELINE_STEP
    entropy_coef: float = REINFORCE_ENTROPY_COEF
    max_grad_norm: float = REINFORCE_MAX_GRAD_NORM
    reward_outcome_map: (
        Mapping[str, float] | tuple[tuple[str, float], ...]
    ) = ()

    def __post_init__(self) -> None:
        finite_scalars = (
            self.learning_rate,
            self.baseline_step,
            self.entropy_coef,
            self.max_grad_norm,
        )
        if not np.isfinite(finite_scalars).all():
            raise ValueError("REINFORCE scalar settings must be finite")
        if self.learning_rate <= 0:
            raise ValueError("REINFORCE learning rate must be positive")
        if (
            isinstance(self.batch_size, bool)
            or not isinstance(self.batch_size, int)
            or self.batch_size <= 0
        ):
            raise ValueError("REINFORCE batch size must be positive")
        if not 0.0 <= self.baseline_step <= 1.0:
            raise ValueError("REINFORCE baseline step must be in [0, 1]")
        if self.entropy_coef < 0:
            raise ValueError("REINFORCE entropy coefficient cannot be negative")
        if self.max_grad_norm <= 0:
            raise ValueError("REINFORCE maximum gradient norm must be positive")

        raw_items = (
            self.reward_outcome_map.items()
            if isinstance(self.reward_outcome_map, Mapping)
            else self.reward_outcome_map
        )
        normalized_items: list[tuple[str, float]] = []
        seen_outcomes: set[str] = set()
        for outcome, reward in raw_items:
            if not isinstance(outcome, str) or not outcome:
                raise ValueError("REINFORCE reward outcomes must be non-empty strings")
            if outcome in seen_outcomes:
                raise ValueError("REINFORCE reward outcomes must be unique")
            numeric_reward = float(reward)
            if not np.isfinite(numeric_reward):
                raise ValueError("REINFORCE outcome rewards must be finite")
            seen_outcomes.add(outcome)
            normalized_items.append((outcome, numeric_reward))
        object.__setattr__(self, "reward_outcome_map", tuple(normalized_items))

    def to_dict(self) -> dict:
        values = asdict(self)
        values["reward_outcome_map"] = dict(self.reward_outcome_map)
        return values


DEFAULT_SDFT_SETTINGS = OnlineSDFTSettings(
    replay_size=64,
    replay_recency_half_life=32.0,
    behavior_mode="uncertainty_interrupt_probe",
    behavior_epsilon=0.02,
    exploration_taper_start_step=160,
    exploration_taper_half_life=5.0,
    interrupt_probe_mix=0.15,
    interrupt_probe_half_life=80.0,
    interrupt_probe_max_confidence=0.60,
)
DEFAULT_REINFORCE_SETTINGS = REINFORCESettings(
    reward_outcome_map=REINFORCE_TRAINING_OUTCOME_REWARDS,
)
DEFAULT_RFT_STUDENT_SETTINGS = OnlineSDFTSettings(
    learning_rate=RFT_LR,
)


@dataclass(frozen=True)
class RFTSettings:
    """Isolated proposal and LoRA-student settings for causal online RFT."""

    student_settings: OnlineSDFTSettings = DEFAULT_RFT_STUDENT_SETTINGS
    candidate_count: int = RFT_CANDIDATE_COUNT
    sampling_temperature: float = RFT_SAMPLING_TEMPERATURE
    sampling_mode: str = RFT_SAMPLING_MODE

    def __post_init__(self) -> None:
        if not isinstance(self.student_settings, OnlineSDFTSettings):
            raise TypeError("RFT student settings must be OnlineSDFTSettings")
        if (
            isinstance(self.candidate_count, bool)
            or not isinstance(self.candidate_count, int)
            or self.candidate_count != 1
        ):
            raise ValueError(
                "causal online RFT currently requires candidate_count=1"
            )
        if isinstance(self.sampling_temperature, bool):
            raise ValueError(
                "RFT sampling temperature must be positive and finite"
            )
        temperature = float(self.sampling_temperature)
        if not np.isfinite(temperature) or temperature <= 0.0:
            raise ValueError(
                "RFT sampling temperature must be positive and finite"
            )
        object.__setattr__(self, "sampling_temperature", temperature)
        if self.sampling_mode != "categorical":
            raise ValueError("causal online RFT requires categorical sampling")

    def to_dict(self) -> dict[str, Any]:
        return {
            "student_settings": self.student_settings.to_dict(),
            "candidate_count": self.candidate_count,
            "sampling_temperature": self.sampling_temperature,
            "sampling_mode": self.sampling_mode,
        }


DEFAULT_RFT_SETTINGS = RFTSettings()


class StudentPolicy(Protocol):
    """Minimal policy interface shared by the real LFM and test doubles."""

    def start_run(self, learning_rate: float | None) -> None: ...

    def probs(
        self,
        context: str,
        examples: list[dict] | None = None,
    ) -> np.ndarray: ...

    def teacher_probs(
        self,
        observation: TeacherObservation,
        examples: list[dict] | None = None,
    ) -> np.ndarray: ...

    def update(
        self,
        batch: list[tuple[str, np.ndarray]],
        sample_weights: np.ndarray | None = None,
    ) -> float: ...

    def base_probs(self, context: str) -> np.ndarray: ...

    def update_support(
        self,
        batch: list[tuple[str, np.ndarray]],
        sample_weights: np.ndarray | None = None,
    ) -> float: ...

    def reinforce_update(
        self,
        batch: list[tuple[str, int, float]],
        entropy_coef: float,
        max_grad_norm: float,
    ) -> float: ...


class LiquidLLMPolicy:
    """LFM2.5 student whose A/B/C next-token logits define route scores."""

    def __init__(
        self,
        model_id: str = MODEL_ID,
        device: str = "auto",
        local_files_only: bool = False,
        prompt_style: str = PROMPT_STYLE,
        sdft_settings: OnlineSDFTSettings = DEFAULT_SDFT_SETTINGS,
    ):
        import torch
        from peft import LoraConfig, get_peft_model
        from transformers import AutoTokenizer, Lfm2ForCausalLM

        self.torch = torch
        self.model_id = model_id
        self.sdft_settings = sdft_settings
        if prompt_style not in PROMPT_STYLES:
            raise ValueError(
                f"prompt_style must be one of {PROMPT_STYLES}, got {prompt_style!r}"
            )
        self.prompt_style = prompt_style
        self.teacher_temperature = TEACHER_TEMPERATURE
        self.teacher_reasoning_tokens = 0
        self.last_teacher_assessment: str | None = None
        if device == "auto":
            device = (
                "cuda"
                if torch.cuda.is_available()
                else "mps"
                if torch.backends.mps.is_available()
                else "cpu"
            )
        self.device = torch.device(device)
        # T4-class Colab GPUs are optimized for FP16 and do not accelerate
        # BF16. Keep CPU/MPS in FP32 so the published CPU protocol is
        # unchanged, while CUDA runs use tensor cores for practical reruns.
        self.model_dtype = (
            torch.float16
            if self.device.type == "cuda"
            else torch.float32
        )
        torch.manual_seed(0)

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            local_files_only=local_files_only,
        )
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        base_model = Lfm2ForCausalLM.from_pretrained(
            model_id,
            local_files_only=local_files_only,
            dtype=self.model_dtype,
        )
        adapter_config = LoraConfig(
            r=sdft_settings.lora_rank,
            lora_alpha=sdft_settings.lora_alpha,
            lora_dropout=sdft_settings.lora_dropout,
            target_modules=list(sdft_settings.lora_target_modules),
            layers_to_transform=(
                None
                if sdft_settings.lora_layers_to_transform is None
                else list(sdft_settings.lora_layers_to_transform)
            ),
            layers_pattern=(
                None
                if sdft_settings.lora_layers_to_transform is None
                else "layers"
            ),
            bias="none",
            task_type="CAUSAL_LM",
            init_lora_weights=True,
            ensure_weight_tying=False,
        )
        self.model = get_peft_model(base_model, adapter_config).to(self.device)
        self.model.eval()
        self.model.config.use_cache = False

        self.action_token_ids = []
        for code in ACTION_CODES:
            token_ids = self.tokenizer.encode(code, add_special_tokens=False)
            if len(token_ids) != 1:
                raise ValueError(
                    f"action code {code!r} is not one token: {token_ids}"
                )
            self.action_token_ids.append(token_ids[0])

        self._initial_adapter = {
            name: parameter.detach().cpu().clone()
            for name, parameter in self.model.named_parameters()
            if parameter.requires_grad
        }
        self.optimizer: Any | None = None
        self._restore_adapter_trainability()

    @property
    def trainable_parameters(self) -> int:
        """Return stable optimizer-visible adapter capacity for this setting."""
        body_a_scale = self.sdft_settings.lora_a_learning_rate_scale
        head_a_scale = self.sdft_settings.lm_head_lora_a_learning_rate_scale
        lm_head_parameter_ids = (
            {
                id(parameter)
                for parameter in self.model.get_output_embeddings().parameters()
            }
            if head_a_scale is not None
            else set()
        )
        return sum(
            parameter.numel()
            for name, parameter in self.model.named_parameters()
            if name in self._initial_adapter
            and not (
                "lora_A." in name
                and (
                    head_a_scale
                    if id(parameter) in lm_head_parameter_ids
                    else body_a_scale
                )
                == 0.0
            )
        )

    def start_run(self, learning_rate: float | None) -> None:
        """Reset the one shared adapter before every method and seed."""
        for name, parameter in self.model.named_parameters():
            if name not in self._initial_adapter:
                continue
            parameter.data.copy_(self._initial_adapter[name].to(self.device))
        self._restore_adapter_trainability()
        self.optimizer = None
        if learning_rate is not None:
            trainable = [
                (name, parameter)
                for name, parameter in self.model.named_parameters()
                if parameter.requires_grad
            ]
            optimizer_parameters: Any
            body_a_scale = self.sdft_settings.lora_a_learning_rate_scale
            head_a_scale = self.sdft_settings.lm_head_lora_a_learning_rate_scale
            needs_lm_head_identity = (
                self.sdft_settings.lm_head_learning_rate is not None
                or head_a_scale is not None
            )
            lm_head_parameter_ids: set[int] = set()
            if needs_lm_head_identity:
                output_module = self.model.get_output_embeddings()
                lm_head_parameter_ids = {
                    id(parameter) for parameter in output_module.parameters()
                }
                if not lm_head_parameter_ids:
                    raise RuntimeError(
                        "configured LM-head adapter parameters are missing"
                    )
            if (
                self.sdft_settings.lm_head_learning_rate is None
                and body_a_scale == 1.0
                and head_a_scale in {None, 1.0}
            ):
                optimizer_parameters = [parameter for _, parameter in trainable]
            else:
                parameter_groups: dict[float, list[Any]] = {}
                for name, parameter in trainable:
                    parameter_lr = learning_rate
                    is_lm_head = id(parameter) in lm_head_parameter_ids
                    if (
                        self.sdft_settings.lm_head_learning_rate is not None
                        and is_lm_head
                    ):
                        parameter_lr = self.sdft_settings.lm_head_learning_rate
                    if "lora_A." in name:
                        parameter_lr *= (
                            head_a_scale
                            if is_lm_head and head_a_scale is not None
                            else body_a_scale
                        )
                    parameter_groups.setdefault(parameter_lr, []).append(
                        parameter
                    )
                optimizer_parameters = [
                    {"params": parameters, "lr": parameter_lr}
                    for parameter_lr, parameters in parameter_groups.items()
                ]
            self.optimizer = self.torch.optim.AdamW(
                optimizer_parameters,
                lr=learning_rate,
                betas=(self.sdft_settings.optimizer_beta1, 0.999),
                weight_decay=self.sdft_settings.optimizer_weight_decay,
            )
        self.model.eval()

    def _restore_adapter_trainability(self) -> None:
        """Reapply factor freezing after PEFT's adapter-disabled context."""
        body_a_scale = self.sdft_settings.lora_a_learning_rate_scale
        head_a_scale = self.sdft_settings.lm_head_lora_a_learning_rate_scale
        lm_head_parameter_ids = (
            {
                id(parameter)
                for parameter in self.model.get_output_embeddings().parameters()
            }
            if head_a_scale is not None
            else set()
        )
        for name, parameter in self.model.named_parameters():
            if name in self._initial_adapter:
                a_scale = (
                    head_a_scale
                    if id(parameter) in lm_head_parameter_ids
                    else body_a_scale
                )
                parameter.requires_grad_(
                    not (a_scale == 0.0 and "lora_A." in name)
                )

    def configure_online_sdft(self, settings: OnlineSDFTSettings) -> None:
        """Select teacher settings for the adapter-disabled shared model."""
        architecture_fields = (
            "lora_rank",
            "lora_alpha",
            "lora_dropout",
            "lora_target_modules",
            "lora_layers_to_transform",
        )
        if any(
            getattr(settings, field) != getattr(self.sdft_settings, field)
            for field in architecture_fields
        ):
            raise ValueError(
                "LiquidLLMPolicy LoRA architecture must match Online-SDFT settings"
            )
        self.sdft_settings = settings
        self.teacher_temperature = settings.teacher_temperature
        self.teacher_reasoning_tokens = settings.reasoning_tokens
        self.last_teacher_assessment = None

    def render_prompt(
        self,
        context: str,
        examples: list[dict] | None = None,
    ) -> str:
        messages = self.student_messages(context, examples)
        self._assert_role_alternation(messages)
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    def student_messages(
        self,
        context: str,
        examples: list[dict] | None = None,
    ) -> list[dict]:
        """Build a valid chat sequence for one serving-time decision."""
        history = list(examples or [])

        if self.prompt_style == "causal_demos":
            labeled_history = [
                row
                for row in history
                if row.get("observed_user_selection") in ACTIONS
            ]
            if not labeled_history:
                return [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {
                        "role": "user",
                        "content": self._student_query(context, False),
                    },
                ]
            messages = [
                {
                    "role": "system",
                    "content": self.student_system_prompt(
                        has_examples=bool(labeled_history)
                    ),
                }
            ]
            for row in labeled_history:
                selection = row["observed_user_selection"]
                code = ACTION_CODES[ACTIONS.index(selection)]
                messages.extend(
                    [
                        {
                            "role": "user",
                            "content": (
                                f"Past notification: {row['context']}\nRoute:"
                            ),
                        },
                        {"role": "assistant", "content": code},
                    ]
                )
            messages.append(
                {
                    "role": "user",
                    "content": f"Current notification: {context}\nRoute:",
                }
            )
            return messages

        if self.prompt_style == "legacy":
            lines = []
            for index, row in enumerate(history, start=1):
                selection = row.get("observed_user_selection", "UNKNOWN")
                lines.append(f"example {index} notification: {row['context']}")
                lines.append(
                    f"example {index} completed interaction: "
                    f"executed={row.get('executed_action', 'UNKNOWN')}; "
                    f"eventual_user_action="
                    f"{row.get('eventual_user_action', 'UNKNOWN')}; "
                    f"observed_selection={selection}"
                )
                if selection in ACTIONS:
                    code = ACTION_CODES[ACTIONS.index(selection)]
                    lines.append(f"example {index} observed route: {code}")
                else:
                    lines.append(f"example {index} route: UNLABELED")
            if lines:
                lines.append(
                    "Treat these as user-specific evidence, not universal rules."
                )
            lines.extend(
                [f"current notification: {context}", "current route:"]
            )
            return [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": "\n".join(lines)},
            ]

        messages = [
            {
                "role": "system",
                "content": self.student_system_prompt(
                    has_examples=bool(history)
                ),
            }
        ]
        query = self._student_query(context, bool(history))
        blocks = [
            self._structured_history_block(context, index, row)
            for index, row in enumerate(history, start=1)
        ]
        content = ""
        if blocks:
            content = (
                "Past completed interactions:\n"
                + "\n".join(blocks)
                + "\nUNKNOWN is unlabeled.\n\n"
            )
        messages.append({"role": "user", "content": content + query})
        return messages

    def _structured_history_block(
        self,
        context: str,
        index: int,
        row: dict,
    ) -> str:
        selection = row.get("observed_user_selection", "UNKNOWN")
        callback = FactualCallback(
            action_taken=row.get("executed_action", "UNKNOWN"),
            outcome=row.get("eventual_user_action", "UNKNOWN"),
            observed_user_selection=selection,
            delay_minutes=int(row.get("delay_minutes", 0)),
        )
        label = "This interaction is unlabeled."
        if selection in ACTIONS:
            code = ACTION_CODES[ACTIONS.index(selection)]
            label = f"Its observed route was {code} for {selection}."
        return (
            f"{index}. "
            + self._history_relevance(context, row)
            + row["context"]
            + " "
            + narrative_mobile_teacher_evidence(callback)
            + " "
            + label
        )

    def _history_relevance(self, context: str, row: dict) -> str:
        if self.prompt_style != "history_relevance":
            return ""
        current_category = self._context_field(context, "category")
        current_regime = self._context_field(context, "regime")
        example_category = self._context_field(row["context"], "category")
        example_regime = self._context_field(row["context"], "regime")
        tag = (
            "EXACT_MATCH"
            if current_category == example_category
            and current_regime == example_regime
            else "DIFFERENT_CONTEXT"
        )
        return f"Relevance: {tag}.\n"

    def _student_query(self, context: str, has_examples: bool) -> str:
        query = f"Notification: {context}\nRoute:"
        if has_examples and self.prompt_style in {
            "history_guarded",
            "history_relevance",
        }:
            query = (
                "NEW DECISION: do not repeat the last route. Ignore past "
                "labels whose category or regime differs. If none matches "
                "both fields, decide as if no history were shown.\n" + query
            )
        return query

    @staticmethod
    def _assert_role_alternation(messages: list[dict]) -> None:
        """Reject malformed chat histories before tokenization."""
        roles = [message["role"] for message in messages]
        if not roles or roles[0] != "system":
            raise ValueError("chat must start with exactly one system message")
        expected = "user"
        for role in roles[1:]:
            if role != expected:
                raise ValueError(
                    f"chat roles must alternate after system, got {roles}"
                )
            expected = "assistant" if role == "user" else "user"
        if roles[-1] != "user":
            raise ValueError("chat must end with the current user query")

    @staticmethod
    def _context_field(context: str, name: str) -> str | None:
        """Read category/regime from prose, with legacy field compatibility."""
        prose_patterns = {
            "category": r"This (?:is a )?([\w-]+) notification",
            "regime": r"during the ([\w-]+) period",
        }
        if name in prose_patterns:
            match = re.search(prose_patterns[name], context)
            if match:
                return match.group(1)
        prefix = f"{name}="
        for part in context.replace("\n", ";").split(";"):
            stripped = part.strip()
            if stripped.startswith("Metadata: "):
                stripped = stripped.removeprefix("Metadata: ").strip()
            if stripped.startswith(prefix):
                return stripped[len(prefix):].strip()
        return None

    def student_system_prompt(self, has_examples: bool = False) -> str:
        """Return the serving instruction for the configured prompt style."""
        history_only_styles = {
            "causal_demos",
            "interaction_match",
            "history_guarded",
            "history_relevance",
        }
        if self.prompt_style in history_only_styles and not has_examples:
            return SYSTEM_PROMPT
        suffix = INTERRUPT_PROMPT_SUFFIXES.get(self.prompt_style)
        if suffix is None:
            return SYSTEM_PROMPT
        return f"{SYSTEM_PROMPT}\n{suffix}"

    def render_teacher_prompt(
        self,
        observation: TeacherObservation,
        examples: list[dict] | None = None,
        assessment: str | None = None,
    ) -> str:
        """Render one completed, phone-observable trajectory for hindsight."""
        messages = self.teacher_messages(observation, examples, assessment)
        self._assert_role_alternation(messages)
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    def teacher_messages(
        self,
        observation: TeacherObservation,
        examples: list[dict] | None = None,
        assessment: str | None = None,
    ) -> list[dict]:
        """Build a neutral hindsight prompt without turning evidence into a label."""
        if examples:
            raise ValueError(
                "the hindsight teacher accepts exactly one completed trajectory"
            )
        assessment_block = ""
        if assessment:
            assessment_block = (
                "\n\nTeacher evidence assessment:\n"
                + assessment.strip()
            )
        user_content = (
            f"Notification:\n{observation.context}\n\n"
            + f"Observed callback:\n{observation.evidence}"
            + assessment_block
            + "\n\nRoute:"
        )
        return [
            {"role": "system", "content": TEACHER_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ]

    def render_teacher_assessment_prompt(
        self,
        observation: TeacherObservation,
    ) -> str:
        """Ask the same model for a bounded, auditable evidence assessment."""
        messages = [
            {"role": "system", "content": TEACHER_REASONING_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Notification:\n{observation.context}\n\n"
                    f"Observed callback:\n{observation.evidence}\n\n"
                    "Assessment:"
                ),
            },
        ]
        self._assert_role_alternation(messages)
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    def _generate_teacher_assessment(
        self,
        observation: TeacherObservation,
    ) -> str | None:
        tokens = int(getattr(self, "teacher_reasoning_tokens", 0))
        if tokens == 0:
            return None
        prompt = self.render_teacher_assessment_prompt(observation)
        encoded = self.tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
        )
        prompt_lengths = encoded["attention_mask"].sum(-1).tolist()
        self.assert_prompt_token_budget(prompt_lengths)
        input_tokens = max(map(int, prompt_lengths))
        if input_tokens + tokens > PROMPT_TOKEN_BUDGET:
            raise ValueError(
                "teacher assessment exceeds the operational token budget: "
                f"input={input_tokens}, requested={tokens}, "
                f"budget={PROMPT_TOKEN_BUDGET}"
            )
        input_length = encoded["input_ids"].shape[1]
        encoded = {
            key: value.to(self.device)
            for key, value in encoded.items()
        }
        try:
            generated = self.model.generate(
                **encoded,
                max_new_tokens=tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
            if self.device.type == "mps":
                self.torch.mps.synchronize()
            generated_ids = generated[0, input_length:].detach().cpu()
            vocab_size = getattr(
                getattr(self.model, "config", None),
                "vocab_size",
                None,
            )
            if vocab_size is None:
                try:
                    vocab_size = len(self.tokenizer)
                except TypeError:
                    vocab_size = None
            if generated_ids.numel() and (
                int(generated_ids.min()) < 0
                or (
                    vocab_size is not None
                    and int(generated_ids.max()) >= int(vocab_size)
                )
            ):
                raise ValueError("teacher generated an invalid token id")
            assessment = self.tokenizer.decode(
                generated_ids.tolist(),
                skip_special_tokens=True,
            ).strip()
        except (OverflowError, RuntimeError, ValueError):
            if self.device.type == "mps":
                self.torch.mps.empty_cache()
            return ASSESSMENT_FALLBACK
        if not assessment:
            return ASSESSMENT_FALLBACK
        assessment = " ".join(assessment.split())
        if _assessment_crosses_boundary(assessment):
            return ASSESSMENT_FALLBACK
        return assessment

    def _action_logits(self, prompts: list[str]):
        encoded = self.tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        )
        if "attention_mask" in encoded:
            token_counts = encoded["attention_mask"].sum(-1).tolist()
        else:
            token_counts = [encoded["input_ids"].shape[-1]] * len(prompts)
        self.assert_prompt_token_budget(token_counts)
        encoded = {
            key: value.to(self.device)
            for key, value in encoded.items()
        }
        # Compute normalization and the distillation loss in FP32 even when
        # the frozen CUDA backbone runs in FP16.
        logits = self.model(**encoded).logits[:, -1, :].float()
        action_ids = self.torch.tensor(
            self.action_token_ids,
            device=self.device,
        )
        return logits.index_select(-1, action_ids)

    @staticmethod
    def assert_prompt_token_budget(token_counts: list[int]) -> None:
        """Enforce the declared operational input budget before every forward."""
        over_budget = [
            int(count)
            for count in token_counts
            if int(count) > PROMPT_TOKEN_BUDGET
        ]
        if over_budget:
            raise ValueError(
                "prompt exceeds the operational token budget: "
                f"max={max(over_budget)}, budget={PROMPT_TOKEN_BUDGET}"
            )

    def probs(
        self,
        context: str,
        examples: list[dict] | None = None,
    ) -> np.ndarray:
        self.model.eval()
        with self.torch.no_grad():
            logits = self._action_logits([self.render_prompt(context, examples)])
            probabilities = self.torch.softmax(
                logits / STUDENT_TEMPERATURE,
                dim=-1,
            )[0]
        values = probabilities.float().cpu().numpy()
        if not np.isfinite(values).all():
            values = np.ones(len(ACTIONS), dtype=float)
        values = np.clip(values, 1e-8, None)
        return values / values.sum()

    def base_probs(self, context: str) -> np.ndarray:
        """Score a student prompt with this same model's adapter disabled."""
        self.model.eval()
        try:
            with self.torch.no_grad(), self.model.disable_adapter():
                logits = self._action_logits([self.render_prompt(context)])
                probabilities = self.torch.softmax(
                    logits / STUDENT_TEMPERATURE,
                    dim=-1,
                )[0]
        finally:
            self._restore_adapter_trainability()
        values = probabilities.float().cpu().numpy()
        if not np.isfinite(values).all():
            values = np.ones(len(ACTIONS), dtype=float)
        values = np.clip(values, 1e-8, None)
        return values / values.sum()

    def teacher_probs(
        self,
        observation: TeacherObservation,
        examples: list[dict] | None = None,
    ) -> np.ndarray:
        """Use the same model with LoRA disabled for fixed-base hindsight."""
        self.model.eval()
        try:
            with self.torch.no_grad():
                # This is deliberately one physical model instance. Disabling
                # its adapter gives the fixed-initial teacher while the
                # privileged hindsight prompt supplies the only distinction.
                with self.model.disable_adapter():
                    assessment = self._generate_teacher_assessment(observation)
                    self.last_teacher_assessment = assessment
                    logits = self._action_logits(
                        [
                            self.render_teacher_prompt(
                                observation,
                                examples,
                                assessment,
                            )
                        ]
                    )[0]
                    probabilities = self.torch.softmax(
                        logits / self.teacher_temperature,
                        dim=-1,
                    )
        finally:
            self._restore_adapter_trainability()
        values = probabilities.float().cpu().numpy()
        if not np.isfinite(values).all():
            values = np.ones(len(ACTIONS), dtype=float)
        values = np.clip(values, 1e-8, None)
        return values / values.sum()

    def _validated_loss_weights(
        self,
        sample_weights: np.ndarray | None,
        batch_size: int,
    ):
        """Return detached positive FP32 weights for one optimizer batch."""
        if sample_weights is None:
            return None
        values = np.asarray(sample_weights, dtype=float)
        if values.shape != (batch_size,):
            raise ValueError("sample weights must contain one value per row")
        if not np.isfinite(values).all() or np.any(values <= 0.0):
            raise ValueError("sample weights must be finite and positive")
        return self.torch.tensor(
            values,
            device=self.device,
            dtype=self.torch.float32,
        )

    def update(
        self,
        batch: list[tuple[str, np.ndarray]],
        sample_weights: np.ndarray | None = None,
    ) -> float:
        """Apply one soft-target cross-entropy update to LoRA only."""
        if self.optimizer is None:
            raise RuntimeError(
                "start_run must receive a learning rate before LoRA update"
            )
        self.model.train()
        prompts = [self.render_prompt(context) for context, _ in batch]
        targets = self.torch.tensor(
            np.stack([target for _, target in batch]),
            device=self.device,
            dtype=self.torch.float32,
        )
        logits = self._action_logits(prompts)
        per_example_loss = -(
            targets * self.torch.log_softmax(logits, dim=-1)
        ).sum(-1)
        weights = self._validated_loss_weights(sample_weights, len(batch))
        loss = (
            per_example_loss.mean()
            if weights is None
            else (weights * per_example_loss).sum() / weights.sum()
        )
        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        self.torch.nn.utils.clip_grad_norm_(
            (
                parameter
                for parameter in self.model.parameters()
                if parameter.requires_grad
            ),
            max_norm=self.sdft_settings.max_grad_norm,
        )
        self.optimizer.step()
        if self.device.type == "mps":
            self.torch.mps.synchronize()
            self.torch.mps.empty_cache()
        return float(loss.detach().cpu())

    def reinforce_update(
        self,
        batch: list[tuple[str, int, float]],
        entropy_coef: float,
        max_grad_norm: float,
    ) -> float:
        """Apply one factual-reward action-token update to LoRA only."""
        if self.optimizer is None:
            raise RuntimeError(
                "start_run must receive a learning rate before LoRA update"
            )
        if not batch:
            raise ValueError("REINFORCE update batch cannot be empty")
        if not np.isfinite(entropy_coef) or entropy_coef < 0.0:
            raise ValueError(
                "REINFORCE entropy coefficient must be finite and non-negative"
            )
        if not np.isfinite(max_grad_norm) or max_grad_norm <= 0.0:
            raise ValueError(
                "REINFORCE maximum gradient norm must be finite and positive"
            )
        actions_array = np.asarray(
            [action for _, action, _ in batch],
            dtype=int,
        )
        advantages_array = np.asarray(
            [advantage for _, _, advantage in batch],
            dtype=float,
        )
        if np.any(actions_array < 0) or np.any(actions_array >= len(ACTIONS)):
            raise ValueError("REINFORCE actions must index a configured route")
        if not np.isfinite(advantages_array).all():
            raise ValueError("REINFORCE advantages must be finite")

        self.model.train()
        prompts = [self.render_prompt(context) for context, _, _ in batch]
        actions = self.torch.tensor(
            actions_array,
            device=self.device,
            dtype=self.torch.long,
        )
        advantages = self.torch.tensor(
            advantages_array,
            device=self.device,
            dtype=self.torch.float32,
        )
        logits = self._action_logits(prompts) / STUDENT_TEMPERATURE
        log_probabilities = self.torch.log_softmax(logits, dim=-1)
        selected_log_probabilities = log_probabilities.gather(
            1,
            actions.unsqueeze(1),
        ).squeeze(1)
        probabilities = log_probabilities.exp()
        entropy = -(probabilities * log_probabilities).sum(-1)
        loss = -(
            advantages.detach() * selected_log_probabilities
            + entropy_coef * entropy
        ).mean()
        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        self.torch.nn.utils.clip_grad_norm_(
            (
                parameter
                for parameter in self.model.parameters()
                if parameter.requires_grad
            ),
            max_norm=max_grad_norm,
        )
        self.optimizer.step()
        if self.device.type == "mps":
            self.torch.mps.synchronize()
            self.torch.mps.empty_cache()
        return float(loss.detach().cpu())

    def update_support(
        self,
        batch: list[tuple[str, np.ndarray]],
        sample_weights: np.ndarray | None = None,
    ) -> float:
        """Maximize probability assigned to each callback's causal support."""
        if self.optimizer is None:
            raise RuntimeError(
                "start_run must receive a learning rate before LoRA update"
            )
        if not batch:
            raise ValueError("causal support update batch cannot be empty")
        self.model.train()
        prompts = [self.render_prompt(context) for context, _ in batch]
        supports_array = np.stack(
            [np.asarray(support, dtype=float) for _, support in batch]
        )
        if supports_array.shape != (len(batch), len(ACTIONS)):
            raise ValueError("causal support masks must have one value per route")
        if not np.isfinite(supports_array).all():
            raise ValueError("causal support masks must be finite")
        if not np.logical_or(supports_array == 0.0, supports_array == 1.0).all():
            raise ValueError("causal support masks must be binary")
        support_sizes = supports_array.sum(axis=1)
        if not np.all(support_sizes >= 1):
            raise ValueError("causal support masks cannot be empty")
        if not np.all(support_sizes < len(ACTIONS)):
            raise ValueError("uninformative full support must remain censored")
        supports = self.torch.tensor(
            supports_array,
            device=self.device,
            dtype=self.torch.bool,
        )
        logits = self._action_logits(prompts)
        log_probabilities = self.torch.log_softmax(logits, dim=-1)
        supported_log_probabilities = log_probabilities.masked_fill(
            ~supports,
            -self.torch.inf,
        )
        per_example_loss = -self.torch.logsumexp(
            supported_log_probabilities,
            dim=-1,
        )
        weights = self._validated_loss_weights(sample_weights, len(batch))
        loss = (
            per_example_loss.mean()
            if weights is None
            else (weights * per_example_loss).sum() / weights.sum()
        )
        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        self.torch.nn.utils.clip_grad_norm_(
            (
                parameter
                for parameter in self.model.parameters()
                if parameter.requires_grad
            ),
            max_norm=self.sdft_settings.max_grad_norm,
        )
        self.optimizer.step()
        if self.device.type == "mps":
            self.torch.mps.synchronize()
            self.torch.mps.empty_cache()
        return float(loss.detach().cpu())


@dataclass
class InteractionRecord:
    """One completed causal interaction retained by ICL or RAG."""

    observation: StudentObservation
    action: int
    feedback: dict

    def prompt_example(self) -> dict:
        return {
            "context": self.observation.text,
            "executed_action": self.feedback.get(
                "action_taken", ACTIONS[self.action]
            ),
            "eventual_user_action": self.feedback.get(
                "outcome", "UNKNOWN"
            ),
            "delay_minutes": self.feedback.get("delay_minutes", 0),
            "observed_user_selection": self.feedback.get(
                "observed_user_selection", "UNKNOWN"
            ),
        }


def mixed_context_similarity(
    query: StudentObservation,
    candidate: StudentObservation,
    text_weight: float = RAG_TEXT_WEIGHT,
) -> float:
    """Gower-style similarity over the shared decision-time fields only.

    Category and regime use exact match; hour uses circular distance; and
    importance uses normalized absolute distance. Deadline, urgency, affinity,
    the hidden dataset answer, and the post-action user selection are never
    retrieval keys.
    """
    category_count = len(CATEGORIES)
    query_features = query.features
    candidate_features = candidate.features

    category_similarity = float(
        np.argmax(query_features[:category_count])
        == np.argmax(candidate_features[:category_count])
    )

    query_importance = query_features[category_count]
    candidate_importance = candidate_features[category_count]
    importance_similarity = 1.0 - abs(
        float(query_importance - candidate_importance)
    )

    query_clock = query_features[category_count + 1 : category_count + 3]
    candidate_clock = candidate_features[
        category_count + 1 : category_count + 3
    ]
    clock_cosine = float(
        np.clip(np.dot(query_clock, candidate_clock), -1.0, 1.0)
    )
    hour_similarity = 1.0 - float(np.arccos(clock_cosine) / np.pi)

    query_regime = int(round(2 * query_features[category_count + 3]))
    candidate_regime = int(
        round(2 * candidate_features[category_count + 3])
    )
    regime_similarity = float(query_regime == candidate_regime)
    metadata_similarity = float(
        np.mean(
            [
                category_similarity,
                importance_similarity,
                hour_similarity,
                regime_similarity,
            ]
        )
    )
    if not 0.0 <= text_weight <= 1.0:
        raise ValueError("text_weight must be between 0 and 1")
    if text_weight == 0.0:
        return metadata_similarity
    text_similarity = notification_text_similarity(
        query.text,
        candidate.text,
    )
    return float(
        (1.0 - text_weight) * metadata_similarity
        + text_weight * text_similarity
    )


_CONTENT_STOPWORDS = {
    "a", "an", "and", "at", "before", "during", "for", "from", "in",
    "is", "it", "local", "message", "notification", "of", "on", "out",
    "says", "score", "the", "this", "time", "to", "was", "with",
}


def _notification_content_tokens(text: str) -> set[str]:
    """Tokenize only the visible title/body portion of a serving prompt."""
    content = text.split(" This is a ", 1)[0].lower()
    translation = str.maketrans(
        {character: " " for character in string.punctuation}
    )
    return {
        token
        for token in content.translate(translation).split()
        if len(token) > 1 and token not in _CONTENT_STOPWORDS
    }


def notification_text_similarity(query_text: str, candidate_text: str) -> float:
    """Return Jaccard similarity over decision-visible title/body tokens."""
    query_tokens = _notification_content_tokens(query_text)
    candidate_tokens = _notification_content_tokens(candidate_text)
    union = query_tokens | candidate_tokens
    if not union:
        return 0.0
    return len(query_tokens & candidate_tokens) / len(union)


def feedback_surface_propensity(behavior_distribution: np.ndarray) -> float:
    """Return the decision-time chance of an observable feedback surface."""
    behavior = np.asarray(behavior_distribution, dtype=float)
    if behavior.shape != (len(ACTIONS),):
        raise ValueError("behavior distribution must contain one value per route")
    if not np.isfinite(behavior).all() or np.any(behavior < 0.0):
        raise ValueError("behavior distribution must be finite and non-negative")
    if not np.isclose(float(behavior.sum()), 1.0, rtol=0.0, atol=1e-8):
        raise ValueError("behavior distribution must sum to one")
    propensity = float(
        behavior[ACTIONS.index("INTERRUPT")]
        + behavior[ACTIONS.index("LATER")]
    )
    if propensity <= 0.0:
        raise ValueError("feedback-surface propensity must be positive")
    return propensity


class OnlineAgent:
    """Base class for one method on one chronological stream."""

    name = "Base"
    learning_rate: float | None = None
    samples_from_policy = False
    uses_teacher = False
    stores_interactions = False
    replay_size = REPLAY_SIZE
    online_batch_size = ONLINE_BATCH_SIZE
    update_steps = 1

    def __init__(
        self,
        policy: StudentPolicy,
        icl_examples: int = ICL_K,
        rag_examples: int = RAG_K,
    ):
        self.policy = policy
        if icl_examples < 0 or rag_examples < 0:
            raise ValueError("example counts must be non-negative")
        self.icl_examples = icl_examples
        self.rag_examples = rag_examples
        self.memory: list[InteractionRecord] = []
        self.replay: list[tuple[Any, ...]] = []
        self.policy.start_run(self.learning_rate)

    def prompt_examples(
        self,
        observation: StudentObservation,
    ) -> list[dict]:
        del observation
        return []

    def action_probs(self, observation: StudentObservation) -> np.ndarray:
        return self.policy.probs(
            observation.text,
            self.prompt_examples(observation),
        )

    def reliable_memory(self) -> list[InteractionRecord]:
        """Return callbacks that identify one route on the executed surface.

        Memory retains every matured factual interaction for audit, including
        ambiguous digest opens and censored archive outcomes. Only singleton
        causal evidence becomes a labeled prompt example: otherwise the
        executed route can prime its own future selection without identifying
        the user's preferred route.
        """
        return [
            record
            for record in self.memory
            if record.feedback.get("observed_user_selection") != "UNKNOWN"
            and causal_evidence_reliability(
                record.action,
                record.feedback,
            ) == "reliable_singleton"
        ]

    def training_target(
        self,
        teacher_distribution: np.ndarray,
        teacher_action: int,
        *,
        action: int,
        feedback: dict,
        decision_distribution: np.ndarray | None,
        candidate_action: int | None = None,
    ) -> np.ndarray | None:
        del (
            teacher_distribution,
            teacher_action,
            action,
            feedback,
            decision_distribution,
            candidate_action,
        )
        return None

    def observe(
        self,
        observation: StudentObservation,
        action: int,
        teacher_distribution: np.ndarray | None,
        teacher_action: int | None,
        feedback: dict,
        rng: np.random.Generator,
        teacher_observation: TeacherObservation | None = None,
        decision_distribution: np.ndarray | None = None,
        candidate_action: int | None = None,
        behavior_distribution: np.ndarray | None = None,
    ) -> None:
        """Retain direct history or apply a hindsight distillation update."""
        if hasattr(self, "last_observation_update_count"):
            self.last_observation_update_count = 0
        if self.stores_interactions:
            self.memory.append(
                InteractionRecord(
                    observation=observation,
                    action=action,
                    feedback=feedback,
                )
            )
            return
        if feedback.get("observed_user_selection") == "UNKNOWN":
            # Censoring is not a negative label and not a teacher-generated
            # hard label. The teacher distribution may still be logged by the
            # experiment for audit, but it cannot enter memory or an update.
            getattr(self, "_decision_base_cache", {}).pop(
                id(observation),
                None,
            )
            return
        if teacher_distribution is None or teacher_action is None:
            if self.uses_teacher:
                raise ValueError("teacher-supervised method requires a teacher")
            return
        target = self.training_target(
            teacher_distribution,
            teacher_action,
            action=action,
            feedback=feedback,
            decision_distribution=decision_distribution,
            candidate_action=candidate_action,
        )
        if target is None:
            getattr(self, "_decision_base_cache", {}).pop(
                id(observation),
                None,
            )
            return

        feedback_propensity = (
            None
            if behavior_distribution is None
            else feedback_surface_propensity(behavior_distribution)
        )
        propensity_mode = getattr(
            getattr(self, "settings", None),
            "propensity_weight_mode",
            "none",
        )
        if propensity_mode != "none" and feedback_propensity is None:
            raise ValueError(
                "propensity-weighted replay requires the decision-time "
                "behavior distribution"
            )

        selection = feedback["observed_user_selection"]
        replay_label = (
            "AMBIGUOUS"
            if ACTIONS[action] == "LATER"
            and feedback.get("outcome") == "OPENED_DIGEST"
            else selection
        )
        replay_prompt_example = (
            InteractionRecord(
                observation=observation,
                action=action,
                feedback=feedback,
            ).prompt_example()
            if getattr(self, "replay_prompt_examples", 0) > 0
            and causal_evidence_reliability(action, feedback)
            == "reliable_singleton"
            else None
        )
        self.replay.append(
            (
                observation.text,
                target,
                replay_label,
                getattr(self, "_decision_base_cache", {}).pop(
                    id(observation),
                    None,
                ),
                replay_prompt_example,
                feedback_propensity,
            )
        )
        self.replay = self.replay[-self.replay_size:]
        if len(self.replay) < getattr(self, "warmup_examples", 1):
            return
        settings = getattr(self, "settings", None)
        if (
            replay_label == "AMBIGUOUS"
            and getattr(settings, "ambiguous_update_mode", "immediate")
            == "defer"
        ):
            return
        for update_step in range(self.update_steps):
            force_newest = bool(
                update_step == 0
                or getattr(settings, "force_newest_every_step", True)
            )
            indices = [len(self.replay) - 1] if force_newest else []
            candidate_stop = len(self.replay) - int(force_newest)
            candidates = np.arange(candidate_stop)
            sample_size = min(
                self.online_batch_size - len(indices),
                len(candidates),
            )
            if sample_size:
                probabilities = None
                replay_strategy = getattr(self, "replay_strategy", "uniform")
                if replay_strategy == "selection_balanced":
                    recency_half_life = getattr(
                        settings,
                        "replay_recency_half_life",
                        None,
                    )
                    if recency_half_life is not None:
                        labels = [
                            self.replay[index][2] for index in candidates
                        ]
                        weights = np.empty(len(candidates), dtype=float)
                        for label in dict.fromkeys(labels):
                            positions = np.asarray(
                                [
                                    position
                                    for position, candidate_label in enumerate(
                                        labels
                                    )
                                    if candidate_label == label
                                ],
                                dtype=int,
                            )
                            ages = (
                                len(self.replay)
                                - 1
                                - candidates[positions]
                            )
                            # Subtracting the within-label newest age is a
                            # numerically stable common rescaling of
                            # q = 2 ** (-age / half_life).
                            relative_recency = np.exp2(
                                -(ages - ages.min()) / recency_half_life
                            )
                            group_mass = (
                                getattr(
                                    settings,
                                    "ambiguous_replay_group_weight",
                                    1.0,
                                )
                                if label == "AMBIGUOUS"
                                else 1.0
                            )
                            weights[positions] = (
                                group_mass
                                * relative_recency
                                / relative_recency.sum()
                            )
                    else:
                        counts = Counter(
                            self.replay[index][2]
                            for index in candidates
                        )
                        weights = np.asarray(
                            [
                                (
                                    getattr(
                                        getattr(self, "settings", None),
                                        "ambiguous_replay_group_weight",
                                        1.0,
                                    )
                                    if self.replay[index][2] == "AMBIGUOUS"
                                    else 1.0
                                )
                                / counts[self.replay[index][2]]
                                for index in candidates
                            ],
                            dtype=float,
                        )
                    probabilities = weights / weights.sum()
                indices += rng.choice(
                    candidates,
                    size=sample_size,
                    replace=False,
                    p=probabilities,
                ).tolist()
            self.update_from_replay([self.replay[index] for index in indices])
            if hasattr(self, "last_observation_update_count"):
                self.last_observation_update_count += 1
                self.online_update_count += 1


class BaseAgent(OnlineAgent):
    """Frozen LFM without external memory."""


class ICLAgent(OnlineAgent):
    """Frozen LFM prompted with recent reliable causal interactions."""

    name = "ICL"
    stores_interactions = True

    def prompt_examples(
        self,
        observation: StudentObservation,
    ) -> list[dict]:
        del observation
        if self.icl_examples == 0:
            return []
        return [
            record.prompt_example()
            for record in self.reliable_memory()[-self.icl_examples:]
        ]


class RAGAgent(OnlineAgent):
    """Frozen LFM prompted with similar reliable causal interactions."""

    name = "RAG"
    stores_interactions = True

    def __init__(
        self,
        policy: StudentPolicy,
        icl_examples: int = ICL_K,
        rag_examples: int = RAG_K,
        rag_text_weight: float = RAG_TEXT_WEIGHT,
    ):
        if not 0.0 <= rag_text_weight <= 1.0:
            raise ValueError("rag_text_weight must be between 0 and 1")
        self.rag_text_weight = rag_text_weight
        super().__init__(policy, icl_examples, rag_examples)

    def prompt_examples(
        self,
        observation: StudentObservation,
    ) -> list[dict]:
        records = self.reliable_memory()
        if not records:
            return []
        similarities = [
            (
                mixed_context_similarity(
                    observation,
                    record.observation,
                    self.rag_text_weight,
                ),
                index,
                record,
            )
            for index, record in enumerate(records)
        ]
        closest = sorted(
            similarities,
            key=lambda item: (item[0], item[1]),
            reverse=True,
        )[:self.rag_examples]
        # Put the best match closest to the current query. For equal scores,
        # the newest record is also closest to the query in the prompt.
        closest.sort(key=lambda item: (item[0], item[1]))
        return [record.prompt_example() for _, _, record in closest]


class REINFORCEAgent(OnlineAgent):
    """Batched action-token LoRA policy gradient from factual rewards only."""

    name = "REINFORCE"
    learning_rate = REINFORCE_LR
    online_batch_size = REINFORCE_BATCH_SIZE
    samples_from_policy = True
    uses_teacher = False

    def __init__(
        self,
        policy: StudentPolicy,
        icl_examples: int = ICL_K,
        rag_examples: int = RAG_K,
        settings: REINFORCESettings = DEFAULT_REINFORCE_SETTINGS,
        adapter_settings: OnlineSDFTSettings = DEFAULT_SDFT_SETTINGS,
    ):
        self.settings = settings
        self.learning_rate = settings.learning_rate
        self.online_batch_size = settings.batch_size
        self.reward_outcome_map = dict(settings.reward_outcome_map)
        self.reward_baseline = 0.0
        self.last_training_reward: float | None = None
        self.pending_updates: list[tuple[str, int, float]] = []
        self.online_update_count = 0
        self.last_observation_update_count = 0
        configure = getattr(policy, "configure_online_sdft", None)
        if configure is not None:
            configure(adapter_settings)
        super().__init__(policy, icl_examples, rag_examples)

    def observe(
        self,
        observation: StudentObservation,
        action: int,
        teacher_distribution: np.ndarray | None,
        teacher_action: int | None,
        feedback: dict,
        rng: np.random.Generator,
        teacher_observation: TeacherObservation | None = None,
        decision_distribution: np.ndarray | None = None,
        candidate_action: int | None = None,
        behavior_distribution: np.ndarray | None = None,
    ) -> None:
        """Batch known factual rewards and update the shared LoRA adapter."""
        del (
            teacher_distribution,
            teacher_action,
            rng,
            teacher_observation,
            decision_distribution,
            candidate_action,
            behavior_distribution,
        )
        self.last_observation_update_count = 0
        self.last_training_reward = None
        selection = feedback.get("observed_user_selection")
        if selection not in {*ACTIONS, "UNKNOWN"}:
            raise ValueError(
                "REINFORCE feedback must contain an explicit observed user "
                "selection"
            )
        if selection == "UNKNOWN":
            # A neutral censored event must remain neutral after baseline
            # subtraction, so it produces no policy-gradient update.
            return
        if self.reward_outcome_map:
            outcome = feedback.get("outcome")
            if outcome not in self.reward_outcome_map:
                raise ValueError(
                    "REINFORCE factual outcome is missing from the configured "
                    "reward map"
                )
            reward = self.reward_outcome_map[outcome]
        else:
            reward = float(feedback["reward"])
        if not np.isfinite(reward):
            raise ValueError("REINFORCE factual reward must be finite")
        self.last_training_reward = float(reward)
        advantage = reward - self.reward_baseline
        self.pending_updates.append((observation.text, action, advantage))
        self.reward_baseline += self.settings.baseline_step * (
            reward - self.reward_baseline
        )
        if len(self.pending_updates) < self.online_batch_size:
            return

        self.policy.reinforce_update(
            list(self.pending_updates),
            entropy_coef=self.settings.entropy_coef,
            max_grad_norm=self.settings.max_grad_norm,
        )
        self.pending_updates.clear()
        self.online_update_count += 1
        self.last_observation_update_count = 1


def causal_route_support(action: int, feedback: dict) -> np.ndarray:
    """Return routes identifiable from one factual delivery-surface callback.

    This encodes the public observation contract, not the evaluator's hidden
    preferred route. In particular, opening a digest cannot distinguish a
    missed immediate need from a genuine preference to read later.
    """
    if action < 0 or action >= len(ACTIONS):
        raise ValueError("action index is out of range")
    action_name = ACTIONS[action]
    recorded_action = feedback.get("action_taken", action_name)
    if recorded_action != action_name:
        raise ValueError("feedback must belong to the executed route")

    outcome = feedback.get("outcome")
    support_by_trajectory = {
        ("INTERRUPT", "OPENED_IMMEDIATELY"): ("INTERRUPT",),
        ("INTERRUPT", "OPENED_AFTER_DELAY"): ("LATER",),
        ("INTERRUPT", "DELETED_NOTIFICATION"): ("ARCHIVE",),
        ("LATER", "OPENED_DIGEST"): ("INTERRUPT", "LATER"),
        ("LATER", "DELETED_FROM_DIGEST"): ("ARCHIVE",),
        ("ARCHIVE", "NO_OBSERVABLE_SELECTION"): ACTIONS,
    }
    try:
        supported = support_by_trajectory[(action_name, outcome)]
    except KeyError as error:
        raise ValueError(
            "unknown action/outcome trajectory for causal support: "
            f"{action_name}/{outcome}"
        ) from error
    return np.asarray(
        [route in supported for route in ACTIONS],
        dtype=bool,
    )


def causal_evidence_reliability(action: int, feedback: dict) -> str:
    """Classify callback reliability from public causal support cardinality."""
    support = causal_route_support(action, feedback)
    supported_routes = int(support.sum())
    if supported_routes == 1:
        return "reliable_singleton"
    if supported_routes == 2:
        return "ambiguous_digest_open"
    if support.all():
        return "censored_unknown"
    raise ValueError("causal support must contain at least one route")


class OnlineSDFTAgent(OnlineAgent):
    """Online LoRA student from reliability-conditioned soft evidence."""

    name = "Online-SDFT"
    learning_rate = SDFT_LR
    replay_size = SDFT_REPLAY_SIZE
    online_batch_size = SDFT_BATCH_SIZE
    update_steps = SDFT_UPDATE_STEPS
    uses_teacher = True

    def __init__(
        self,
        policy: StudentPolicy,
        icl_examples: int = ICL_K,
        rag_examples: int = RAG_K,
        settings: OnlineSDFTSettings = DEFAULT_SDFT_SETTINGS,
    ):
        self.settings = settings
        self.learning_rate = settings.learning_rate
        self.online_update_count = 0
        self.last_observation_update_count = 0
        self.replay_size = settings.replay_size
        self.replay_prompt_examples = settings.replay_prompt_examples
        self.online_batch_size = settings.batch_size
        self.update_steps = settings.update_steps
        self.warmup_examples = settings.warmup_examples
        self.replay_strategy = settings.replay_strategy
        self.behavior_mode = settings.behavior_mode
        self.behavior_epsilon = settings.behavior_epsilon
        self.behavior_epsilon_half_life = settings.behavior_epsilon_half_life
        self.exploration_taper_start_step = (
            settings.exploration_taper_start_step
        )
        self.exploration_taper_half_life = (
            settings.exploration_taper_half_life
        )
        self.archive_probe_mix = settings.archive_probe_mix
        self.archive_policy_min_feedback = settings.archive_policy_min_feedback
        self.interrupt_probe_mix = settings.interrupt_probe_mix
        self.interrupt_probe_half_life = settings.interrupt_probe_half_life
        self.interrupt_probe_max_confidence = (
            settings.interrupt_probe_max_confidence
        )
        self.last_prompt_examples_used = 0
        self._decision_base_cache: dict[int, np.ndarray] = {}
        configure = getattr(policy, "configure_online_sdft", None)
        if configure is not None:
            configure(settings)
        super().__init__(policy, icl_examples, rag_examples)

    def prompt_examples(
        self,
        observation: StudentObservation,
    ) -> list[dict]:
        """Return recent reliable replay lessons without retrieval.

        Only factual singleton callbacks enter this FIFO prompt. Ambiguous
        digest opens, censored outcomes, soft targets, and teacher outputs are
        excluded. A zero count keeps pure parametric Online-SDFT unchanged.
        """
        del observation
        if self.replay_prompt_examples == 0:
            self.last_prompt_examples_used = 0
            return []
        eligible = [
            row[4]
            for row in self.replay
            if len(row) > 4 and row[4] is not None
        ]
        examples = eligible[-self.replay_prompt_examples :]
        self.last_prompt_examples_used = len(examples)
        return [dict(example) for example in examples]

    def action_probs(self, observation: StudentObservation) -> np.ndarray:
        """Serve from the shared Liquid model with its current LoRA state."""
        return self.policy.probs(
            observation.text,
            self.prompt_examples(observation),
        )

    def observe(
        self,
        observation: StudentObservation,
        action: int,
        teacher_distribution: np.ndarray | None,
        teacher_action: int | None,
        feedback: dict,
        rng: np.random.Generator,
        teacher_observation: TeacherObservation | None = None,
        decision_distribution: np.ndarray | None = None,
        candidate_action: int | None = None,
        behavior_distribution: np.ndarray | None = None,
    ) -> None:
        """Cache one adapter-disabled student anchor for informative replay."""
        cache_key = id(observation)
        if (
            self.settings.base_kl_weight
            and feedback.get("observed_user_selection") != "UNKNOWN"
        ):
            self._decision_base_cache[cache_key] = self.policy.base_probs(
                observation.text
            )
        try:
            super().observe(
                observation,
                action,
                teacher_distribution,
                teacher_action,
                feedback,
                rng,
                teacher_observation=teacher_observation,
                decision_distribution=decision_distribution,
                candidate_action=candidate_action,
                behavior_distribution=behavior_distribution,
            )
        except Exception:
            self._decision_base_cache.pop(cache_key, None)
            raise

    def update_from_replay(self, rows: list[tuple[Any, ...]]) -> None:
        """Fit only the PEFT adapter on resolved replay contexts and targets."""
        sample_weights = None
        if self.settings.propensity_weight_mode == "feedback_surface_snips":
            if any(len(row) < 6 or row[5] is None for row in rows):
                raise ValueError(
                    "propensity-weighted replay requires stored feedback "
                    "propensities"
                )
            propensities = np.asarray([row[5] for row in rows], dtype=float)
            if (
                not np.isfinite(propensities).all()
                or np.any(propensities <= 0.0)
                or np.any(propensities > 1.0)
            ):
                raise ValueError(
                    "stored feedback propensities must be finite and in (0, 1]"
                )
            sample_weights = np.minimum(
                self.settings.propensity_weight_cap,
                1.0 / np.maximum(propensities, 1e-8),
            )
        if self.settings.base_kl_weight:
            if any(len(row) < 4 or row[3] is None for row in rows):
                raise ValueError("fixed-base KL replay requires cached anchors")
            beta = self.settings.base_kl_weight
            batch = []
            for row in rows:
                target = np.asarray(row[1], dtype=float)
                target = np.clip(target, 1e-8, None)
                target /= target.sum()
                base = np.asarray(row[3], dtype=float)
                base = np.clip(base, 1e-8, None)
                base /= base.sum()
                effective_target = (target + beta * base) / (1.0 + beta)
                batch.append((row[0], effective_target))
        else:
            batch = [(row[0], row[1]) for row in rows]
        if self.settings.target_mode == "support_likelihood":
            if sample_weights is None:
                self.policy.update_support(batch)
            else:
                self.policy.update_support(
                    batch,
                    sample_weights=sample_weights,
                )
            return
        if sample_weights is None:
            self.policy.update(batch)
        else:
            self.policy.update(batch, sample_weights=sample_weights)

    @staticmethod
    def causal_behavior_support(action: int, feedback: dict) -> np.ndarray:
        """Return maximum-entropy guidance over the causal support set."""
        support = causal_route_support(action, feedback).astype(float)
        return support / support.sum()

    def fusion_weights(self, action: int, feedback: dict) -> tuple[float, ...]:
        """Resolve teacher, decision, and behavior weights for one callback."""
        reliability = causal_evidence_reliability(action, feedback)
        if reliability == "reliable_singleton":
            return (
                self.settings.reliable_teacher_weight,
                self.settings.reliable_decision_weight,
                self.settings.reliable_behavior_weight,
            )
        if reliability == "ambiguous_digest_open":
            return (
                self.settings.ambiguous_teacher_weight,
                self.settings.ambiguous_decision_weight,
                self.settings.ambiguous_behavior_weight,
            )
        raise ValueError("censored feedback cannot produce an SDFT target")

    def training_target(
        self,
        teacher_distribution: np.ndarray,
        teacher_action: int,
        *,
        action: int,
        feedback: dict,
        decision_distribution: np.ndarray | None,
        candidate_action: int | None = None,
    ) -> np.ndarray | None:
        """Build a soft target from causal evidence available at release."""
        del teacher_action, candidate_action
        reliability = None
        if self.settings.ambiguous_update_mode == "skip":
            reliability = causal_evidence_reliability(action, feedback)
            if reliability == "ambiguous_digest_open":
                return None
        if self.settings.target_mode == "support_likelihood":
            support = causal_route_support(action, feedback)
            if support.all():
                raise ValueError("censored feedback cannot produce an SDFT target")
            return support.astype(float)
        teacher = np.clip(teacher_distribution, 1e-8, None)
        teacher = teacher / teacher.sum()
        if self.settings.target_mode == "teacher_only":
            return teacher
        if decision_distribution is None:
            raise ValueError("causal fusion requires the frozen decision prior")
        decision = np.clip(decision_distribution, 1e-8, None)
        decision = decision / decision.sum()
        if reliability is None:
            reliability = causal_evidence_reliability(action, feedback)
        if (
            reliability == "ambiguous_digest_open"
            and self.settings.ambiguous_projection == "causal_support"
        ):
            support = causal_route_support(action, feedback).astype(float)
            teacher = teacher * support
            teacher = teacher / teacher.sum()
            decision = decision * support
            decision = decision / decision.sum()
        teacher_weight, decision_weight, behavior_weight = self.fusion_weights(
            action,
            feedback,
        )
        behavior = self.causal_behavior_support(action, feedback)
        target = (
            teacher_weight * teacher
            + decision_weight * decision
            + behavior_weight * behavior
        )
        target = np.clip(target, 1e-8, None)
        return target / target.sum()


class RFTAgent(OnlineSDFTAgent):
    """Teacher-sampled hard-target distillation with causal rejection.

    RFT shares Online-SDFT's frozen LFM base and LoRA architecture. It retains
    its independently configured replay-32 epsilon-greedy schedule and tuned
    learning rate. Its target is one categorical teacher candidate retained
    only when a delayed factual callback gives singleton causal support for
    that route.
    """

    name = "RFT"

    def __init__(
        self,
        policy: StudentPolicy,
        icl_examples: int = ICL_K,
        rag_examples: int = RAG_K,
        rft_settings: RFTSettings = DEFAULT_RFT_SETTINGS,
    ):
        self.rft_settings = rft_settings
        super().__init__(
            policy,
            icl_examples,
            rag_examples,
            rft_settings.student_settings,
        )
        self.last_rft_candidate_action: int | None = None
        self.last_rft_accepted: bool | None = None
        self.last_rft_reason: str | None = None

    @staticmethod
    def rft_reason(
        action: int,
        feedback: dict,
        candidate_action: int | None,
    ) -> str:
        """Classify one teacher sample using only public causal support."""
        reliability = causal_evidence_reliability(action, feedback)
        if reliability == "censored_unknown":
            return "censored_unknown"
        if candidate_action is None:
            raise ValueError("RFT requires one sampled teacher candidate")
        if candidate_action < 0 or candidate_action >= len(ACTIONS):
            raise ValueError("RFT teacher candidate is out of range")
        if reliability == "ambiguous_digest_open":
            return "ambiguous_unverified"
        support = causal_route_support(action, feedback)
        if support[candidate_action]:
            return "accepted"
        return "teacher_mismatch"

    def observe(
        self,
        observation: StudentObservation,
        action: int,
        teacher_distribution: np.ndarray | None,
        teacher_action: int | None,
        feedback: dict,
        rng: np.random.Generator,
        teacher_observation: TeacherObservation | None = None,
        decision_distribution: np.ndarray | None = None,
        candidate_action: int | None = None,
        behavior_distribution: np.ndarray | None = None,
    ) -> None:
        reason = self.rft_reason(action, feedback, candidate_action)
        self.last_rft_candidate_action = candidate_action
        self.last_rft_reason = reason
        self.last_rft_accepted = (
            None
            if reason == "censored_unknown"
            else reason == "accepted"
        )
        super().observe(
            observation,
            action,
            teacher_distribution,
            teacher_action,
            feedback,
            rng,
            teacher_observation=teacher_observation,
            decision_distribution=decision_distribution,
            candidate_action=candidate_action,
            behavior_distribution=behavior_distribution,
        )

    def training_target(
        self,
        teacher_distribution: np.ndarray,
        teacher_action: int,
        *,
        action: int,
        feedback: dict,
        decision_distribution: np.ndarray | None,
        candidate_action: int | None = None,
    ) -> np.ndarray | None:
        """Return one-hot CE supervision only for a verified teacher sample."""
        del teacher_distribution, teacher_action, decision_distribution
        if self.rft_reason(action, feedback, candidate_action) != "accepted":
            return None
        target = np.zeros(len(ACTIONS), dtype=float)
        target[candidate_action] = 1.0
        return target


AGENT_CLASSES = {
    agent_class.name: agent_class
    for agent_class in (
        BaseAgent,
        ICLAgent,
        RAGAgent,
        REINFORCEAgent,
        RFTAgent,
        OnlineSDFTAgent,
    )
}


def create_agent(
    method: str,
    policy: StudentPolicy,
    icl_examples: int = ICL_K,
    rag_examples: int = RAG_K,
    rag_text_weight: float = RAG_TEXT_WEIGHT,
    sdft_settings: OnlineSDFTSettings = DEFAULT_SDFT_SETTINGS,
    reinforce_settings: REINFORCESettings = DEFAULT_REINFORCE_SETTINGS,
    rft_settings: RFTSettings = DEFAULT_RFT_SETTINGS,
) -> OnlineAgent:
    """Construct one named method or fail loudly on an invalid benchmark arm."""
    try:
        agent_class = AGENT_CLASSES[method]
    except KeyError as error:
        raise ValueError(f"unknown method {method!r}") from error
    if agent_class is RFTAgent:
        return agent_class(
            policy,
            icl_examples,
            rag_examples,
            rft_settings,
        )
    if agent_class is OnlineSDFTAgent:
        return agent_class(
            policy,
            icl_examples,
            rag_examples,
            sdft_settings,
        )
    if agent_class is RAGAgent:
        return agent_class(
            policy,
            icl_examples,
            rag_examples,
            rag_text_weight,
        )
    if agent_class is REINFORCEAgent:
        return agent_class(
            policy,
            icl_examples,
            rag_examples,
            reinforce_settings,
            sdft_settings,
        )
    return agent_class(policy, icl_examples, rag_examples)

"""Prequential experiment loop and artifact orchestration.

This module is the only place where a method interacts with the environment.
The ordering is explicit: release any older feedback whose observation window
has closed, observe the new context, act, freeze the score, execute one route,
and queue its callback for a later round.
"""


import csv
import gc
import hashlib
import json
from collections import Counter

import numpy as np



METHOD_RNG_OFFSETS = {
    "Base": 5,
    "ICL": 18,
    "RAG": 31,
    "RFT": 44,
    "Online-SDFT": 57,
    "REINFORCE": 70,
}
RFT_CANDIDATE_RNG_OFFSET = 83
RFT_CANDIDATE_SAMPLER = "blake2b-event-keyed-uniform-inverse-cdf-v1"


def epsilon_greedy(
    probs: np.ndarray,
    epsilon: float = EXPLORATION_EPSILON,
) -> np.ndarray:
    """Return the declared serving distribution over the student's ranking."""
    if (
        isinstance(epsilon, (bool, np.bool_))
        or not isinstance(epsilon, (int, float, np.integer, np.floating))
        or not np.isfinite(epsilon)
        or not 0.0 <= epsilon <= 1.0
    ):
        raise ValueError("serving epsilon must be finite and in [0, 1]")
    probs = np.asarray(probs, dtype=float)
    if not np.isfinite(probs).all():
        probs = np.ones(len(ACTIONS), dtype=float)
    greedy = one_hot(int(np.argmax(probs)), len(ACTIONS))
    behavior = (
        (1 - epsilon) * greedy
        + epsilon / len(ACTIONS)
    )
    return behavior / behavior.sum()


def policy_sampling(probs: np.ndarray) -> np.ndarray:
    """Normalize the differentiable policy used by REINFORCE to act."""
    behavior = np.asarray(probs, dtype=float)
    if not np.isfinite(behavior).all():
        behavior = np.ones(len(ACTIONS), dtype=float)
    behavior = np.clip(behavior, 1e-8, None)
    return behavior / behavior.sum()


def rft_sampling_distribution(
    teacher_probs: np.ndarray,
    sampling_temperature: float,
) -> np.ndarray:
    """Temperature-scale the original teacher distribution for RFT only."""
    values = np.asarray(teacher_probs, dtype=float)
    if values.shape != (len(ACTIONS),):
        raise ValueError("RFT teacher probabilities must match the action count")
    if not np.isfinite(values).all() or (values < 0.0).any():
        raise ValueError(
            "RFT teacher probabilities must be finite and non-negative"
        )
    total = float(values.sum())
    if total <= 0.0:
        raise ValueError("RFT teacher probabilities must have positive mass")
    temperature = float(sampling_temperature)
    if not np.isfinite(temperature) or temperature <= 0.0:
        raise ValueError("RFT sampling temperature must be positive and finite")
    values = values / total
    logits = np.full_like(values, -np.inf)
    positive = values > 0.0
    logits[positive] = np.log(values[positive]) / temperature
    logits -= np.max(logits)
    proposals = np.exp(logits)
    return proposals / proposals.sum()


def rft_event_uniform(
    seed: int,
    event_id: str,
    step: int,
    *,
    rng_offset: int | None = None,
) -> float:
    """Return a stable event-keyed uniform independent of rollout control flow."""
    offset = RFT_CANDIDATE_RNG_OFFSET if rng_offset is None else int(rng_offset)
    key = json.dumps(
        [RFT_CANDIDATE_SAMPLER, int(seed), str(event_id), int(step), offset],
        separators=(",", ":"),
    ).encode("utf-8")
    bits = int.from_bytes(hashlib.blake2b(key, digest_size=8).digest(), "big")
    mantissa = bits >> 11
    return (mantissa + 0.5) / (1 << 53)


def rft_inverse_cdf_sample(probs: np.ndarray, uniform: float) -> int:
    """Map an event-keyed uniform to one categorical route."""
    values = np.asarray(probs, dtype=float)
    if values.shape != (len(ACTIONS),):
        raise ValueError("RFT proposal probabilities must match the action count")
    if not np.isfinite(values).all() or (values < 0.0).any():
        raise ValueError(
            "RFT proposal probabilities must be finite and non-negative"
        )
    total = float(values.sum())
    if total <= 0.0:
        raise ValueError("RFT proposal probabilities must have positive mass")
    draw = float(uniform)
    if not np.isfinite(draw) or not 0.0 <= draw < 1.0:
        raise ValueError("RFT categorical uniform must be in [0, 1)")
    cumulative = np.cumsum(values / total)
    cumulative[-1] = 1.0
    return min(
        int(np.searchsorted(cumulative, draw, side="right")),
        len(ACTIONS) - 1,
    )


def archive_uniform_probe(
    probs: np.ndarray,
    mix: float,
    baseline_epsilon: float = EXPLORATION_EPSILON,
) -> np.ndarray:
    """Probe feedback-bearing routes at a fixed rate when ARCHIVE ranks first."""
    behavior = epsilon_greedy(probs, baseline_epsilon)
    if int(np.argmax(np.asarray(probs, dtype=float))) != ACTIONS.index("ARCHIVE"):
        return behavior
    probe = np.zeros(len(ACTIONS), dtype=float)
    probe[ACTIONS.index("INTERRUPT")] = 0.5
    probe[ACTIONS.index("LATER")] = 0.5
    behavior = (1.0 - mix) * behavior + mix * probe
    return behavior / behavior.sum()


def uncertainty_interrupt_probe(
    probs: np.ndarray,
    step: int,
    mix: float,
    half_life: float,
    max_confidence: float,
    baseline_epsilon: float = EXPLORATION_EPSILON,
) -> np.ndarray:
    """Add a decaying INTERRUPT probe using only pre-action learner state.

    The epsilon-greedy baseline retains positive support for every route.  The
    extra dose is active only while the normalized student distribution has no
    action above ``max_confidence``; it decays exponentially from ``mix`` at
    the first 1-based decision step.
    """
    values = np.asarray(probs, dtype=float)
    if values.shape != (len(ACTIONS),):
        raise ValueError("student probabilities must match the action count")
    if (
        isinstance(step, bool)
        or not isinstance(step, (int, np.integer))
        or step < 1
    ):
        raise ValueError("interrupt probe step must be a positive integer")
    if (
        isinstance(mix, bool)
        or not isinstance(mix, (int, float, np.integer, np.floating))
        or not np.isfinite(mix)
        or not 0.0 < mix < 1.0
    ):
        raise ValueError("interrupt probe mix must be finite and in (0, 1)")
    if (
        isinstance(half_life, bool)
        or not isinstance(half_life, (int, float, np.integer, np.floating))
        or not np.isfinite(half_life)
        or half_life <= 0.0
    ):
        raise ValueError("interrupt probe half-life must be finite and positive")
    minimum_confidence = 1.0 / len(ACTIONS)
    if (
        isinstance(max_confidence, bool)
        or not isinstance(
            max_confidence,
            (int, float, np.integer, np.floating),
        )
        or not np.isfinite(max_confidence)
        or not minimum_confidence <= max_confidence <= 1.0
    ):
        raise ValueError(
            "interrupt probe maximum confidence must be finite and in "
            f"[{minimum_confidence}, 1]"
        )

    behavior = epsilon_greedy(values, baseline_epsilon)
    confidence = float(np.max(policy_sampling(values)))
    if confidence > max_confidence:
        return behavior
    effective_mix = float(mix) * 2.0 ** (-(int(step) - 1) / float(half_life))
    probe = one_hot(ACTIONS.index("INTERRUPT"), len(ACTIONS))
    behavior = (1.0 - effective_mix) * behavior + effective_mix * probe
    return behavior / behavior.sum()


def archive_policy_feedback_floor(
    probs: np.ndarray,
    minimum_feedback: float,
) -> np.ndarray:
    """Floor feedback-bearing policy mass only when ARCHIVE ranks first."""
    behavior = np.asarray(probs, dtype=float)
    if not np.isfinite(behavior).all():
        behavior = np.ones(len(ACTIONS), dtype=float)
    behavior = np.clip(behavior, 0.0, None)
    total = float(behavior.sum())
    if total <= 0.0:
        behavior = np.ones(len(ACTIONS), dtype=float)
        total = float(behavior.sum())
    behavior = behavior / total

    archive_index = ACTIONS.index("ARCHIVE")
    if int(np.argmax(behavior)) != archive_index:
        return behavior
    feedback_indices = np.asarray(
        [ACTIONS.index("INTERRUPT"), ACTIONS.index("LATER")],
        dtype=int,
    )
    feedback_mass = float(behavior[feedback_indices].sum())
    if feedback_mass >= minimum_feedback:
        return behavior

    if np.isclose(feedback_mass, 0.0, rtol=0.0, atol=1e-12):
        feedback_ratio = np.asarray([0.5, 0.5], dtype=float)
    else:
        feedback_ratio = behavior[feedback_indices] / feedback_mass
    floored = behavior.copy()
    floored[feedback_indices] = minimum_feedback * feedback_ratio
    floored[archive_index] = 1.0 - minimum_feedback
    return floored


def run_method(
    seed: int,
    method: str,
    stream: list[Event],
    policy: StudentPolicy,
    rollout_writer,
    curve_writer,
    environment: NotificationRoutingEnvironment = DEFAULT_ENVIRONMENT,
    icl_examples: int = ICL_K,
    rag_examples: int = RAG_K,
    rag_text_weight: float = RAG_TEXT_WEIGHT,
    sdft_settings: OnlineSDFTSettings = DEFAULT_SDFT_SETTINGS,
    reinforce_settings: REINFORCESettings = DEFAULT_REINFORCE_SETTINGS,
    rft_settings: RFTSettings = DEFAULT_RFT_SETTINGS,
) -> dict:
    """Run one method with delayed, action-dependent feedback."""
    # Common random numbers pair serving exploration and simulator draws across
    # methods. Method-specific randomness is isolated to replay, so an extra
    # update cannot perturb a later action draw.
    action_rng = np.random.default_rng(seed * 1000 + 1)
    feedback_rng = np.random.default_rng(seed * 1000 + 2)
    learning_rng = np.random.default_rng(
        seed * 1000 + METHOD_RNG_OFFSETS[method]
    )
    agent = create_agent(
        method=method,
        policy=policy,
        icl_examples=icl_examples,
        rag_examples=rag_examples,
        rag_text_weight=rag_text_weight,
        sdft_settings=sdft_settings,
        reinforce_settings=reinforce_settings,
        rft_settings=rft_settings,
    )
    cumulative_regret = 0.0
    cumulative_correct = 0
    cumulative_observed_reward = 0.0
    phase_correct = Counter()
    phase_total = Counter()
    phase_regret = Counter()
    pending_feedback: list[dict] = []
    rollout_records: list[dict] = []
    reinforce_batch_records: list[dict] = []

    def release_ready_feedback(now_minute: int) -> None:
        """Apply only lessons whose real observation window has closed."""
        ready = [
            item
            for item in pending_feedback
            if item["available_at_minute"] <= now_minute
        ]
        pending_feedback[:] = [
            item
            for item in pending_feedback
            if item["available_at_minute"] > now_minute
        ]
        for item in ready:
            teacher_probs = None
            teacher_action = None
            rft_candidate_action = None
            rft_candidate_probs = None
            rft_candidate_entropy = None
            rft_candidate_uniform = None
            teacher_assessment = None
            teacher_view = None
            if agent.uses_teacher:
                teacher_view = environment.teacher_observation(
                    item["observation"],
                    item["action"],
                    project_factual_callback(item["feedback"]),
                )
                if teacher_view.observed_user_selection != "UNKNOWN":
                    teacher_probs = policy.teacher_probs(teacher_view)
                    teacher_action = int(np.argmax(teacher_probs))
                    if method == "RFT":
                        rft_candidate_probs = rft_sampling_distribution(
                            teacher_probs,
                            rft_settings.sampling_temperature,
                        )
                        rft_candidate_uniform = rft_event_uniform(
                            seed,
                            item["record"]["event_id"],
                            item["record"]["t"],
                        )
                        rft_candidate_action = rft_inverse_cdf_sample(
                            rft_candidate_probs,
                            rft_candidate_uniform,
                        )
                        positive = rft_candidate_probs[
                            rft_candidate_probs > 0.0
                        ]
                        rft_candidate_entropy = float(
                            -(positive * np.log(positive)).sum()
                        )
                    teacher_assessment = getattr(
                        policy,
                        "last_teacher_assessment",
                        None,
                    )

            agent.observe(
                item["observation"],
                item["action"],
                teacher_probs,
                teacher_action,
                item["feedback"],
                learning_rng,
                teacher_observation=teacher_view,
                decision_distribution=item["decision_distribution"],
                candidate_action=rft_candidate_action,
                behavior_distribution=item["behavior_distribution"],
            )

            record = item["record"]
            record["feedback_released_at_minute"] = now_minute
            if method in {"RFT", "Online-SDFT"}:
                support = causal_route_support(
                    item["action"],
                    item["feedback"],
                )
                record["causal_support"] = [
                    ACTIONS[index]
                    for index, allowed in enumerate(support)
                    if allowed
                ]
            if method == "RFT":
                record["rft_candidate_probs"] = (
                    None
                    if rft_candidate_probs is None
                    else dict(zip(ACTIONS, map(float, rft_candidate_probs)))
                )
                record["rft_candidate_entropy"] = rft_candidate_entropy
                record["rft_candidate_uniform"] = rft_candidate_uniform
                record["rft_candidate_action"] = (
                    None
                    if agent.last_rft_candidate_action is None
                    else ACTIONS[agent.last_rft_candidate_action]
                )
                record["rft_accepted"] = agent.last_rft_accepted
                record["rft_reason"] = agent.last_rft_reason
                record["rft_update_index"] = agent.online_update_count
                record["rft_updates_applied"] = (
                    agent.last_observation_update_count
                )
            if method == "Online-SDFT":
                record["sdft_update_index"] = agent.online_update_count
                record["sdft_updates_applied"] = (
                    agent.last_observation_update_count
                )
                record["sdft_objective"] = agent.settings.target_mode
                reliability = causal_evidence_reliability(
                    item["action"],
                    item["feedback"],
                )
                record["sdft_evidence_reliability"] = reliability
                if (
                    reliability != "censored_unknown"
                    and agent.settings.target_mode != "support_likelihood"
                ):
                    weights = agent.fusion_weights(
                        item["action"],
                        item["feedback"],
                    )
                    record["sdft_fusion_weights"] = dict(
                        zip(("teacher", "decision", "behavior"), weights)
                    )
            if method in {"ICL", "RAG"}:
                memory_reliability = causal_evidence_reliability(
                    item["action"],
                    item["feedback"],
                )
                record["memory_evidence_reliability"] = memory_reliability
                record["lesson_status"] = {
                    "reliable_singleton": "memory_prompt_available",
                    "ambiguous_digest_open": "memory_retained_ambiguous",
                    "censored_unknown": "memory_unlabeled",
                }[memory_reliability]
            elif item["feedback"]["observed_user_selection"] == "UNKNOWN":
                record["lesson_status"] = "censored_no_update"
            elif method == "REINFORCE":
                record["reinforce_training_reward"] = (
                    agent.last_training_reward
                )
                record["reinforce_batch_position"] = (
                    len(reinforce_batch_records) + 1
                )
                reinforce_batch_records.append(record)
                if agent.last_observation_update_count:
                    update_index = agent.online_update_count
                    for batch_record in reinforce_batch_records:
                        batch_record["reinforce_update_index"] = update_index
                        batch_record["lesson_status"] = "feedback_applied"
                    reinforce_batch_records.clear()
                else:
                    record["lesson_status"] = "feedback_buffered"
            elif method == "RFT":
                if record["rft_accepted"]:
                    record["lesson_status"] = (
                        "rft_target_applied"
                        if agent.last_observation_update_count
                        else "rft_target_buffered"
                    )
                else:
                    record["lesson_status"] = {
                        "ambiguous_unverified": (
                            "rft_rejected_ambiguous_support"
                        ),
                        "teacher_mismatch": (
                            "rft_rejected_teacher_candidate"
                        ),
                    }[record["rft_reason"]]
            else:
                if method == "Online-SDFT":
                    if (
                        record["sdft_evidence_reliability"]
                        == "ambiguous_digest_open"
                        and agent.settings.ambiguous_update_mode
                        in {"skip", "defer"}
                    ):
                        record["lesson_status"] = {
                            "skip": "ambiguous_target_skipped",
                            "defer": "ambiguous_target_deferred",
                        }[agent.settings.ambiguous_update_mode]
                    else:
                        target_kind = (
                            "support_target"
                            if agent.settings.target_mode == "support_likelihood"
                            else "soft_target"
                        )
                        record["lesson_status"] = (
                            f"{target_kind}_applied"
                            if agent.last_observation_update_count
                            else f"{target_kind}_buffered"
                        )
                else:
                    record["lesson_status"] = "observed_no_update"
            record["teacher_evidence"] = (
                None if teacher_view is None else teacher_view.evidence
            )
            record["teacher_probs"] = (
                None
                if teacher_probs is None
                else dict(zip(ACTIONS, map(float, teacher_probs)))
            )
            record["teacher_assessment"] = teacher_assessment
            record["teacher_rollout"] = (
                None
                if teacher_action is None
                else ACTIONS[teacher_action]
            )

    for step, event in enumerate(stream, start=1):
        current_minute = (step - 1) * DECISION_INTERVAL_MINUTES
        release_ready_feedback(current_minute)
        observation = environment.student_observation(event)
        student_probs = agent.action_probs(observation)
        behavior_mode = getattr(agent, "behavior_mode", "epsilon_greedy")
        behavior_epsilon = (
            getattr(agent, "behavior_epsilon", EXPLORATION_EPSILON)
            if method == "Online-SDFT"
            else EXPLORATION_EPSILON
        )
        behavior_epsilon_half_life = (
            getattr(agent, "behavior_epsilon_half_life", None)
            if method == "Online-SDFT"
            else None
        )
        if behavior_epsilon_half_life is not None:
            behavior_epsilon = float(behavior_epsilon) * 2.0 ** (
                -(step - 1) / float(behavior_epsilon_half_life)
            )
        samples_on_archive = (
            behavior_mode == "archive_policy_sampling"
            and int(np.argmax(student_probs)) == ACTIONS.index("ARCHIVE")
        )
        if (
            agent.samples_from_policy
            or behavior_mode == "policy_sampling"
            or samples_on_archive
        ):
            behavior_probs = policy_sampling(student_probs)
        elif behavior_mode == "archive_uniform_probe":
            behavior_probs = archive_uniform_probe(
                student_probs,
                getattr(agent, "archive_probe_mix", 0.0),
                behavior_epsilon,
            )
        elif behavior_mode == "archive_policy_feedback_floor":
            behavior_probs = archive_policy_feedback_floor(
                student_probs,
                getattr(agent, "archive_policy_min_feedback", 0.0),
            )
        elif behavior_mode == "uncertainty_interrupt_probe":
            behavior_probs = uncertainty_interrupt_probe(
                student_probs,
                step,
                getattr(agent, "interrupt_probe_mix", 0.0),
                getattr(agent, "interrupt_probe_half_life", None),
                getattr(agent, "interrupt_probe_max_confidence", 1.0),
                behavior_epsilon,
            )
        else:
            behavior_probs = epsilon_greedy(student_probs, behavior_epsilon)
        exploration_taper_weight = 1.0
        exploration_taper_start_step = (
            getattr(agent, "exploration_taper_start_step", None)
            if method == "Online-SDFT"
            else None
        )
        if (
            exploration_taper_start_step is not None
            and step > exploration_taper_start_step
        ):
            exploration_taper_half_life = getattr(
                agent,
                "exploration_taper_half_life",
                None,
            )
            exploration_taper_weight = 2.0 ** (
                -(step - exploration_taper_start_step)
                / float(exploration_taper_half_life)
            )
            greedy_behavior = one_hot(
                int(np.argmax(student_probs)),
                len(ACTIONS),
            )
            behavior_probs = (
                exploration_taper_weight * behavior_probs
                + (1.0 - exploration_taper_weight) * greedy_behavior
            )
            behavior_probs = behavior_probs / behavior_probs.sum()
        feedback_propensity = feedback_surface_propensity(behavior_probs)
        action = int(
            action_rng.choice(len(ACTIONS), p=behavior_probs)
        )

        # Freeze evaluation before any factual outcome or update exists.
        utilities = environment.oracle_utilities(event)
        gold_action = environment.gold_action(event)
        utility_optimal_action = int(np.argmax(utilities))
        step_regret = float(
            utilities[utility_optimal_action] - utilities[action]
        )
        correct = int(action == gold_action)
        cumulative_regret += step_regret
        cumulative_correct += correct
        phase_correct[event.phase] += correct
        phase_total[event.phase] += 1
        phase_regret[event.phase] += step_regret

        # Execute only the chosen route. The simulator samples the future
        # callback now but the learner cannot consume it until its declared
        # action-specific observation window has elapsed.
        feedback = environment.execute(event, action, feedback_rng)
        observed_reward = float(feedback["reward"])
        cumulative_observed_reward += observed_reward
        feedback_available_at = current_minute + int(
            feedback["delay_minutes"]
        )

        record = {
            "seed": seed,
            "method": method,
            "t": step,
            "event_id": event.event_id,
            "phase": event.phase,
            "regime": REGIMES[event.phase],
            "category": event.category,
            "notification_title": event.title,
            "notification_body": event.body,
            "decision_time_minute": current_minute,
            "student_probs": dict(
                zip(ACTIONS, map(float, student_probs))
            ),
            "behavior_mode": behavior_mode,
            "behavior_epsilon": float(behavior_epsilon),
            "exploration_taper_weight": (
                float(exploration_taper_weight)
                if method == "Online-SDFT"
                else None
            ),
            "behavior_probs": dict(
                zip(ACTIONS, map(float, behavior_probs))
            ),
            "action": ACTIONS[action],
            "feedback": feedback,
            "feedback_available_at_minute": feedback_available_at,
            "feedback_released_at_minute": None,
            "lesson_status": "pending",
            "reinforce_training_reward": None,
            "reinforce_update_index": None,
            "reinforce_batch_position": None,
            "rft_candidate_action": None,
            "rft_candidate_probs": None,
            "rft_candidate_entropy": None,
            "rft_candidate_uniform": None,
            "rft_accepted": None,
            "rft_reason": None,
            "rft_update_index": None,
            "rft_updates_applied": None,
            "sdft_update_index": None,
            "sdft_updates_applied": None,
            "sdft_prompt_examples_used": (
                agent.last_prompt_examples_used
                if method == "Online-SDFT"
                else None
            ),
            "causal_support": None,
            "sdft_evidence_reliability": None,
            "sdft_objective": None,
            "sdft_fusion_weights": None,
            "sdft_feedback_propensity": (
                feedback_propensity if method == "Online-SDFT" else None
            ),
            "sdft_propensity_weight": (
                min(
                    sdft_settings.propensity_weight_cap,
                    1.0 / max(feedback_propensity, 1e-8),
                )
                if method == "Online-SDFT"
                and sdft_settings.propensity_weight_mode
                == "feedback_surface_snips"
                else None
            ),
            "teacher_evidence": None,
            "teacher_probs": None,
            "teacher_assessment": None,
            "teacher_rollout": None,
            "gold_action_scoring_only": ACTIONS[gold_action],
            "gold_action_distribution_scoring_only": dict(
                zip(
                    ACTIONS,
                    map(float, environment.gold_action_distribution(event)),
                )
            ),
            "utility_optimal_action_scoring_only": ACTIONS[
                utility_optimal_action
            ],
            "correct_online": correct,
            "observed_feedback_reward": observed_reward,
            "step_regret": step_regret,
            "cum_regret": cumulative_regret,
            "cum_accuracy": cumulative_correct / step,
            "cum_observed_reward": cumulative_observed_reward,
        }
        rollout_records.append(record)
        pending_feedback.append(
            {
                "available_at_minute": feedback_available_at,
                "event": event,
                "observation": observation,
                "action": action,
                "feedback": feedback,
                "decision_distribution": student_probs.copy(),
                "behavior_distribution": behavior_probs.copy(),
                "feedback_propensity": feedback_propensity,
                "record": record,
            }
        )
        if step % 20 == 0:
            print(
                f"  {method} t={step}/{len(stream)} "
                f"acc={cumulative_correct / step:.3f}",
                flush=True,
            )

        curve_writer.writerow(
            {
                "seed": seed,
                "method": method,
                "t": step,
                "phase": event.phase,
                "regime": REGIMES[event.phase],
                "step_correct": correct,
                "step_feedback_reward": observed_reward,
                "step_regret": step_regret,
                "cum_accuracy": cumulative_correct / step,
                "cum_regret": cumulative_regret,
                "cum_observed_reward": cumulative_observed_reward,
            }
        )

    # Do not flush future feedback after the evaluation horizon: a real model
    # could use it later, but it cannot improve any action scored in this run.
    for record in reinforce_batch_records:
        if record["lesson_status"] == "feedback_buffered":
            record["lesson_status"] = "feedback_gradient_unflushed_at_horizon"
    for record in rollout_records:
        if record["lesson_status"] == "pending":
            record["lesson_status"] = "pending_after_horizon"
        rollout_writer.write(json.dumps(record) + "\n")
    rollout_writer.flush()

    return {
        "seed": seed,
        "method": method,
        "online_accuracy": cumulative_correct / len(stream),
        "cum_regret": cumulative_regret,
        "regret_per_decision": cumulative_regret / len(stream),
        "cumulative_observed_reward": cumulative_observed_reward,
        "observed_reward_per_decision": (
            cumulative_observed_reward / len(stream)
        ),
        **{
            f"online_accuracy_{REGIMES[phase]}": (
                phase_correct[phase] / phase_total[phase]
                if phase_total[phase]
                else 0.0
            )
            for phase in range(3)
        },
        **{
            f"regret_{REGIMES[phase]}": phase_regret[phase]
            for phase in range(3)
        },
    }


def experiment_config(
    seeds: int,
    seed_start: int,
    model_id: str,
    policy: LiquidLLMPolicy,
    prompt_style: str = PROMPT_STYLE,
    icl_examples: int = ICL_K,
    rag_examples: int = RAG_K,
    rag_text_weight: float = RAG_TEXT_WEIGHT,
    sdft_settings: OnlineSDFTSettings = DEFAULT_SDFT_SETTINGS,
    environment: NotificationRoutingEnvironment = DEFAULT_ENVIRONMENT,
    reinforce_settings: REINFORCESettings = DEFAULT_REINFORCE_SETTINGS,
    rft_settings: RFTSettings = DEFAULT_RFT_SETTINGS,
) -> dict:
    dataset_fingerprint = environment.stream_fingerprint(
        range(seed_start, seed_start + seeds)
    )
    sdft_trainable_parameters = int(policy.trainable_parameters)
    rft_student_settings = rft_settings.student_settings

    def peft_architecture(settings: OnlineSDFTSettings) -> dict:
        targets = list(settings.lora_target_modules)
        layers = settings.lora_layers_to_transform
        serialized_layers = None if layers is None else list(layers)
        return {
            "implementation": "peft.LoraConfig + peft.get_peft_model",
            "peft_type": "LORA",
            "task_type": "CAUSAL_LM",
            "r": settings.lora_rank,
            "lora_alpha": settings.lora_alpha,
            "lora_dropout": settings.lora_dropout,
            "target_modules": targets,
            "layers_to_transform": serialized_layers,
            "layers_pattern": None if layers is None else "layers",
            "bias": "none",
            "init_lora_weights": True,
            "ensure_weight_tying": False,
            "merged_for_serving": False,
        }

    sdft_peft_architecture = peft_architecture(sdft_settings)
    reinforce_peft_architecture = sdft_peft_architecture.copy()
    rft_peft_architecture = peft_architecture(rft_student_settings)
    same_rft_architecture = (
        rft_peft_architecture == sdft_peft_architecture
    )
    trainability_fields = (
        "lora_a_learning_rate_scale",
        "lm_head_lora_a_learning_rate_scale",
    )
    same_rft_adapter_capacity = same_rft_architecture and all(
        getattr(rft_student_settings, field) == getattr(sdft_settings, field)
        for field in trainability_fields
    )
    rft_trainable_parameters = (
        sdft_trainable_parameters if same_rft_adapter_capacity else None
    )
    replay_schedule_fields = (
        "replay_size",
        "replay_prompt_examples",
        "batch_size",
        "update_steps",
        "warmup_examples",
        "ambiguous_replay_group_weight",
        "teacher_temperature",
        "reasoning_tokens",
        "replay_strategy",
        "replay_recency_half_life",
        "ambiguous_update_mode",
        "force_newest_every_step",
        "base_kl_weight",
        "behavior_mode",
        "behavior_epsilon",
        "behavior_epsilon_half_life",
        "exploration_taper_start_step",
        "exploration_taper_half_life",
        "archive_probe_mix",
        "archive_policy_min_feedback",
        "interrupt_probe_mix",
        "interrupt_probe_half_life",
        "interrupt_probe_max_confidence",
        "propensity_weight_mode",
        "propensity_weight_cap",
    )
    same_rft_replay_schedule = all(
        getattr(rft_student_settings, field) == getattr(sdft_settings, field)
        for field in replay_schedule_fields
    )
    optimizer_fields = (
        "learning_rate",
        "optimizer_weight_decay",
        "optimizer_beta1",
        "max_grad_norm",
        "lm_head_learning_rate",
        "lora_a_learning_rate_scale",
        "lm_head_lora_a_learning_rate_scale",
    )
    same_rft_optimizer_hyperparameters = all(
        getattr(rft_student_settings, field) == getattr(sdft_settings, field)
        for field in optimizer_fields
    )
    return {
        "seeds": seeds,
        "seed_start": seed_start,
        "dataset_version": DATASET_VERSION,
        "dataset_numpy_version": DATASET_NUMPY_VERSION,
        "runtime_numpy_version": np.__version__,
        "dataset_fingerprint": dataset_fingerprint,
        "method_dataset_versions": {
            method: DATASET_VERSION for method in METHODS
        },
        "method_dataset_fingerprints": {
            method: dataset_fingerprint for method in METHODS
        },
        "paired_method_stream": (
            "one immutable generated event stream per seed is reused by all "
            "methods before any method-specific action or feedback randomness"
        ),
        "gold_action_sampling": {
            "distribution": (
                "probability-power temperature scaling of normalized "
                "evaluator-utility weights"
            ),
            "negative_utility_handling": (
                "subtract the event minimum only when it is negative"
            ),
            "temperature": PREFERENCE_SAMPLING_TEMPERATURE,
            "exponent": 1.0 / PREFERENCE_SAMPLING_TEMPERATURE,
            "formula": (
                "q_i = p_i**(1 / temperature) / "
                "sum_j p_j**(1 / temperature)"
            ),
            "numerical_implementation": (
                "stable log space with exact zero support preserved"
            ),
            "draw_timing": "once per event from an isolated seeded RNG",
        },
        "notification_context": (
            "decision-relevant synthetic title/body + category + local time "
            "+ regime + local importance"
        ),
        "stream_length": STREAM_LENGTH,
        "phase_length": PHASE_LENGTH,
        "decision_interval_minutes": DECISION_INTERVAL_MINUTES,
        "digest_delivery_delay_minutes": DIGEST_DELIVERY_DELAY_MINUTES,
        "feedback_windows_minutes": FEEDBACK_WINDOWS_MINUTES,
        "feedback_observation_matrix": {
            "INTERRUPT": (
                "immediate open -> INTERRUPT; delayed read -> LATER; "
                "notification deletion -> ARCHIVE"
            ),
            "LATER": (
                "digest open -> LATER, including after a missed immediate "
                "preference; digest deletion -> ARCHIVE"
            ),
            "ARCHIVE": "no delivered surface -> UNKNOWN",
        },
        "observed_reward": dict(OBSERVED_OUTCOME_REWARDS),
        "actions": ACTIONS,
        "methods": METHODS,
        "student_model": model_id,
        "student_policy": (
            "next-token A/B/C probabilities; REINFORCE, RFT, and "
            "Online-SDFT each train the same reset-per-arm PEFT LoRA adapter"
        ),
        "adaptive_model_sharing": (
            "one physical Liquid model instance is reused sequentially; its "
            "LoRA adapter is reset to the identical initialization before "
            "every method and seed. Only RFT and Online-SDFT disable the "
            "adapter for same-model hindsight teaching"
        ),
        "student_backbone": "frozen Liquid LFM base weights",
        "student_backbone_trainable_parameters": 0,
        "device": str(policy.device),
        "exploration_epsilon": EXPLORATION_EPSILON,
        "behavior_policy": (
            "Base, ICL, RAG, and RFT use 6% epsilon-greedy; REINFORCE "
            "samples its current LoRA policy; Online-SDFT uses its configured "
            "uncertainty-triggered INTERRUPT probe and post-step-160 taper"
        ),
        "randomness_pairing": (
            "common per-seed action and feedback RNG streams across methods; "
            "method-specific RNG only for learning internals; RFT teacher "
            "candidate sampling uses an event-keyed deterministic uniform "
            "that cannot "
            "shift action, feedback, or replay draws"
        ),
        "replay_size": REPLAY_SIZE,
        "online_batch_size": ONLINE_BATCH_SIZE,
        "prompt_style": prompt_style,
        "history_rendering": (
            "alternating notification/route demonstrations from reliable "
            "singleton callbacks only; ambiguous and UNKNOWN interactions "
            "remain retained but are not prompted"
        ),
        "teacher_prompt_version": TEACHER_PROMPT_VERSION,
        "prompt_token_budget": PROMPT_TOKEN_BUDGET,
        "icl_examples": icl_examples,
        "rag_examples": rag_examples,
        "rag_text_weight": rag_text_weight,
        "rag_similarity": (
            f"{1.0 - rag_text_weight:.2f} metadata similarity (equal-weight "
            "category/regime match + importance + circular hour) + "
            f"{rag_text_weight:.2f} visible title/body token Jaccard similarity"
        ),
        "reinforce_lr": reinforce_settings.learning_rate,
        "reinforce_batch_size": reinforce_settings.batch_size,
        "reinforce_policy": (
            "action-token REINFORCE trains only the same reset LFM PEFT LoRA "
            "adapter used by RFT and Online-SDFT"
        ),
        "reinforce_trainable_parameters": sdft_trainable_parameters,
        "online_reinforce_peft_architecture": reinforce_peft_architecture,
        "reinforce_initialization": (
            "the common PEFT LoRA initialization restored before this arm"
        ),
        "online_reinforce_settings": reinforce_settings.to_dict(),
        "reinforce_training_reward": {
            "source": "matured executed-surface factual outcome only",
            "outcome_map": dict(reinforce_settings.reward_outcome_map),
            "unknown_selection": (
                "censored before outcome mapping; no gradient target"
            ),
            "reported_metric": (
                "learner-only shaping; rollout observed_feedback_reward and "
                "cumulative_observed_reward use the shared observed_reward map"
            ),
        },
        "reinforce_optimizer": (
            f"AdamW autograd over exactly {reinforce_settings.batch_size} "
            "newly matured known factual-outcome callbacks; learner reward "
            "from reinforce_training_reward outcome map; each row consumed "
            "once; no replay; incomplete horizon batch not flushed"
        ),
        "reinforce_baseline": (
            "fixed zero causal baseline; step=0.0"
            if reinforce_settings.baseline_step == 0.0
            else f"causal reward EMA; step={reinforce_settings.baseline_step}"
        ),
        "reinforce_entropy_coef": reinforce_settings.entropy_coef,
        "reinforce_max_grad_norm": reinforce_settings.max_grad_norm,
        "reinforce_capacity_match": {
            "settings_source": "online_sdft_settings LoRA architecture",
            "same_physical_model": True,
            "same_frozen_base_model": True,
            "same_lora_architecture": True,
            "same_adapter_parameter_count": True,
            "same_adapter_initialization": True,
            "adapter_reset_before_arm": True,
            "uses_hindsight_teacher": False,
        },
        "rft_protocol_version": RFT_PROTOCOL_VERSION,
        "rft_settings_provenance": RFT_SETTINGS_PROVENANCE,
        "online_rft_settings": rft_settings.to_dict(),
        "online_rft_peft_architecture": rft_peft_architecture,
        "online_rft_trainable_parameters": rft_trainable_parameters,
        "rft_candidate_policy": (
            f"K={rft_settings.candidate_count} "
            f"{rft_settings.sampling_mode} sample at temperature "
            f"{rft_settings.sampling_temperature:g} from the same model with "
            "its adapter disabled, which is the fixed-initial hindsight "
            "distribution used by Online-SDFT"
        ),
        "rft_candidate_sampler": {
            "scheme": RFT_CANDIDATE_SAMPLER,
            "key_fields": [
                "seed",
                "event_id",
                "t",
                "rft_candidate_rng_offset",
            ],
            "inverse_cdf": True,
            "stateful_rng": False,
        },
        "rft_acceptance_filter": (
            "accept only when the delayed public causal support is a reliable "
            "singleton containing the sampled teacher route; reject teacher "
            "mismatches and every ambiguous digest open; UNKNOWN is censored"
        ),
        "rft_target": (
            "one-hot cross-entropy target on the accepted sampled teacher "
            "route; rejected and censored rows never enter replay"
        ),
        "rft_capacity_match": {
            "settings_source": "online_rft_settings.student_settings",
            "same_frozen_base_model": True,
            "same_lora_architecture": same_rft_architecture,
            "same_adapter_parameter_count": same_rft_adapter_capacity,
            "same_adapter_initialization": same_rft_architecture,
            "same_replay_schedule": same_rft_replay_schedule,
            "same_optimizer_hyperparameters": (
                same_rft_optimizer_hyperparameters
            ),
            "same_adapter_disabled_teacher": True,
            "same_teacher_forward_budget": (
                "one hindsight distribution for each matured non-UNKNOWN "
                "callback"
            ),
            "configured_differences": (
                "temperature-8 sample-filter hard targets and a tuned 7e-4 "
                "learning rate with replay-32 epsilon-greedy serving versus "
                "Online-SDFT's replay-64 recency/probe/taper schedule and "
                "reliability-conditioned soft targets"
            ),
        },
        "rft_candidate_rng_offset": RFT_CANDIDATE_RNG_OFFSET,
        "sdft_lr": sdft_settings.learning_rate,
        "sdft_replay_size": sdft_settings.replay_size,
        "sdft_replay_prompt_examples": sdft_settings.replay_prompt_examples,
        "sdft_batch_size": sdft_settings.batch_size,
        "sdft_update_steps": sdft_settings.update_steps,
        "sdft_warmup_examples": sdft_settings.warmup_examples,
        "sdft_distill_temperature": SDFT_DISTILL_TEMPERATURE,
        "online_sdft_settings": sdft_settings.to_dict(),
        "adaptive_default_configuration_provenance": {
            "Online-SDFT": {
                "selection_seeds": [0, 1, 2],
                "disjoint_confirmation": False,
                "interpretation": (
                    "user-requested in-sample tuning on the canonical streams"
                ),
            },
            "RFT": {
                "selection_seeds": [1200],
                "disjoint_confirmation": False,
                "interpretation": (
                    "single-stream temperature and learning-rate screen"
                ),
            },
            "REINFORCE": {
                "selection_seeds": [0, 1, 2],
                "disjoint_confirmation": False,
                "selection_rule": (
                    "strict candidate gate: pooled exact-action correct > "
                    "Base and pooled total regret < Base; rank accuracy first, "
                    "then regret"
                ),
                "interpretation": (
                    "user-requested in-sample tuning on the canonical streams; "
                    "no disjoint confirmation set"
                ),
            },
        },
        "online_sdft_peft_architecture": sdft_peft_architecture,
        "online_sdft_student": (
            "PEFT LoRA adapter trained online on the same Liquid LFM used "
            "for hindsight teaching; base-model weights stay frozen and the "
            "adapter is never merged"
        ),
        "online_sdft_trainable_parameters": sdft_trainable_parameters,
        "online_sdft_optimizer": {
            "type": "AdamW",
            "learning_rate": sdft_settings.learning_rate,
            "lm_head_learning_rate": sdft_settings.lm_head_learning_rate,
            "lora_a_learning_rate_scale": (
                sdft_settings.lora_a_learning_rate_scale
            ),
            "lm_head_lora_a_learning_rate_scale": (
                sdft_settings.lm_head_lora_a_learning_rate_scale
            ),
            "weight_decay": sdft_settings.optimizer_weight_decay,
            "max_grad_norm": sdft_settings.max_grad_norm,
            "batch_size": sdft_settings.batch_size,
            "update_steps_per_release": sdft_settings.update_steps,
        },
        "sdft_replay_prompt_policy": (
            "most-recent reliable singleton rows from the bounded SDFT "
            "training replay in FIFO release order; no similarity retrieval; "
            "ambiguous, UNKNOWN, teacher, and soft-target data excluded"
            if sdft_settings.replay_prompt_examples
            else "disabled; serving uses only learned finite parameters"
        ),
        "sdft_evidence_reliability": {
            "reliable_singleton": {
                "callbacks": (
                    "immediate/delayed interrupt open or deletion from a "
                    "delivered surface"
                ),
                "teacher_weight": sdft_settings.reliable_teacher_weight,
                "decision_weight": sdft_settings.reliable_decision_weight,
                "behavior_weight": sdft_settings.reliable_behavior_weight,
            },
            "ambiguous_digest_open": {
                "callbacks": (
                    "digest open supports INTERRUPT or LATER without "
                    "distinguishing them"
                ),
                "teacher_weight": sdft_settings.ambiguous_teacher_weight,
                "decision_weight": sdft_settings.ambiguous_decision_weight,
                "behavior_weight": sdft_settings.ambiguous_behavior_weight,
            },
            "censored_unknown": "no target or update",
        },
        "teacher_model": model_id,
        "teacher_student_model_sharing": {
            "model_instances": 1,
            "same_base_parameters": True,
            "student_forward": "LoRA adapter enabled",
            "teacher_forward": "same model with LoRA adapter disabled",
            "teacher_reference": "fixed initial frozen base-model policy",
            "separate_teacher_checkpoint": False,
        },
        "teacher_policy": (
            "the student and teacher are one shared Liquid LFM model; teacher "
            "forwards disable the student's LoRA adapter, so the frozen base "
            "is the fixed-initial reference for the full run; notification "
            "title/body + metadata + decision-time importance + delayed "
            "observed user selection are supplied for the executed route "
            "only; explicit delivery-surface guidance keeps a digest open "
            "ambiguous between INTERRUPT and LATER; UNKNOWN stays censored; "
            "no scalar reward, counterfactual, shadow policy, or evaluator "
            "label"
        ),
        "teacher_temperature": sdft_settings.teacher_temperature,
        "evaluation": (
            "prequential one-stream; sampled-preference accuracy and "
            "utility-optimal regret; predict then learn"
        ),
        "update_timing": (
            "feedback is queued until its action-specific observation window "
            "closes; no end-of-horizon flush"
        ),
        "learning_signal": (
            "evidence-reliability-conditioned same-LM soft hindsight target "
            "for Online-SDFT; the teacher's argmax is diagnostic only; "
            "RFT draws one separate categorical teacher candidate and keeps "
            "its hard target only after reliable singleton verification; "
            "ICL/RAG retain direct completed interactions but prompt only "
            "reliable singleton route evidence; learner-only scalar mapped "
            "from the matured executed-surface factual outcome for "
            "REINFORCE, while reported observed reward retains the shared "
            "environment map; hidden sampled preference for shared accuracy "
            "only; utility-optimal route for regret only"
        ),
    }

"""Aggregation, qualitative selection, and plotting.

This module reads completed chronological traces. It never participates in an
action or update, which keeps evaluation-only gold fields out of methods.
"""


import csv
import json
import math
from collections import Counter, defaultdict

import numpy as np



def mean_ci(values: list[float]) -> dict:
    array = np.asarray(values, dtype=float)
    if len(array) == 1:
        return {"mean": float(array[0]), "std": 0.0, "ci95": 0.0}
    std = float(array.std(ddof=1))
    return {
        "mean": float(array.mean()),
        "std": std,
        "ci95": float(1.96 * std / math.sqrt(len(array))),
    }


def summarize_metrics(metrics: list[dict]) -> dict:
    metric_names = [
        key
        for key in metrics[0]
        if key not in {"seed", "method"}
    ]
    return {
        method: {
            metric: mean_ci(
                [
                    float(row[metric])
                    for row in metrics
                    if row["method"] == method
                ]
            )
            for metric in metric_names
        }
        for method in METHODS
    }


def find_qualitative_examples(
    rollouts: list[dict],
    limit: int = 8,
) -> list[dict]:
    """Select later steps uniquely solved by Online-SDFT."""
    def public_feedback(row: dict) -> dict:
        feedback = dict(row["feedback"])
        if "match_status" in feedback:
            feedback["observed_surface_match_status"] = feedback.pop(
                "match_status"
            )
        return feedback

    grouped = defaultdict(dict)
    for row in rollouts:
        grouped[(row["seed"], row["t"])][row["method"]] = row

    qualitative = []
    for (seed, step), rows in sorted(grouped.items()):
        if (
            step > PHASE_LENGTH
            and len(rows) == len(METHODS)
            and rows["Online-SDFT"]["correct_online"] == 1
            and sum(
                rows[method]["correct_online"]
                for method in METHODS[:-1]
            )
            == 0
        ):
            qualitative.append(
                {
                    "seed": seed,
                    "t": step,
                    "regime": rows["Online-SDFT"]["regime"],
                    "category": rows["Online-SDFT"]["category"],
                    "notification_title": rows["Online-SDFT"][
                        "notification_title"
                    ],
                    "notification_body": rows["Online-SDFT"][
                        "notification_body"
                    ],
                    "gold_action_scoring_only": rows["Online-SDFT"][
                        "gold_action_scoring_only"
                    ],
                    "methods": {
                        method: {
                            "action": rows[method]["action"],
                            "feedback": public_feedback(rows[method]),
                            "teacher_rollout": rows[method][
                                "teacher_rollout"
                            ],
                            "sdft_evidence_reliability": rows[method].get(
                                "sdft_evidence_reliability"
                            ),
                            "sdft_fusion_weights": rows[method].get(
                                "sdft_fusion_weights"
                            ),
                            "rft_candidate_action": rows[method].get(
                                "rft_candidate_action"
                            ),
                            "rft_accepted": rows[method].get("rft_accepted"),
                            "rft_reason": rows[method].get("rft_reason"),
                        }
                        for method in METHODS
                    },
                }
            )
        if len(qualitative) >= limit:
            break
    return qualitative


def _rft_diagnostic_counts(rows: list[dict]) -> dict:
    """Aggregate auditable RFT filtering and optimizer counts."""
    allowed_reasons = {
        None,
        "accepted",
        "teacher_mismatch",
        "ambiguous_unverified",
        "censored_unknown",
    }
    observed_reasons = {row.get("rft_reason") for row in rows}
    if not observed_reasons <= allowed_reasons:
        raise ValueError("unknown RFT audit reason")
    if any(
        row.get("rft_accepted") is True
        and row.get("rft_reason") != "accepted"
        for row in rows
    ):
        raise ValueError("accepted RFT rows must use the accepted reason")
    attempted = sum(row.get("rft_candidate_action") is not None for row in rows)
    accepted = sum(row.get("rft_accepted") is True for row in rows)
    rejected = sum(row.get("rft_accepted") is False for row in rows)
    rejection_reasons = Counter(
        row["rft_reason"]
        for row in rows
        if row.get("rft_accepted") is False
        and row.get("rft_reason") is not None
    )
    if attempted != accepted + rejected:
        raise ValueError("RFT attempted rows must partition into accepted/rejected")
    if rejected != sum(rejection_reasons.values()):
        raise ValueError("RFT rejection reasons must account for every rejection")
    proposal_actions = [
        row["rft_candidate_action"]
        for row in rows
        if row.get("rft_candidate_action") is not None
    ]
    accepted_actions = [
        row["rft_candidate_action"]
        for row in rows
        if row.get("rft_accepted") is True
    ]
    if any(action not in ACTIONS for action in proposal_actions):
        raise ValueError("RFT proposal counts contain an unknown route")
    if any(action not in ACTIONS for action in accepted_actions):
        raise ValueError("RFT accepted counts contain an unknown route")
    proposal_counts = {
        action: proposal_actions.count(action)
        for action in ACTIONS
    }
    accepted_counts = {
        action: accepted_actions.count(action)
        for action in ACTIONS
    }
    if sum(proposal_counts.values()) != attempted:
        raise ValueError("RFT proposal route counts must account for every attempt")
    if sum(accepted_counts.values()) != accepted:
        raise ValueError("RFT accepted route counts must account for every acceptance")
    entropies = []
    for row in rows:
        candidate = row.get("rft_candidate_action")
        entropy = row.get("rft_candidate_entropy")
        if candidate is None:
            if entropy is not None:
                raise ValueError("RFT proposal entropy requires a sampled route")
            continue
        if entropy is None or isinstance(entropy, bool):
            raise ValueError("RFT sampled routes require proposal entropy")
        value = float(entropy)
        if (
            not math.isfinite(value)
            or not 0.0 <= value <= math.log(len(ACTIONS)) + 1e-12
        ):
            raise ValueError(
                "RFT proposal entropy is outside the categorical range"
            )
        entropies.append(value)
    if len(entropies) != attempted:
        raise ValueError("RFT proposal entropy must account for every attempt")
    update_count = max(
        (int(row.get("rft_update_index") or 0) for row in rows),
        default=0,
    )
    return {
        "attempted": attempted,
        "accepted": accepted,
        "rejected": rejected,
        "rejection_reasons": dict(sorted(rejection_reasons.items())),
        "acceptance_rate": accepted / attempted if attempted else None,
        "proposal_counts": proposal_counts,
        "accepted_counts": accepted_counts,
        "mean_proposal_entropy": (
            float(np.mean(entropies)) if entropies else None
        ),
        "update_count": update_count,
        "censored_unknown": sum(
            row.get("rft_reason") == "censored_unknown"
            for row in rows
        ),
        "pending_after_horizon": sum(
            row.get("lesson_status") == "pending_after_horizon"
            for row in rows
        ),
    }


def summarize_rft_diagnostics(rollouts: list[dict]) -> dict:
    """Return per-seed and total RFT rejection diagnostics."""
    rft_rows = [row for row in rollouts if row.get("method") == "RFT"]
    by_seed = defaultdict(list)
    for row in rft_rows:
        by_seed[int(row["seed"])].append(row)
    per_seed = {
        str(seed): _rft_diagnostic_counts(rows)
        for seed, rows in sorted(by_seed.items())
    }
    total = _rft_diagnostic_counts(rft_rows)
    total["update_count"] = sum(
        counts["update_count"] for counts in per_seed.values()
    )
    return {"per_seed": per_seed, "total": total}


def write_compact_results(
    output_dir: Path,
    config: dict,
    metrics: list[dict],
    rollouts: list[dict],
) -> dict:
    summary = summarize_metrics(metrics)
    qualitative = find_qualitative_examples(rollouts)
    (output_dir / "qualitative_examples.json").write_text(
        json.dumps(qualitative, indent=2) + "\n"
    )
    payload = {
        "config": config,
        "summary": summary,
        "qualitative_examples": len(qualitative),
        "rft_diagnostics": summarize_rft_diagnostics(rollouts),
    }
    (output_dir / "summary.json").write_text(
        json.dumps(payload, indent=2) + "\n"
    )
    return summary


### 5.3 Choose strict or portable execution

This cell keeps strict mode enabled on a local kernel and selects portable mode
automatically on Colab. Portable CUDA/CPU runs use the exact bundled canonical
events, but their final byte comparison is diagnostic rather than a release
claim because their numerical kernels differ from MPS.

In [ ]:
# Strict mode remains the local Apple-MPS/FP32 default. Colab selects portable
# CUDA/CPU mode automatically because it cannot satisfy the MPS byte gate.
STRICT_BYTE_REPRODUCTION = not bool(globals().get("in_colab", False))
REPRO_OUTPUT_DIR = Path("notebook_reproduction_outputs") / "bandit"

### 5.4 Run every seed and method

The pinned model snapshot is loaded fresh for each seed. Within a seed, all six
methods reuse one physical model and reset the common LoRA snapshot before acting.

In [ ]:
# @title Run the canonical six-method benchmark in memory { display-mode: "form" }
if not globals().get("_ONLINE_SDFT_RUNTIME_READY", False):
    raise RuntimeError(
        "Section 5.1 did not finish successfully. Run the setup cell and "
        "follow its recovery instruction before loading or running Online-SDFT."
    )

REFERENCE_ARTIFACT_NAMES = ('per_seed_metrics.csv', 'summary.json', 'qualitative_examples.json')
CANONICAL_REPRODUCTION = {'seeds': (0, 1, 2), 'model_revision': '13a53837c4906b4f7405932532ba85d182bb013b', 'runtime': {'python': '3.11.9', 'numpy': '2.4.6', 'torch': '2.13.0', 'transformers': '5.13.1', 'peft': '0.19.1', 'device': 'mps', 'dtype': 'float32'}, 'dataset_fingerprint': '986cdf1a7d5fcc04c2b33f1bf90a1fc4f24a97ee85e663370382d8a67e4c932d', 'stream_bundle': {'format': 'online-sdft-event-stream-v1-float-hex', 'event_count': 720, 'sha256': '0d5758bbb42bcb51a04b79510ddc05bc8238b8f2fafc1ce0ad282b9b1497fa9b', 'zlib_base64': '...'}, 'source_fingerprint': {'algorithm': 'sha256', 'files': {'online_sdft/__init__.py': '12f2462b5c4a0c6ad3a7e2ae76e6f49dc16c14cb60177df9ae3ab22a537eeb1d', 'online_sdft/config.py': '11bb28cd9ce56ac6de910060c77ee71be195e53f0a262b16e240a1856c496cdc', 'online_sdft/environment.py': '027196ea15e1e289384cba165ec80b80cb6b14be2009fcbb8d1104328d44853f', 'online_sdft/experiment.py': '140023f7e9f89d7902e74291bd82a33e2591f22d9357f1ed77c7b8f6e4569d67', 'online_sdft/methods.py': 'b475f0cb6891211461d929633e6d80faddb312ae918310542e373e429d3a3265', 'online_sdft/privilege.py': '81cfb83dfa58f10e7a039e2214eaf24834af5c31c55de46ed2219fcc286437b6', 'online_sdft/reporting.py': '7f2b6ee59278a275203c1bf274d51d19fa2b87151637de541a5616f4937a9d78', 'requirements.txt': 'bc99d57546d0c65338ad87e02bcaaa639e6415b1a7068ae43c09059cf2bad3ce', 'run.py': '14f00e7f12a472310910b12ae258ba42f7fce7874decfc86595e6ee7064fd3fc', 'scripts/run_mps_seeds_and_merge.sh': '654bd59720321f316945aef187eba2e370f480184c0199d95b5e7c600c22a3ab'}, 'sha256': '4d293549d5a14fd8ac4d3a7ceb3e1d0db0d32dde943f9cd08db2e12ee652f31e'}, 'artifact_sha256': {'per_seed_metrics.csv': '3ae82f2dc292ab94fad7a9f42c2e25b4bc7068a1c90faad4fbc19b388c4ae47e', 'summary.json': '7e0eaa138fd954dbf89436c155dcc38c4c45df6a97792d3a99fd3de6b83f2791', 'qualitative_examples.json': '2857fe88af1ffb58b3ff1adb072f3fc2d88662cf9ddb3bff52652d3c2993bdec'}, 'artifact_zlib_base64': {'per_seed_metrics.csv': 'eNptll9vHEUQxN/zWcar6enumZ5H/gSEhEAC3i1jHxCRxMhOQHx7fj27jn0XTvKdb2+2p6a6qnofT6e78u704Y/7u3L//u2b96frm9vbjw83t/+W24/vrh9Ovz+cPpT94/qv08P13en2zeOb+/f588e3Nx/e/H26vv/18fTw9+mO5f/cPFDp/Pv5fRfbXP9zOv15d/PvZ9fv31/d3rx9+/n13367+uP+48PjE6qnAsfXp/uevj4tf1XLlzePp1K3NqW/eI0ivXFxjGq1uU3RYFUfvVk1VdPpNqx4bMPzdrNxViGvrR9i/a7KBxVFK1e0ThsSVqxvKj3cmxoF6+jFdQuv3sVCandrYPzuq++pYUVU+NHdwsHVNBIT4HSGyASXaYty5VstV3WrrYa+fLVVhD9Z0DShaWx8m2qjD+NgWkw3HSbmPbpZE/fiLGbXzi7dNXSC6acvvs3NKTGCo7Y2QDzDfGjWbr26gLdxplm9iNuWm3bTRUji6AvGKDo3seZSHbwhUVrbRh3Ds0qHqk43NvpQKcYGfUokgtff/fDNjz999Tr3Y0UyPjtUsMRstYBGDothoNOpXeDGsy+w0/ScnySn7b1sO0B6a3V61Mkb3WiFM0D2MFCZq9QoNmCm6+B/104fE9c3vyQzZ3pKQUlqZXRVSnbhptxlWtD7oTAI4a1YXTQhil03C4vvX3pS5y1vqnM0aEgV0EOnY+qVLzO7NIvWLWat7ASdM+bs4Ppxmebq568XvrUBve4OqdkoQwVra24ZIaOJegMZ3TAEVbep56+EvwMb6z3WtvQ3Ox7cHb2MLWRSWvscjpqR/DZj0I2qKKzP3l/Jswf9TLAIvhtKwHvaLYW/mzAQDkR6q03FpMy2HYI+xNX02XR8pNx7bHlOo1KdgqWTaa2c1GbHNX1E8eVG86rNQG8CtMN6F05aToQhgyyEQC9sOTHwRsOYlf8Qra9eplji7FifLHjgzfdsPBpKjc/WgFlMuJU9yIVeI2WSRqxw5w1Bhfc6QLgb0fql3Orcogm51hQ8yCBX4deIIShOcqMSelBnh+DOgfn/4dKBUTjHFJ8NXFyRbfbWVeY0dNwiYb1056VyCJUtHdkwKo4QbysaImOWhOgOv6yKnT9C+JK/lTG+v3l6sDYsODOWfSZmCtsAZvCT9VxDmFGadIhUUiLcfeomZ8RNIeZnm5G5j4zbUhZHI5LIqJ7OIM/M9uTvVQ/K2sHYk7dsa/RP2a9Nk9Q+udZIk6gcGqFFXiKPOjqko9ibdElkF069HE02UXhU6LclVVuaJ2+RGTAhzyVLW/KHiPTsiCt2dtvuARy6zwGjv/k3pA3cMLZJO3DvCLXMkkIqe9LKjpFmBmv75N1z62pGdqcCgSxJms11ElY1+kHQdtK36CE/2v4ZxsPEz1PUJxwn/6iOhMH4EOHC4EKbkgRHZnZuRk9qT9XXhHh42D/zMGcf6XiiLhOpLQ97utcRU3V2Klc5t3Kaym6Q51bbYo3AnagFpwmW8BzpcCjeDRUJ7ivm5IdVhNfW5GhAOkzr/xcryJhFVUjVXVI8fdCVPjy932e5knpMeDsfMiuF7NN89ZysFozsHA7RleU5sYjOsEFMG8kSKVRXLMYhaK5Iwntp3otHo8w1tDFIELqLWvqueSehBg0G9+T5pFy1kQ4BJr3Zldb8k21XQ9M9TBseFXwO8OWAZQgy+GGVDvA4kfOOwGfSaXIH6oR3TFjq0eEtH3ty4giTpfaVXjG1KsJFbmNaGX3bJ8ERcke6yfN0ZRISMk4pZNEZ6UXZeNDAzhMbvmcOzlwa9JKxK4JLgHJu1XHBVWY4lsQ+EzmhK0l4JIhhU4rAHnOwEPJ7mND6l4+R43n2j/0jEjLz2KG45VTgBnIEGhAl0xR+qEkrSt+WpjFzMMm8yqv/AI3HlGA=', 'summary.json': 'eNrtXW2P3Dhy/n6/QhgguCTnboukSEk3uQC+Xe/Gidc+ePcuCIJAUEvsGZ27W32S2t7Zw/73PEXqhepm97x5kQQ5L7CeaVFksVhVz1NFsv3XXwXBVVHv1tXN1W+Dv+I3/N5qXbb4VbyYfs/aLm86fBj2H5Z5l7e6yz7ppq3qHZ6g3TbfdVWx6KpuoxerurxbtLd5s190Yciujl7cHbb7O/d1voyWamjVHNDTVt/TauhrXe1udLNvqh2JeJUmqijXLI9LuS6KMCr4Sog1W63TMGfrIlrzKE9jrROplRIiDkXCyyRXsY6KVPBy6H6ru9u6zI6m2o6aQpPf48mD5472b756+5jmH159+6jmr9+8++b9h69eP+qlb354TPP3u02104vvv37Qa+atn/36dFbNq9MvtIyT1r9ch/26fMEO3ZX7gt3aRfpyHR6t/rM7ntnHPq8ahJreTNqu0fmWRql3Oqi220OXrzY6uNE73eSdLgP9Se+6wLYL9roJKFIFVRs0+tDip9VdkG82ge2vDVZ6XTc6yHd3/UeLdq+Lal0VQV508OygboI1uljlxcegyXdlvd3pth3CwU29KTPbMmvz7R6auHHttqwgSrU6dH2o2jf1Kl9Vm6q7W+zrz5Cv09s9yX6AGG2RUwdBvQ52dbPFLz+ZKeWbQ97VzQLd0JvBZ13d3HbttAQ7fZN31Sed9S2yW0jay3LVHlZdAxmD7lb3+tlWu2p72Ab1boPebvUuqDpS0tDP1LMjHkX6ZcjGR/rHPVbBhFcW4sn4ACrdHjY5jf2XrAp+F+yz6h//8e9Z8NKd7T/g1/awzf6Mx3/2PHZmd9jqpoJusgoa1luMmQ8Kba0BbOqboN3nhQ4+V91toH+k+f6kmxpD7Pd10wX7Rre6+aQdwy2b/HMGSOn1VO/wOpmM1dG6qbcwDKil3hjTIkvCXx/efTs30V3dkcEYkTIAZ6d/NJBTwpAIHBaN3mANySzvdlgDRMbARMaXFBmD3wR4Vd/UDf24qTHNgGAOvzT6xv5gP8XkMZEcUg7WZ8082+jdTXeLIXk0QPH+FgFzepCMEN3LlCHAQhtQKWZ/6DQFWyaHRtWNbrus1BuYQnNHP+R3bkM+dDd4Rva5gmd8bp1Gowu8effD6w8f/viHH7I33333+us3r36ggDaZ0fT86zfff/fm++9fvXWEmTd4/fbVf7z+Ovvw+tXXrhxo9Bbdfjj67NWHr/7lzZ9eW8XMlmyUu16RUdil2+Zw1R+9otNyItrossJSBfUeDrP452B8fB0YFcE4sB4lPTLiXAeuaVAbbX7A80G0E/l75Y9jmM9fBNWu2BxKig35GgsX5HDhluLZJBUMfK0bDfOAOLaTiyNO2oEFB/1ik5kfmjU5Et7547t/e/f+39/Nzd2qDDG50Z/zpnT19f4Pr99hecZ1fvsfeCqdyNA3ePUNJmVXEw0WzGmBD1//gCbv3v/w5ps3X7364c37d9SGn/by9ZtvX3//g4lKXJ508M2H999NTWZjvHufvf/9968//OnV79++zr7HC1/1wyCKzeZqIzuZ83+e2sTR2p0o1vz+XzOeM+vJcJoZH5lxCQ8PmGG4F39nY7bdoUQky7Y1VpfW+W31l0NVvnrz8u033/GlXHARfnd11Hhfb6rizlgF4tiiqz/CEF+9/P3Lr4IJvCrdXgejXC8CyPMCsbIMHFkCnRe3AZCn2hnkATzqgIJwt0CQXeTNNvjDazR7W394FeRlvoddD8KYXwnPjOgZkccxSMPSb+9agoPAzicwjeAirQmODti3+i8HzKkC4t9dA+La2WC2IeQJutpIWJECTMeAR3qr+sl6bs8SNEXDnieY2RIiLGnOd6SBEwUA/A04Ud/DmOjIaGJhhb5F3GwJzAF9UBdN8mg9KEytMGuaPCDpJ6xGP20sYrCCDR3RgZM3M7MGJEm2zxuMDTlaN23Tn6rC9L/dj50A3Dd1YwOj3rfVxsAt/EP1DVb6Nv9U1Y1jMGTQLwIYMgzi1bfWIEgvWItA/V3Qd7O4aaC2O8d+AsOcdGtWqDg0DcGvWSnb9/VMq+itb2ly1APFrAPWHdAIULtbIIjf3JhINrqqsVxt5NnXbbdoO71fMBUGXb6fjG5idxmxzt7ginq7hQmQyRoq2dNC6mvkhaAEPeVs8byp23YgmNcntJLaGtpFlrDRebOjqG7geJdvyK0wR2MMsJYCw1QlRfeBXNrpg5YYkrL4qAl1SlpS4nQtcYvDriIGBrPLO+oBIBS0t9W662V/MQr+gthto/fAroDI0Lj69rOsBf806DmEfrMMMKyuuB0eDs+g4u2+A0O/2xhbKvJDC3pR6m09dnsLAcFzgBy7Ug8Kzjdm6h1NzsXLl00NKhFQB/Bsa4qtZWUgVJXxrBYvAeKwHvDaDU2pNcq9DvLtCrZRH1qzUj2QWTX3QR29bCk2NZoMh1KDAzQENyd92dmMZPGqX5Csn6VTfIAVglLphZ3v4pO4mivEBNBsdShvNLHCWCX986rYZOCpxvKdykqT35z7mIhlZp3d+KJ0nrWgsRtEyc54Ip6FZHk55dXB9Cz4e0RESGk7GYnny55pbmlhwTcnpknstGoKsPkmuK0PzT/gA9P5J9BIE9smImuh4l/zogAtcAadjKrawTAL8NLGxpJwyCacZzPjSk4eT9HGLmMPUFMsMcHOWsER7JiAeQI5wZAWeiL4qeRnQimLeSj43EscmfW6y/IG4b3TRZ9JjZzpNKWh5su3CL5fmQAHjZtPYD+2JwvnI/ybz7q7vfG6t+8/vHKSt7z9OD766tUfQa2zt99Nz5vJf/Ebxfss3yB1mDQ/fFw29R7uaJfN6b8hqSDQwVrrwG0CyvzgAH8ex8InH08++XTySas36yzvut0So9mn/cP/mgQC025aOBatxq6lWDcbmk/9RdOPavoxmX5kofMzPzfUHhIhRpEW7SeTCldV3loKvXPIHPGHzChuwObfwjAPesqdd6DZun+KJbKxcA0AmNog6b0Bzcb8MmLcTpMZR53sbM5aSCpygB7AJsM/IjfwDQRl8gHLcjpEacTB7dU5c4YrUbCeJXkDkGVUhTlybtKSx6nNx1hv6pwA+ci24BGwuTsk1HpNHuY82uY/ZjdNXmY7u/TuM5uUZLAezBv8Md87Yj4gRXlIknIhTfH25CQqvj6OMpV5F/fnKrCEM/ZggpVZlJNMrUUs7zmfqbaUgf5RF8BbMJU+/8P/OmBF0KvSRNTJwv9/KXic9mH3cVd/3sEHNroYvKyAN7seNGgMutlD/VQGCMheK+K1NmY6MVgT1toSZ1MVJsaQM4FvGgxD7mM7aerNBj0HY/49FjHs8hrwKg5UeTOp01Gebli4AUQkU5D1+DEt5Bk7qvdUJPtJE1xcvSrz7b8H+aGraUpB/YkKZlRug6xJsNOf8fdgU0ZXgyEtBrWMVO066Gca9DL05O6M/bp6vbbZZVN/pjSgPWxpRqb0AV1b9npNRZOa0BUs8rZuqp8okzMMhxjeenNobyeC5xIQG5RMwlX9iI5NHdHyu2B4eh1QzPodbOS0h3Ohy2lyNoQ5bYp8nxdUzDW0bOa9fQTOJjfuo3RbggwMj3ue4xKP0eyIFmVDIj1WCGYYZZrYlNMo5UIrSx3mDOe0Vc+4JuoEDR1M/fh82xNMmzUdWhl+l1n3y3Kj0Vk7SpqyMdfOejbvh9O14R1dXdQbl+f37yx6zkxqW3xkC6dinSzGdOTK6WxcLPSKnM3UbwfTmr++aWK9iBamPEAaXXiWboRjp+eZafTJv+fZOZiOnWg4T/wEP33UpzVOiuJGUz/Q0wrsKY01QE/vuB3Dt7eHvdujQ9ss/QTP++j7/IStXuSrw8MLtNVHXH3U1UdeL9JXh1UOYtzHY2dMdsZlZ2x2xmfnjNbhtJ7xIYFrDhntPuk5naKm2+xWI1BdfGV32Gycd0bAGAhuqYv87mQxpmYrpKrMPE+d5xd4nivYkU0fyTIWArLegG+aGtbmpNGhdJoPif5832s+cqPztjZDmuTz2Acm83ILIeuDiSSzbmwVYwhHZ2UaG467N/e2HKtzTsvUqxXf4N6Gp4OzMy19g/tbkoc4PMpqqt+xu/KEJSoD6Zs7e8igZ2DApg0F1dLzQqPBzQraDoVfbqr1RfvoQ9SwbuPOituvRWawHNogM/XgIXdx8aZPbbKPm3MqGFU0jNYXRzNbHL3yNfWVYb0tLk3Xreua6qc9xTPM4t7WlzRJaAXiaYqt2dZspR0t/NDCFHFos3AksSdtTamuOey7C/0dt7kg3El3iC2mgFxqC8hzY0ajPTyb2FcfwoaFmiX63paFSYkiT/YwlnNH4sNc1mMKvEeRJ3Fy27HJFFpGLnK0RzeRhL8Vn/5WfPrlik+OoT2kNErtJieYarn/9jsWOMbcbwgFeTc7HpPYBHGs6totNHPgg/aDhoJuv+1Wvgg+31bI9qrWvGIJd59POBtv7hGdsRjsKwLPZLcSNjPqDRzdGmtfbfKPmq8Wzh7Not+VgQCUVOhFUa4Xn5wzdGiVrSu9KY9MmTadXCM1fWbV7LM5Vs7kbBAu6vUa6cCpLfeiZBDlJAdDRNDrw4beP5sm5UWh92abAJLTLo4ty9OHzski0v1wPGJ/WGHJh1x6OJpD9Ubvnk6962wFYFhzqLwcN8fM/tA1XiQKMX66rVqTL2u782P3bKfdIOd4xfW0LdQGQw3HXe2+VGN3nRe3dReYvb1Fn9/3pRxM1e7wmonTrvMlOXVfpxlKRk39mY5dmSoKQVRfvJgb3ePrAG56uDzJCf8vpfim6cD/oE+K9ifRyjSa0olboE4zi0Ke5sPwQ7hwqgKnAgwsGS5s6svjfp45kHAmmNAOrylTDfUwxOnFYHNDFWxajGk/GzKt+7M8ra09jEFwkfT2tbAuF9xSZcxaorX4POgOtJ1JtYRpg5mIsw2VVpcLwY825IM+zgcUFOApTgj8dTu8paKg59QvDY96aVhhMCyMEcB6sj3jiDmVFSmD/KJeD+XP9uokmHhDFqjAeOKcKlvjxiFzP50XLVTkeXa2amFbeTcdzRN/3cI+O1u4sI/JFqrN5lwq6S3Z3benwty6sWfSD6rTnNuOOVejuVSh8ddnnkDn7qvM/JKc7iHVmC9J7B5YfXl87eVBlZfLdZeLu2sPqbk8quJyf73lYrXlYbWWB1ZaHlpneWCV5YE1lodWWB5aX3lsdeWRtZVLlRXBvWLcW1d5YFXlYk3ltKLinAnLjrL/q9O3ZsUVfvb52RLDfYUVpsILTd1e3fOy95RUHl5QOVtOcc5ZX6qmJBf7Oq6lhEs1VQgeVkm5t47y8/HR0FKv88OmywbaZPU522T5q/+Kyl9n8NBb+nDDza2/z0vp7i/+sjoQ/881tGNlarYDoZ2xz0GBezp4Njgn8s5m0dBx1ZZShGq36FNgcDlzI8TmGCBJCIUmQ7bHDa9OakwfHjdFxsPwC8/FJnCL/g6Om8ATOxygY2EYaVuAe+48s3DuPv2PL9c0amOTD7p4URWdczzzBv/7bbCv6405SpEX3aI/KFrUTUP56T8HdDy2P4NqmnV1h4VsNOh3F/yTeXxNV4w+UioJey7uAkjVdi9o6Xd9w6tfxIzMtvmgjsBVR+BUDn72lX4Mff1bkfFvRcZfvMhoEyVbyyCBTo9ymhKkOQlCLwzWbmqFzpl5U+Oj3Pz08P21Od/Rn83vJwYfye+C/ug9ua97nL8aijd2KlceiR9xZNS0d0/bjD40mLg5fTOtwuUU8UFs/RdMRi6kIJeSjAdkp9neHPqADOZaMJ+ZiyfvnyrNQ7Xneix3mAP1plxpTyPBNsi4deAsl9uz/mR5TubUOdylGrOD6TTIDMPG408zMvxyKJGO5MpeAYOZjre47I3E06taV57963M7xPduIXtzmhN8dpIWe1mQhL0wT/dWW5+BtM79DMzSXKIyFSo65GZqJ7ubQ9Xe9uXf7X2zvDhJ5t99dd8/3a3ra7RZf+ivvyk3VH2bwFrkvJQ1SPagO1dD49lFLee202hT9sFww6k93ji0h9bm8eWonG8H6MuXRi43cGoToMqrk7TceeF4y2XWweBWpz2MtxKn83TDHkwfU52Q2/vpJLemOXVT4o7/FR/3tf1Sh1OkGK9pjP5uAMDO3gbvvihPFz2odNufhnTgwUhyPTbsNdDObnP1Pf56fqHsRdDai2TOxPy7T6NSDA6Z5wicQXPYHd0Znd0THi90/CYY7xabq8Kz+xpDIBkOeFKAo2vwPYc1MycX3FQ9DJpr4f3pX7tZ0d+hoTy1KqpuiDh349lgOGZphvuoEY0pJjn+PW22rHT3GfTecXRagv5u7FAIJ3id9l8MEyU0yYdToS8Cs4Wgm/4c6Qtas7L+3BuLucM0XpEPNvlqpI73FZeu+tfGu/njdUEyjT6BuR72cxbTBduJoNN8+jv3CwPcI6O/pgu5ZWWv3e8stgxy9VA2XT0fb5LBWiDDwVxpQ7d2X9NedBlvkDl3lgN77zooNnWrLYvXu3JRrxfDgVdzzHUYd8RssJ5dbsLTgGaLs1V7Oo349jtbvXcYk42CZD9Ocn1tjKlX+6/pGsENkJ5mBZC72dXmfpo1LrpuY+6cWTfsHX22C3x6C47UbU2OFONsfVgAt3ejPTuJMN7Rp67phuLLD6++7W9+QTSToA1nhcv5LTG6FWZZhB3D07v1mUGT46nm/vy2tWU6C67LaQP7wcfuScPOTdvPt9WG7jPZQ+OTk/dHpO2UWvegt959qpp6tzXf+0Anp2+rsiQY7rcp9/Ng1L81WrhdrRMbN1Nem+uDJn91bgf0lwuOvoFnxLJ8Q+vb3Zqv8cBoXCqnElht9NGZVYcbv8zMZmGWLfcmuDO+5pHiK1lEeViovBR5rHmuY6XVOkrLgqmCRcVKhSyOy3Waa5GvOM+liLVesdkeujuMLSkNg7DViidFmRZaqrxQpU5ZGKqwiNFJzFaapVJLsQ5zDlGY0jwKc5ZIVUSpKsri3CDOuvQjhTxmqdI5k5ppnqQigew5U1IXSbhKQkxkxaKV5mGYrovVKikZCyPBkzKKEkhwdqQfEf0qZyAWhSEX61in6yQt4zTkOo54ylZlwnMhNJcpW3NepkLGa6bLOC7iVbJWOpIqLVV8bqD+0ms/yiqK5ZqETlLGGYsUK1OeKvSvyiRc52W5EgyrlbJEsFBGXItYaMiBZRRcyXOjwKI+wUxudD9Owor1KhHlOpfJmoU6zkORas5ZpHPYRyKifC0LwQopSx0pXeIR9FfwREUiXqlz41gngxH348RrvlJay5THSc5jifytYKs1j6NSshJd5nyVxEwyJeJSy4jlUjGyQxHn0HIyP58IvtGYqki77Oz3hayKNC1lLCNVwpilEEleJrEO+arI81xhTipicsXyOFRJriNRhGko0wJiwfLnaQCYxLjW6xA6WUPTUcyhaFjvitTOZbLKI76O14WOE0xCF+siURLWjFlijGhdivXMfNuiqfZd+xK9Z1tkYaYIlyEoZyb9XbZ0OuFKyWhVytTkt2wtmEojmes1w1xWOS1yuI6SkMG4Q4YZp3IldVyoMCzgmnDQ02pgHyXQd1RyGCVcW+aYWJnkRUROX+iV0KwMy1VYCo7wlkZinRZlmJQrrhnXWkkOUQay/qu+96v2sN3mzZS/DV80NUar3iCGcDgPTVud2x0DnrA0mv64+VTblf2mQiilUHEC+0mTSEi3UVGlsm8lsC4qVoniXAoen2Ylh23WlwJ9sjAVLZVS8GCZCpEm6kQUvhSSsTQUTMZpEgsenQrC0UcICQQsDAadsBMxrAgmDx+46BndYCowuohFsIgIf/t0E6ZxCqEVZEp4MjvcOqkGwY6Up3gSS4UAEfl0c+b2k1c2xZYRU86f6FRZ6VIiPMo4CqVSLI3FqWhCLKMkkhyLKlIJ6z+R60iYh2iNSxo3Hv8oj9oYjAXBSKSMREx8C4lGkIkliUqiOEpYkspT6eYmjqRYfyzzs6Y+V5mKvesZc6aSOBEqSmKEgdS7oGEy6yqO7hUNNJjKCudES9m9kkUiSRMpYxHCAwXzygXcjiXgN01j+GmU3C/Wmgj3oWnPCAaY84cEGbp/OPeHhITDC+EbWGXMMD3ni5fWTYklJ1+OE6UA6qfSRMsIQIa4HMecA7FPJZHwwJBBAiIbTMVnY8KlRYrkUiRwZBEBigTz2TXiF7CVIaQCxiMmfJJAITJkAr0wqRJ+VpKL6yLVMowVEwlsIQ1TwTyhklH/LBYiVjL2R0oSkiEswVsRuZh3r8R+2eAjQSXi8z8+C0LARExN4Ul039oPKVhvmE9Ixgx9ARIejSkCgTIhHSBMMwZu6xEFYKXShBOZjGIAi08U6CgJESupG2gteQaqyCimiBZGIZADOOmNQkT0JJYYyIIgpIQ/CnHAIUAKFAgWxUL1XFjhs1qjlYaxJWgQVgjIJcFtTyVh8FAhoDwBkg0tsi+BIgixYvbHHxOTGJEIoRruz+Mw8ZuRlAwWEML9KaCDzTwTSMC9vUFRIr4oDqaURgA1oI1XGiwqcpGEYYkF4DF5LnZE/gjN5iGa+QGDgd1I+CBIF35IoujZiAHDnmGZd9nAoUGqEZ7CJDyieg6YwVOAx1LxFGqS/CnoIdJlmKgwwXqAvMrQoysGkEPKgiwOKgCICE+spCgC5SBmizAB9sv4SRAilkAx0EkGn0UnvrCNdVBYEE6QiZY+0rYU0F9IkZ2nIEbiaRgSJUuFiBhRMgj7EB5ZsEDg/nGaUramIp8oDEtEiAfeEal0IpAzDLHfL/tYDEnnfyKfIZGqkK6KVMkkSeLYa0exgKoxDWTw+Is/HkRCtVQcMJUAiDCK8uhKLQEvCmgGAgA1eEiIwJSkQIYVIo5KFj8DQZCncZgr3Ehh8VjiI9gM0QUtQBRFDBLn5WcMdCmhpDQSLAL7F8/Fj1guZ0T2NGYnHAsbJUjPEAJTKTxypYLSPJUQ4kJXvmj9eDgRhoVeTHMFwjoyIfiUjLHSXo0J6IvqG1jFEOlc/Fwg4feo6zRWKvmQWPlcSEm8AMcAnYB4zJ5xYo7Cn7WpCGHDmhYsXT4bUeScCfgCAYOaEmA78h4wEeXFEwbOjmhJ6CTBjVP1RDyJkQbAubAkHO7lESaWAmgDE0aSi2TByycBZ7ECCIDgQu7kSXAiyNsQl1UsABVIqU81wzi0F1GNIkaslHHqw7YImTFRX4AfnDLhT0xKEOCAAugKETnloNoeqI2WyFZSjjQMxN4P/Ayzgv0BhxXAEqr0Q4rnlN0DgQVhcWZQXvIEeAbywYKh2DTy4wqVxbDEMQAG4CLTxwOLjJdpSlEa2oLbeMgJX1J6xjksCpEelpX4EjmsLHJJEQMTuAS7fE7NC94GtggziDHomZJXBElSqngBNfCfn1nC4BCSEOU5oW+YPBdcFpwtTxUk4iVWAAwEi2VCzqkwEZLhGCELUQnLjZzz+YCywATh4vfZEZIm5BtUTU5DEYfSGwogG4iciuEZaIGZPLfMFc+pkxdVeHSeXZ2t38jkmbgijpI6n2RH9TevXPPqm3o+xMCBZ136a17wRDBzaE4gJVeJPykHf4hMOoEIAtuP46fAjJQgixKxgbSAuJx4ak0cnAYGw8OUYZU8phWBwCewe0R1DqiK+ZNylnSZEoGOowjcI2YeYkBBDAqBAYcxS5G6eDgdMRR4Hze8DgD9xLoXQh1QFXlcoqjqfqoWuaSMBlkaoheSWxn7anCAOLCqFO8LBFU/vszOoj8QWaRT9abM3mtElLMhFCnwJKCkPysnMEmQK8B8IjTkj89YJOFGSHWhVCgmT0VhYinxDHk/Qo5IZBh7oRgrxmFAYKAMnT4HWCIMQl4N3YOjpam3gKJIYJgQEiQVRv6gpEQiEqq0AO5g3Py5uJLESw/wwtSAFCSyVAAP38aOhGswsCtJdTHkUtGXyVSUiR2XcgIgiuIRaF3CaPcp9O5RIE4pRYVEAXOCZbJnwoq4H+6otIUUW0jYfkqVkjMFMKQoKiSwg2bTZ9e/pDdYi3hW/vLH6ohHWLsoQsZENT3xbBxR82G99Om4sOEvWSoifBy2HpLamXwKikRqKdNIcQ4eEsqU+6IlSDh8Do8l4Na7d0JUOEbUFgyypOmTMhXCbIlAJhiV2ER0WhVUyxRxW0XIK8lAIt+OQLwki0GOESMTBaN8YqLCk6Vh05StJFQT9JXhqJQN+wB9i1IReWtfxg1ScJUY4IhA7QUS//2tBwJKHPJHAkoYiwcgCo/lYyElApuAHQBVEhamisUeRgLPT0CkOPwfpJ/7GAlpFcunBIJ4pJ6zjcKQHDJYLkyCUt/QW2hmVGfFaCGlpSE/s01p6C2PU9qWgYU+G1GwJMsZ0/Uk5jEAMaUsV9C/IhX7di4Y1RqIddEUgYiexPwJ+JJKpJsiGf/4OTgocCyo1okAciY5AIvitGUWRSoCCj03bZFzTq/ux5cz5OkIXuJnwksc/6/aXjlKfbzLB3tO4RCSznRERM39IQEUCyw6DaNQkFc/rR5GuydMspj8C/lazL2RPcG6KQawo11Ub2BHCFUKsQPERoknbdAny9ioGAhDW4GefR56TEcIYsQDjuDu2+dRkqhDRAVqcILkiYkKgmVIFR06BwXs9eyuMGQhiKKxQPiG1qQvXHIYM4JcRKFOxUqkc4QZj4LRP4xQdTY+Od90YVRgvqFkOsI8iXtlLnrTVzQ5sBTOZ0NX6+w/JkH7wLPvw+u/MogeJLPTgfargijn4yefm4uo5ksR5mqbX7rf2TPPphN3J8W9o9J/VZLZVBsb/HwioPmGp/GGGYBg9l909GV4NX3LgDmyfyKe++9pRbOvMB3//SsxF9X5J7zkBQkRu+8fUXoHVGfGi73DkWVm4yz7L4KydwoUkTMgJMCSz6Jdf9p/+CIk4VqA544Rm5nCXu/on/3KzNH2rD/Vb2rSJz7FzptdqvxWJ7jf6maL+lSrY/dbnXyw1YmjP080Oua1Aff4yNzokuca3ZESvrzVsZhgh/L9+W7bkdXNq6YeqxPyAVZ3mrfwC7Fudq/QsTou/VaXhF/A6kR0r9WpB1sdR+aIjGr8f/I0s5OhP9bxc2Ynnm123DtidGbA9PFmlySMiiahRGqsztvdPOnz2N3s6NY5u0u8WaH5AoXZ1RLH+hxi69pe7H6Nz2h5XKijj88Z3tlgN0uAPVbHxClt9ZoccnewK4k0MUSKOO2aXTS4+eLPjnkNaz87kudEHHVGLL+dzQYSyjPO7MCWx6Z/dq7/X7CvKKGEQEkw2YnsHlsXm1zIZ1xiiihnTYunEw/81c+/+m9q6yiC', 'qualitative_examples.json': 'eNrtnd9P20gQx9/5K0Z56FNBSQ5E25cqBNNGDQlK0lbodLI29iTexvFau2tSdLr//cZ2EiVgB0yBwt1IQGLvD9uz3531x7Nr/twD+Jt+AWoG0a99gPrbfNPS9/fvlhsap3KOtKemon1PhGFtmeAJi1Olr9OkuYjEFPUqKVJWTiRlkCpyrbRhVv6LkBDRkQwI8NGThlKBfoYotBcUlh0rP6v/IkRhEEQca3WFoDRo/IGeBRsgfc1T8aeHcVoMFtIGMoJmHeYySiyag1XtUxX6rvCyyo2ntIymrorC7CCd3sgZDL5ejFaZ52gD5RtKy+1Eu07oQBvbtCevLC3fbVH5ZdksaUIXOxbebKvAuohrxQwLC6bGDUQUYZim+nKKxroyGquf25lUYj2Vt03/wuk5p+5p55MzHN3INTaor9B3E/p0DRmr9Iwpt4+huHaXZqM8jWZ9K13jQuhMLAfNo+LjmERPhIfuXFgvcI0VNklrqp23Ru3PtXWRfzYsZVF4AZ2dVmFIV0W5oyQMNzIYf2JdvJI+RlQztbgUYxlKe12SdZKk6nIXKKeBNbczacrjiciXPqnYXbdhQS7hparK+kdBqiblrUvu3biwWqfdLRNLa9D+3PnmPEQut4tuC2arB9GGS20qr1DTNZTJp9d3+ydDZ/CtddJ13KHTddqjTr93byF97X3p9b/3dkupeVgqpXo1Ja0O93/S0qD1ibXEWnocLTmd3ll/0HZ4KOOh7J6SORux+2H3U1VLNQ8jo3Rq6WgWqUVUu6WsfhTKCPeHp+UKu3lnXkljRYW3VRYnJnA3pXaXb+qcnzunHfI43ct766rkNG45qWJdHVXVVSUPVWzgXeKq5ZshuoYQKkS7ZbQSsW010/Is8j5zdMMmORwWpY0xEFdS5eXeF17fkzrEvdXf7Ijl/Nw4PK4C0CSwOWoPdxP0hbieY2RhImSIPkyIg69VosEJ5/AG2uqAyNgvwfAVSn+NU7Pk5eJlfTnorrH5EAJKNWAVzBDjDLKziu+F0ZsjcVWEfmkdvXVGVbqnTrd1+Sx3IvuNyv28MxwyU/ONCN+IPC5TsydiT/SrFM3+iP0RQzZr6XVANj/z+71ELeZjOU1UYtylyVSMvwTVZUx9c2TfQur6C0bq4/ptpJ5M9jNYLYRqEWLkizvC0idkMxVBrFUWUTbXkUfkO1daq8VOkm7BH/X9XGpE0GgJgjOEhoGMBEgDhtrCT1ak3vxQr8PF+ZMzNI8UPFIwUbOWOErNWmK+ZkWxopivWUvM18zXzNd38HWjIl//UtB6gP5CKR/6iZ1Ia1Ebjl5zzIhZm+9A+A6Eo9fsiZiu2R8xXbOWmK6ZrpmuXz9dvzt+iuj1J60WNgBN7YILoPbW1gBhcGO99nknT3dRBCDMjMCcmDpl5h9KZquyLZ3jAYxEnO5MGzMD6bT1FaSz2J9lSTWPGzxuMF+zljiWzVpi2mZFsaKYtllLvCCbF2TzguyK/N1sHD9FdPsyjURfUH+BN3AuUnROQ80QCAMmkHG8dozF/D0ipo6pz4gppjPG8WdMAiUYJ+YWWpNnXU9FhzHaBRKGN9J55GRNOKwwoXzbr7+mN5ud0oAxoi5+NuifV33EVjiacRDpWXCbNcOaqYrVrBnWzOt7YRkr53ViMmvm8TVTHVweUTrFB7+hoIkIDZbS8eqK5tJkZngQHfPzlxf5/OX2o48X5bJ2gWxj883ch1VeLEYWm5OJ76DYjTCwGJNhs1hv/l5u8IUJxiptzV0g+z0gNE1DyIEgaBUwV+nc7LfgqSTMY8vUcN4sqzjJJnH7EIoxhuYjT8vmyZAcNmZnzmFj1hJricPGrCgOG7OWeJI2T9LmSdrPzNaNer0KXBvlSRHe/V+vTLpMOWVg4mI0RkzxjqCwIALXQoYQKjUzMKWLsgfQ0tnKZ5hoRDCk63SONhG1NLBApK7if3z6WPCLoenVI9Ref9Q567RblUaH+z1DPSoB6iYD9cOBmgXEAvpvLG5mAXHImMN/HDJmzXDImEPGHDL+jSHjvb/2/gVDap+j'}}

import base64
import hashlib
import importlib.metadata as package_metadata
import sys
import tempfile
import zlib
from io import StringIO
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from huggingface_hub.errors import LocalEntryNotFoundError


def _canonical_runtime_status():
    expected = CANONICAL_REPRODUCTION["runtime"]
    actual = {
        "python": ".".join(map(str, sys.version_info[:3])),
        "numpy": np.__version__,
        "torch": torch.__version__.split("+")[0],
        "transformers": package_metadata.version("transformers"),
        "peft": package_metadata.version("peft"),
        "device": "mps" if torch.backends.mps.is_available() else (
            "cuda" if torch.cuda.is_available() else "cpu"
        ),
        "dtype": "float32" if torch.backends.mps.is_available() else (
            "float16" if torch.cuda.is_available() else "float32"
        ),
    }
    mismatches = {
        key: {"expected": expected[key], "actual": actual[key]}
        for key in expected
        if actual[key] != expected[key]
    }
    return actual, mismatches


def _reference_artifact_bytes():
    return {
        name: zlib.decompress(base64.b64decode(encoded))
        for name, encoded in CANONICAL_REPRODUCTION[
            "artifact_zlib_base64"
        ].items()
    }


def _canonical_streams():
    bundle = CANONICAL_REPRODUCTION["stream_bundle"]
    payload = zlib.decompress(base64.b64decode(bundle["zlib_base64"]))
    observed_sha256 = hashlib.sha256(payload).hexdigest()
    if observed_sha256 != bundle["sha256"]:
        raise RuntimeError(
            "embedded canonical stream bundle failed its SHA-256 check: "
            f"{observed_sha256} != {bundle['sha256']}"
        )
    document = json.loads(payload)
    if document["format"] != bundle["format"]:
        raise RuntimeError("embedded canonical stream format does not match")

    streams = {}
    for entry in document["seeds"]:
        seed = int(entry["seed"])
        if seed in streams:
            raise RuntimeError(f"duplicate canonical stream seed: {seed}")
        streams[seed] = [
            Event(
                event_id=record["event_id"],
                phase=int(record["phase"]),
                category=record["category"],
                scenario_id=record["scenario_id"],
                scenario_tier=record["scenario_tier"],
                title=record["title"],
                body=record["body"],
                hour=float.fromhex(record["hour"]),
                useful_horizon_minutes=record["useful_horizon_minutes"],
                importance=float.fromhex(record["importance"]),
                deadline=float.fromhex(record["deadline"]),
                affinity=float.fromhex(record["affinity"]),
                busy=float.fromhex(record["busy"]),
                x=np.asarray(
                    [float.fromhex(value) for value in record["x"]],
                    dtype=float,
                ),
                z={
                    key: float.fromhex(value)
                    for key, value in record["z"].items()
                },
                sampled_preference=record["sampled_preference"],
            )
            for record in entry["events"]
        ]
    if sum(map(len, streams.values())) != bundle["event_count"]:
        raise RuntimeError("embedded canonical stream event count does not match")
    return streams


def _compact_artifact_bytes(config, metrics, rollouts):
    metrics_buffer = StringIO()
    writer = csv.DictWriter(
        metrics_buffer,
        fieldnames=list(metrics[0]),
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerows(metrics)

    with tempfile.TemporaryDirectory(prefix="online-sdft-notebook-") as temp:
        compact_dir = Path(temp)
        write_compact_results(compact_dir, config, metrics, rollouts)
        return {
            "per_seed_metrics.csv": metrics_buffer.getvalue().encode("utf-8"),
            "summary.json": (compact_dir / "summary.json").read_bytes(),
            "qualitative_examples.json": (
                compact_dir / "qualitative_examples.json"
            ).read_bytes(),
        }


def run_canonical_lora_benchmark(strict=True):
    actual_runtime, runtime_mismatches = _canonical_runtime_status()
    if strict and runtime_mismatches:
        details = "; ".join(
            f"{name}: expected {values['expected']}, got {values['actual']}"
            for name, values in runtime_mismatches.items()
        )
        raise RuntimeError(
            "Strict byte reproduction requires the audited MPS/FP32 runtime. "
            + details
            + ". Set STRICT_BYTE_REPRODUCTION=False for a portable, "
            "non-byte-identical run."
        )

    try:
        model_path = snapshot_download(
            repo_id=MODEL_ID,
            revision=CANONICAL_REPRODUCTION["model_revision"],
            local_files_only=True,
        )
    except LocalEntryNotFoundError:
        model_path = snapshot_download(
            repo_id=MODEL_ID,
            revision=CANONICAL_REPRODUCTION["model_revision"],
            token=False,
        )
    selected_device = "mps" if strict else "auto"
    seeds = tuple(CANONICAL_REPRODUCTION["seeds"])
    canonical_streams = _canonical_streams()
    if set(canonical_streams) != set(seeds):
        raise RuntimeError("embedded canonical stream seeds do not match")
    curve_fields = [
        "seed", "method", "t", "phase", "regime", "step_correct",
        "step_feedback_reward", "step_regret", "cum_accuracy",
        "cum_regret", "cum_observed_reward",
    ]
    rollout_buffer = StringIO()
    curve_buffer = StringIO()
    curve_writer = csv.DictWriter(
        curve_buffer,
        fieldnames=curve_fields,
        lineterminator="\n",
    )
    curve_writer.writeheader()
    metrics = []
    config = None

    for seed_index, seed in enumerate(seeds, start=1):
        policy = None
        resolved_device = None
        try:
            policy = LiquidLLMPolicy(
                model_id=model_path,
                device=selected_device,
                local_files_only=True,
            )
            resolved_device = str(policy.device)
            if config is None:
                config = experiment_config(
                    seeds=len(seeds),
                    seed_start=seeds[0],
                    model_id=MODEL_ID,
                    policy=policy,
                )
                runtime_dataset_fingerprint = config["dataset_fingerprint"]
                canonical_dataset_fingerprint = (
                    CANONICAL_REPRODUCTION["dataset_fingerprint"]
                )
                if strict:
                    assert runtime_dataset_fingerprint == (
                        canonical_dataset_fingerprint
                    )
                config["dataset_fingerprint"] = canonical_dataset_fingerprint
                config["method_dataset_fingerprints"] = {
                    method: canonical_dataset_fingerprint
                    for method in METHODS
                }
                config["source_fingerprint"] = json.loads(json.dumps(
                    CANONICAL_REPRODUCTION["source_fingerprint"]
                ))
                assert config["methods"] == METHODS
                assert config["online_sdft_trainable_parameters"] == 172032
                assert config["online_rft_trainable_parameters"] == 172032
                assert config["reinforce_trainable_parameters"] == 172032
                print(
                    f"Loaded pinned {MODEL_ID}@"
                    f"{CANONICAL_REPRODUCTION['model_revision'][:12]} on "
                    f"{resolved_device}; six paired methods, common rank-4 LoRA",
                    flush=True,
                )

            stream = canonical_streams[seed]
            for method in METHODS:
                metrics.append(
                    run_method(
                        seed,
                        method,
                        stream,
                        policy,
                        rollout_buffer,
                        curve_writer,
                    )
                )
                print(
                    f"seed {seed_index}/{len(seeds)} (id={seed}) · {method}",
                    flush=True,
                )
                gc.collect()
                if resolved_device == "mps":
                    torch.mps.synchronize()
                    torch.mps.empty_cache()
        finally:
            if policy is not None:
                del policy
            gc.collect()
            if resolved_device == "mps":
                torch.mps.synchronize()
                torch.mps.empty_cache()
            elif resolved_device == "cuda":
                torch.cuda.empty_cache()

    if config is None:
        raise RuntimeError("canonical seed registry is empty")
    rollouts = [
        json.loads(line)
        for line in rollout_buffer.getvalue().splitlines()
        if line
    ]
    curves = list(csv.DictReader(StringIO(curve_buffer.getvalue())))
    assert len(metrics) == len(seeds) * len(METHODS)
    assert len(rollouts) == len(seeds) * len(METHODS) * STREAM_LENGTH
    assert len(curves) == len(rollouts)

    artifact_bytes = _compact_artifact_bytes(config, metrics, rollouts)
    REPRO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, payload in artifact_bytes.items():
        (REPRO_OUTPUT_DIR / name).write_bytes(payload)
    return {
        "metrics": metrics,
        "rollouts": rollouts,
        "curves": curves,
        "config": config,
        "artifact_bytes": artifact_bytes,
        "reference_bytes": _reference_artifact_bytes(),
        "runtime": actual_runtime,
        "runtime_mismatches": runtime_mismatches,
        "strict": strict,
    }


benchmark = run_canonical_lora_benchmark(STRICT_BYTE_REPRODUCTION)
metrics = benchmark["metrics"]
rollouts = benchmark["rollouts"]
curves = benchmark["curves"]
config = benchmark["config"]
print(
    f"Finished {len(CANONICAL_REPRODUCTION['seeds'])} seeds × "
    f"{len(METHODS)} methods × {STREAM_LENGTH} decisions"
)

### 5.5 Verify metrics and exact artifact bytes

The cell rebuilds `per_seed_metrics.csv`, `summary.json`, and
`qualitative_examples.json` with the embedded production serializer. It then
compares the generated bytes directly with the audited release bytes and shows
both SHA-256 digests. **Byte-identical reproduction** passes only when all three
files match; strict mode raises on the first artifact mismatch.

In [ ]:
# @title Verify all six methods and exact result bytes { display-mode: "form" }
from IPython.display import Markdown, display

if not all(
    name in globals()
    for name in ("benchmark", "metrics", "rollouts", "config")
):
    raise RuntimeError("Run Section 5.4 before displaying results.")

summary = summarize_metrics(metrics)
table = [
    "| Method | Accuracy | Cumulative regret | Reward / decision |",
    "| --- | ---: | ---: | ---: |",
]
for method in METHODS:
    values = summary[method]
    table.append(
        f"| {method} | "
        f"{100 * values['online_accuracy']['mean']:.2f}% ± "
        f"{100 * values['online_accuracy']['ci95']:.2f} | "
        f"{values['cum_regret']['mean']:.2f} ± "
        f"{values['cum_regret']['ci95']:.2f} | "
        f"{values['observed_reward_per_decision']['mean']:.3f} ± "
        f"{values['observed_reward_per_decision']['ci95']:.3f} |"
    )
display(Markdown("\n".join(table)))

assert config["teacher_model"] == MODEL_ID
assert config["student_backbone"] == "frozen Liquid LFM base weights"
assert config["student_backbone_trainable_parameters"] == 0
assert config["online_sdft_trainable_parameters"] == 172032
assert config["online_rft_trainable_parameters"] == 172032
assert config["reinforce_trainable_parameters"] == 172032
assert config["methods"] == METHODS
assert config["teacher_student_model_sharing"]["model_instances"] == 1
assert config["teacher_student_model_sharing"]["student_forward"] == "LoRA adapter enabled"
assert config["teacher_student_model_sharing"]["teacher_forward"] == "same model with LoRA adapter disabled"
assert config["teacher_policy"].startswith("the student and teacher are one shared Liquid LFM model;")
assert "delayed observed user selection" in config["teacher_policy"]
assert "digest open ambiguous between INTERRUPT and LATER" in config["teacher_policy"]
assert "UNKNOWN stays censored" in config["teacher_policy"]
assert "no scalar reward" in config["teacher_policy"]
assert "no end-of-horizon flush" in config["update_timing"]
assert len(metrics) == 3 * len(METHODS)
assert {row["method"] for row in metrics} == set(METHODS)
assert len(rollouts) == 3 * len(METHODS) * STREAM_LENGTH

sdft_rollouts = [
    row for row in rollouts if row["method"] == "Online-SDFT"
]
released_sdft = [
    row for row in rollouts
    if row["method"] == "Online-SDFT"
    and row["feedback_released_at_minute"] is not None
]
censored = [
    row for row in released_sdft
    if row["feedback"]["observed_user_selection"] == "UNKNOWN"
]
applied = [
    row for row in released_sdft
    if row["lesson_status"] == "soft_target_applied"
]
assert sdft_rollouts and released_sdft and censored and applied
assert all(
    row["feedback_released_at_minute"]
    >= row["feedback_available_at_minute"]
    > row["decision_time_minute"]
    for row in released_sdft
)
assert all(
    row["teacher_probs"] is None
    and row["sdft_fusion_weights"] is None
    and row["lesson_status"] == "censored_no_update"
    for row in censored
)


def _first_byte_difference(actual, expected):
    limit = min(len(actual), len(expected))
    for offset in range(limit):
        if actual[offset] != expected[offset]:
            return offset
    return None if len(actual) == len(expected) else limit


artifact_comparison = {}
comparison_table = [
    "| Artifact | Bytes | Generated SHA-256 | Reference SHA-256 | Exact |",
    "| --- | ---: | --- | --- | :---: |",
]
for name in REFERENCE_ARTIFACT_NAMES:
    actual = benchmark["artifact_bytes"][name]
    expected = benchmark["reference_bytes"][name]
    actual_sha = hashlib.sha256(actual).hexdigest()
    expected_sha = hashlib.sha256(expected).hexdigest()
    manifest_sha = CANONICAL_REPRODUCTION["artifact_sha256"][name]
    assert expected_sha == manifest_sha
    exact = actual == expected
    artifact_comparison[name] = {
        "exact": exact,
        "actual_sha256": actual_sha,
        "expected_sha256": expected_sha,
        "first_different_byte": _first_byte_difference(actual, expected),
    }
    comparison_table.append(
        f"| `{name}` | {len(actual):,} | `{actual_sha[:12]}…` | "
        f"`{expected_sha[:12]}…` | {'✅' if exact else '❌'} |"
    )
display(Markdown("\n".join(comparison_table)))

all_exact = all(item["exact"] for item in artifact_comparison.values())
if all_exact:
    display(Markdown(
        "✅ **Byte-identical reproduction passed.** All three tracked compact "
        "artifacts match exactly across all three seeds and all six methods. "
        f"Generated files are in `{REPRO_OUTPUT_DIR}`."
    ))
else:
    details = "; ".join(
        f"{name}: first differing byte "
        f"{values['first_different_byte']}"
        for name, values in artifact_comparison.items()
        if not values["exact"]
    )
    message = "Artifact bytes differ from the audited MPS release: " + details
    if benchmark["strict"]:
        raise AssertionError(message)
    display(Markdown(
        "⚠️ **Portable protocol run completed, but it is not byte-identical.** "
        + message
    ))

## 6. Plots and audits

Only useful after Section 5 has produced `metrics`, `curves`, and `rollouts` in
memory.

In [ ]:
# @title Plot all six learning trajectories { display-mode: "form" }
import matplotlib.pyplot as plt

if "curves" not in globals():
    raise RuntimeError("Run Section 5.4 before plotting learning trajectories.")

method_colors = {
    "Base": "#D2D3D6",
    "ICL": "#B7BAC0",
    "RAG": "#969BA4",
    "REINFORCE": "#D97706",
    "RFT": "#75639A",
    "Online-SDFT": "#4A7A3E",
}

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))
for method in METHODS:
    rows = [row for row in curves if row["method"] == method]
    accuracy_by_step = defaultdict(list)
    regret_by_step = defaultdict(list)
    for row in rows:
        accuracy_by_step[int(row["t"])].append(float(row["cum_accuracy"]))
        regret_by_step[int(row["t"])].append(float(row["cum_regret"]))
    ts = sorted(accuracy_by_step)
    accuracy = [100 * np.mean(accuracy_by_step[t]) for t in ts]
    regret = [np.mean(regret_by_step[t]) for t in ts]
    width = 2.8 if method == "Online-SDFT" else 1.5
    axes[0].plot(ts, accuracy, color=method_colors[method], lw=width, label=method)
    axes[1].plot(ts, regret, color=method_colors[method], lw=width, label=method)

for ax in axes:
    for boundary in (PHASE_LENGTH, 2 * PHASE_LENGTH):
        ax.axvline(boundary, color="#667085", ls="--", lw=1)
    ax.grid(alpha=0.22)
    ax.set_xlabel("Online decisions")
for midpoint, regime in zip(
    (PHASE_LENGTH / 2, 1.5 * PHASE_LENGTH, 2.5 * PHASE_LENGTH),
    REGIMES,
):
    axes[0].text(midpoint, 3, regime, ha="center", va="bottom", color="#475467")
axes[0].set(title="Cumulative online accuracy", ylabel="Accuracy so far (%)", ylim=(0, 100))
axes[1].set(title="Utility gap accumulated in arrival order", ylabel="Cumulative regret")
axes[1].legend(ncol=2, frameon=False)
fig.suptitle("Six-arm paired benchmark · three-seed mean", fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# @title Assert the ARCHIVE feedback invariant { display-mode: "form" }
from IPython.display import Markdown, display

if "rollouts" not in globals():
    raise RuntimeError("Run Section 5.4 before auditing archived decisions.")

archive_rows = [row for row in rollouts if row["action"] == "ARCHIVE"]
archive_outcomes = Counter(row["feedback"]["outcome"] for row in archive_rows)
assert all("gold_action_scoring_only" in row for row in rollouts)

if archive_rows:
    assert set(archive_outcomes) == {"NO_OBSERVABLE_SELECTION"}
    assert all(row["feedback"]["observed_user_selection"] == "UNKNOWN"
               for row in archive_rows)
    archive_by_method = Counter(row["method"] for row in archive_rows)
    display(Markdown(
        f"**{len(archive_rows):,} archived decisions** produced only factual outcomes: "
        f"`{dict(archive_outcomes)}`. Every archived decision kept the user "
        f"selection UNKNOWN. By method: `{dict(archive_by_method)}`."
    ))
else:
    display(Markdown(
        "ℹ️ This sampled stream did not execute `ARCHIVE`; the branch was not "
        "exercised, so no empirical archive claim is made for this run."
    ))

In [ ]:
# @title Plot executed action counts { display-mode: "form" }
import matplotlib.pyplot as plt

if "rollouts" not in globals():
    raise RuntimeError("Run Section 5.4 before plotting action counts.")

counts = {
    method: Counter(
        row["action"] for row in rollouts if row["method"] == method
    )
    for method in METHODS
}
fig, ax = plt.subplots(figsize=(10.5, 4.8))
action_colors = {"INTERRUPT": "#D95C59", "LATER": "#D9903D", "ARCHIVE": "#667085"}
x = np.arange(len(METHODS))
width = 0.24
for action_index, action in enumerate(ACTIONS):
    values = [counts[method][action] for method in METHODS]
    bars = ax.bar(
        x + (action_index - 1) * width,
        values,
        width,
        label=action,
        color=action_colors[action],
    )
    ax.bar_label(bars, padding=2, fontsize=8)
ax.set_xticks(x, METHODS)
ax.set(title="Executed actions by method · 720 decisions each", ylabel="Decisions")
ax.legend(frameon=False, ncol=3)
ax.grid(axis="y", alpha=0.2)
plt.show()
for method in METHODS:
    print(method + " | " + " | ".join(
        f"{action}={counts[method][action]}" for action in ACTIONS
    ))

## 7. Takeaway

Act with the semantic serving context. Wait for the selected route's observable
user selection, or `UNKNOWN`. Build a reliability-conditioned soft target from
same-network hindsight, the frozen decision-time prior, and causal support. Never use
either evaluator-only answer. Skip learning when the selection is censored, and queue
valid LoRA-adapter updates for future requests only.

Treat this as an executable simulator reproduction, not a phone-scale energy or
latency claim. Passing the byte gate proves that this notebook reproduced the
published simulator artifacts on the audited runtime; it is not production
deployment evidence.

When you close the notebook, the transferable idea is the causal loop, not the
particular notification categories. For the longer argument and figures, see the
[repository](https://github.com/lin826/Online-SDFT-Benchmark).